# БЛОК A. Окружение

In [ ]:
# ============================================================
# A1: УСТАНОВКА БИБЛИОТЕК
# ============================================================
# Устанавливаем все необходимые пакеты с пиннингом версий.
# После установки проверяем, что каждый пакет доступен.
#
# ВАЖНО: если Colab покажет предупреждение
# "You must restart the runtime" — сделайте:
#   Runtime → Restart runtime
# и продолжите со следующей ячейки.
# ============================================================

import subprocess
import sys
import time
from datetime import datetime

print("=" * 70)
print("УСТАНОВКА БИБЛИОТЕК")
print("=" * 70)
print(f"Время начала: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# ------------------------------------------------------------
# Список пакетов с версиями
# ------------------------------------------------------------
PACKAGES = {
    "loguru":         "0.7.3",
    "earthengine-api": "1.7.39",
    "geemap":         "0.38.8",
    "openpyxl":       "3.1.5",
    "pandas":         "2.2.3",
    "matplotlib":     "3.10.0",
}

print(f"Пакетов к установке: {len(PACKAGES)}")
print("-" * 70)
for pkg, ver in PACKAGES.items():
    print(f"  {pkg:<20} == {ver}")
print("-" * 70)
print()

# ------------------------------------------------------------
# Установка
# ------------------------------------------------------------
t_start = time.time()

for pkg, ver in PACKAGES.items():
    pkg_spec = f"{pkg}=={ver}"
    print(f"[УСТАНОВКА] {pkg_spec} ...", end=" ", flush=True)

    t0 = time.time()
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg_spec],
        capture_output=True,
        text=True
    )
    elapsed = time.time() - t0

    if result.returncode == 0:
        print(f"✅ OK ({elapsed:.1f} с)")
    else:
        print(f"❌ ОШИБКА ({elapsed:.1f} с)")
        print("  stderr:", result.stderr.strip()[:500])

t_total = time.time() - t_start
print()
print(f"Общее время установки: {t_total:.1f} с")
print()

# ------------------------------------------------------------
# Верификация: проверяем, что каждый пакет импортируется
# ------------------------------------------------------------
print("=" * 70)
print("ВЕРИФИКАЦИЯ УСТАНОВЛЕННЫХ ПАКЕТОВ")
print("=" * 70)

import importlib.metadata

verification_results = {}

# loguru
try:
    import loguru
    verification_results["loguru"] = loguru.__version__
    print(f"  loguru             ✅ {loguru.__version__}")
except Exception as e:
    verification_results["loguru"] = f"❌ {e}"
    print(f"  loguru             ❌ {e}")

# earthengine-api
try:
    import ee
    verification_results["earthengine-api"] = ee.__version__
    print(f"  earthengine-api    ✅ {ee.__version__}")
except Exception as e:
    verification_results["earthengine-api"] = f"❌ {e}"
    print(f"  earthengine-api    ❌ {e}")

# geemap
try:
    import geemap
    verification_results["geemap"] = geemap.__version__
    print(f"  geemap             ✅ {geemap.__version__}")
except Exception as e:
    verification_results["geemap"] = f"❌ {e}"
    print(f"  geemap             ❌ {e}")

# openpyxl
try:
    import openpyxl
    verification_results["openpyxl"] = openpyxl.__version__
    print(f"  openpyxl           ✅ {openpyxl.__version__}")
except Exception as e:
    verification_results["openpyxl"] = f"❌ {e}"
    print(f"  openpyxl           ❌ {e}")

# pandas
try:
    import pandas
    verification_results["pandas"] = pandas.__version__
    print(f"  pandas             ✅ {pandas.__version__}")
except Exception as e:
    verification_results["pandas"] = f"❌ {e}"
    print(f"  pandas             ❌ {e}")

# matplotlib
try:
    import matplotlib
    verification_results["matplotlib"] = matplotlib.__version__
    print(f"  matplotlib         ✅ {matplotlib.__version__}")
except Exception as e:
    verification_results["matplotlib"] = f"❌ {e}"
    print(f"  matplotlib         ❌ {e}")

print()

# ------------------------------------------------------------
# Итог
# ------------------------------------------------------------
all_ok = all(
    not str(v).startswith("❌")
    for v in verification_results.values()
)

print("=" * 70)
if all_ok:
    print("✅ ВСЕ ПАКЕТЫ УСТАНОВЛЕНЫ И ИМПОРТИРУЮТСЯ КОРРЕКТНО")
else:
    print("⚠️  НЕКОТОРЫЕ ПАКЕТЫ ИМЕЮТ ПРОБЛЕМЫ — проверьте вывод выше")
print("=" * 70)
print(f"Время завершения: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()
print("❗ ЕСЛИ COLAB ПОКАЗАЛ ПРЕДУПРЕЖДЕНИЕ О RESTART RUNTIME:")
print("   Runtime → Restart runtime, затем продолжите со следующей ячейки.")
print("   Это нормально — Colab перезапускает ядро после обновления пакетов.")

In [ ]:
# ============================================================
# A2: МОНТИРОВАНИЕ GOOGLE DRIVE
# ============================================================
# Монтируем Google Drive и проверяем доступ к папке проекта.
#
# Если Drive уже смонтирован — перемонтируем принудительно,
# чтобы избежать устаревших соединений после restart runtime.
#
# ВАЖНО ДЛЯ ПЕРЕДАЧИ НОУТБУКА:
#   Если вы получили этот ноутбук от коллеги, поменяйте
#   только PROJECT_FOLDER_NAME ниже — на имя вашей папки
#   на Google Drive. Все остальные пути построятся сами.
# ============================================================

import time
from pathlib import Path
from datetime import datetime

logger_start = datetime.now()

print("=" * 70)
print("МОНТИРОВАНИЕ GOOGLE DRIVE")
print("=" * 70)
print(f"Время начала: {logger_start.strftime('%Y-%m-%d %H:%M:%S')}")
print()

# ------------------------------------------------------------
# 👇 ЗДЕСЬ УКАЖИТЕ ИМЯ ВАШЕЙ ПАПКИ НА GOOGLE DRIVE
# ------------------------------------------------------------
# Если вы получили этот ноутбук от коллеги и хотите запустить
# на своём Drive — поменяйте только эту строку.
# Все остальные пути построятся автоматически.
#
# Папка должна лежать в корне вашего Google Drive:
#   /content/drive/MyDrive/<PROJECT_FOLDER_NAME>/
#
PROJECT_FOLDER_NAME = 'Moscow_LandCover_Analysis'
# ------------------------------------------------------------

MOUNT_POINT = '/content/drive'
PROJECT_PATH = Path(MOUNT_POINT) / 'MyDrive' / PROJECT_FOLDER_NAME

print(f"Точка монтирования:       {MOUNT_POINT}")
print(f"Имя папки проекта:        {PROJECT_FOLDER_NAME}")
print(f"Ожидаемый путь к проекту: {PROJECT_PATH}")
print()

# ------------------------------------------------------------
# 1. Монтирование
# ------------------------------------------------------------
t0 = time.time()
try:
    from google.colab import drive

    print("[МОНТИРОВАНИЕ] Подключаю Google Drive ...", flush=True)
    print("   Сейчас появится ссылка для авторизации.")
    print("   Войдите в тот же Google-аккаунт, где лежит проект.")
    print()

    drive.mount(MOUNT_POINT, force_remount=True)
    elapsed = time.time() - t0

    print()
    print(f"✅ Drive смонтирован за {elapsed:.1f} с")

except ImportError:
    print("❌ ОШИБКА: не найден модуль google.colab.")
    print("   Эта ячейка работает только в Google Colab.")
    raise

except Exception as e:
    print(f"❌ ОШИБКА при монтировании: {e}")
    print("   Проверьте:")
    print("   1) Вы авторизованы в том же Google-аккаунте.")
    print("   2) Разрешили доступ к Google Drive в окне авторизации.")
    raise

print()

# ------------------------------------------------------------
# 2. Верификация монтирования
# ------------------------------------------------------------
print("=" * 70)
print("ВЕРИФИКАЦИЯ МОНТИРОВАНИЯ")
print("=" * 70)

mount_path = Path(MOUNT_POINT)

# 2.1. Точка монтирования существует
if mount_path.exists():
    print(f"  Точка монтирования     ✅ {mount_path}")
else:
    print(f"  Точка монтирования     ❌ {mount_path} не найдена")
    raise RuntimeError("Drive не смонтирован.")

# 2.2. Внутри есть MyDrive
mydrive_path = mount_path / 'MyDrive'
if mydrive_path.exists():
    print(f"  MyDrive                ✅ {mydrive_path}")
else:
    print(f"  MyDrive                ❌ {mydrive_path} не найдена")
    raise RuntimeError("MyDrive недоступен — возможно, Drive смонтирован не полностью.")

# 2.3. Папка проекта существует
if PROJECT_PATH.exists():
    print(f"  Папка проекта          ✅ {PROJECT_PATH}")
else:
    print(f"  Папка проекта          ❌ {PROJECT_PATH} не найдена")
    print()
    print("  Что делать:")
    print(f"  1) Проверьте, что папка '{PROJECT_FOLDER_NAME}' лежит")
    print(f"     в корне вашего Google Drive: {mydrive_path}")
    print("  2) Если папка называется иначе — исправьте")
    print("     переменную PROJECT_FOLDER_NAME в начале этой ячейки.")
    raise RuntimeError(f"Папка проекта '{PROJECT_FOLDER_NAME}' не найдена.")

# 2.4. Проверка на запись — создаём временный файл
test_file = PROJECT_PATH / '.mount_test'
try:
    test_file.write_text("ok", encoding="utf-8")
    content = test_file.read_text(encoding="utf-8")
    test_file.unlink()  # удаляем
    if content == "ok":
        print(f"  Права на запись        ✅ (тестовый файл создан и удалён)")
    else:
        print(f"  Права на запись        ⚠️  содержимое не совпало")
except Exception as e:
    print(f"  Права на запись        ❌ {e}")
    raise RuntimeError("Нет прав на запись в папку проекта.")

print()

# ------------------------------------------------------------
# 3. Итог
# ------------------------------------------------------------
elapsed_total = (datetime.now() - logger_start).total_seconds()

print("=" * 70)
print("✅ GOOGLE DRIVE ГОТОВ К РАБОТЕ")
print("=" * 70)
print(f"PROJECT_PATH: {PROJECT_PATH}")
print(f"Общее время:  {elapsed_total:.1f} с")
print(f"Время завершения: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()
print("Переменные, доступные в следующих ячейках:")
print(f"  MOUNT_POINT         = '{MOUNT_POINT}'")
print(f"  PROJECT_FOLDER_NAME = '{PROJECT_FOLDER_NAME}'")
print(f"  PROJECT_PATH        = {PROJECT_PATH}")
print()
print("Следующая ячейка A3 проверит и при необходимости создаст")
print("всю структуру папок проекта внутри PROJECT_PATH.")

In [ ]:
# ============================================================
# A3: КОНФИГУРАЦИЯ ПУТЕЙ И СТРУКТУРЫ ПРОЕКТА
# ============================================================
# Строим все пути от PROJECT_PATH (определён в A2).
# Проверяем, что структура папок соответствует спецификации.
# Если чего-то не хватает — создаём.
#
# Логирование: пока дефолтное (loguru в stderr).
# Красивый формат + файл настроим в A4.
# ============================================================

from pathlib import Path
from loguru import logger

logger.info("=" * 70)
logger.info("КОНФИГУРАЦИЯ ПУТЕЙ И СТРУКТУРЫ ПРОЕКТА")
logger.info("=" * 70)
logger.info(f"Папка проекта: {PROJECT_PATH}")

# ------------------------------------------------------------
# 1. Годы для каждого временного ряда
# ------------------------------------------------------------
HISTORICAL_YEARS = [2000, 2007, 2013, 2019, 2026]
GREATER_YEARS    = [2013, 2019, 2026]

# ------------------------------------------------------------
# 2. Пары для карт разницы
# ------------------------------------------------------------
# Формат: (территория, год_начала, год_конца)
DIFF_PAIRS = [
    ('Historical', 2000, 2013),
    ('Historical', 2013, 2026),
    ('Greater',    2013, 2026),
    ('Historical', 2000, 2026),
]

logger.info(f"Годы (Историческая Москва): {HISTORICAL_YEARS}")
logger.info(f"Годы (Большая Москва):      {GREATER_YEARS}")
logger.info(f"Пар для карт разницы:       {len(DIFF_PAIRS)}")

# ------------------------------------------------------------
# 3. Словарь всех путей проекта
# ------------------------------------------------------------
# Строим от PROJECT_PATH — единый источник правды.
PATHS = {
    'BASE':         PROJECT_PATH,
    'DOCS':         PROJECT_PATH / '00_Docs',
    'DATA':         PROJECT_PATH / '01_Data',
    'BOUNDARIES':   PROJECT_PATH / '01_Data' / 'Boundaries',
    'OUTPUT':       PROJECT_PATH / '02_Output',
    'OUT_HIST':     PROJECT_PATH / '02_Output' / 'Historical_Moscow',
    'OUT_GREATER':  PROJECT_PATH / '02_Output' / 'Greater_Moscow',
    'OUT_DIFF':     PROJECT_PATH / '02_Output' / 'Difference',
    'INTERMEDIATE': PROJECT_PATH / '03_Intermediate',
    'LOGS':         PROJECT_PATH / '04_Logs',
}

# Подпапки годов для Исторической Москвы
for year in HISTORICAL_YEARS:
    PATHS[f'OUT_HIST_{year}'] = PATHS['OUT_HIST'] / str(year)

# Подпапки годов для Большой Москвы
for year in GREATER_YEARS:
    PATHS[f'OUT_GREATER_{year}'] = PATHS['OUT_GREATER'] / str(year)

# Подпапки для карт разницы
for territory, y1, y2 in DIFF_PAIRS:
    key = f'OUT_DIFF_{territory}_{y1}_{y2}'
    PATHS[key] = PATHS['OUT_DIFF'] / f'{territory}_{y1}_{y2}'

logger.info(f"Всего путей в конфигурации: {len(PATHS)}")

# ------------------------------------------------------------
# 4. Проверка и создание структуры
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Проверка структуры папок...")

created = []
existing = []
errors = []

for name, path in PATHS.items():
    try:
        if path.exists():
            existing.append(name)
        else:
            path.mkdir(parents=True, exist_ok=True)
            created.append(name)
    except Exception as e:
        errors.append((name, str(path), str(e)))

logger.info(f"  Уже существовало: {len(existing)}")
logger.info(f"  Создано новых:    {len(created)}")

if created:
    logger.info("  Новые папки:")
    for name in created:
        logger.info(f"    + {name}: {PATHS[name]}")

# ------------------------------------------------------------
# 5. Верификация: реально ли всё на месте?
# ------------------------------------------------------------
missing = [name for name, path in PATHS.items() if not path.exists()]

if missing:
    logger.error(f"❌ Не созданы папки: {missing}")
    raise RuntimeError("Структура папок неполная — проверьте права доступа к Drive.")

if errors:
    for name, path, err in errors:
        logger.error(f"  ! {name}: {path} — {err}")
    raise RuntimeError("Были ошибки при создании папок.")

logger.success(f"✅ Структура папок проверена: {len(PATHS)} папок.")

# ------------------------------------------------------------
# 6. Проверка обязательных файлов (границы)
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Проверка файлов границ...")

boundary_files = {
    'Historical (старая Москва)': PATHS['BOUNDARIES'] / 'moscow_historical.geojson',
    'Greater (Большая Москва)':   PATHS['BOUNDARIES'] / 'moscow_greater.geojson',
}

boundaries_ok = True
for label, path in boundary_files.items():
    if path.exists():
        size_kb = path.stat().st_size / 1024
        logger.info(f"  ✅ {label}: {path.name} ({size_kb:.1f} КБ)")
    else:
        logger.error(f"  ❌ {label}: файл не найден — {path}")
        boundaries_ok = False

if not boundaries_ok:
    logger.warning(
        "⚠️  Не все файлы границ на месте. "
        "Загрузите GeoJSON в 01_Data/Boundaries перед запуском анализа."
    )

# ------------------------------------------------------------
# 7. Итог и список переменных для следующих ячеек
# ------------------------------------------------------------
logger.success("=" * 70)
logger.success("✅ КОНФИГУРАЦИЯ ПУТЕЙ ЗАВЕРШЕНА")
logger.success("=" * 70)

logger.info("Доступные переменные после этой ячейки:")
logger.info(f"  PATHS               — словарь {len(PATHS)} путей")
logger.info(f"  HISTORICAL_YEARS    = {HISTORICAL_YEARS}")
logger.info(f"  GREATER_YEARS       = {GREATER_YEARS}")
logger.info(f"  DIFF_PAIRS          = {len(DIFF_PAIRS)} пар")
logger.info(f"  PROJECT_PATH        = {PROJECT_PATH}")

logger.info("Ключевые пути:")
for key in ['BOUNDARIES', 'OUT_HIST', 'OUT_GREATER', 'OUT_DIFF', 'INTERMEDIATE', 'LOGS']:
    logger.info(f"  PATHS['{key}'] = {PATHS[key]}")

In [ ]:
# ============================================================
# A4: ИМПОРТЫ И НАСТРОЙКА ЛОГИРОВАНИЯ
# ============================================================
# С этого момента и далее весь вывод — только через loguru.
# Никаких print() в последующих ячейках.
#
# Логи пишутся в два места:
#   1) stderr — короткий формат, для наблюдения в реальном времени
#   2) файл в 04_Logs — подробный формат, с именем функции и строкой
#
# Папка 04_Logs уже создана в A3 (PATHS['LOGS']).
# ============================================================

# ------------------------------------------------------------
# 1. Стандартные библиотеки
# ------------------------------------------------------------
import os
import sys
import json
import time
import pickle
import warnings
from pathlib import Path
from datetime import datetime

# ------------------------------------------------------------
# 2. Сторонние библиотеки
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 3. Loguru — импортируем до его настройки
# ------------------------------------------------------------
from loguru import logger
from importlib.metadata import version as _pkg_version

# ------------------------------------------------------------
# 4. Глушим предупреждения сторонних библиотек
# ------------------------------------------------------------
warnings.filterwarnings('ignore', category=UserWarning)

# ------------------------------------------------------------
# 5. Настройка логирования
# ------------------------------------------------------------

# 5.1. Убираем дефолтный handler loguru (иначе логи дублируются)
logger.remove()

# 5.2. Папка для логов берётся из PATHS (определён в A3)
LOGS_DIR = PATHS['LOGS']
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# 5.3. Имя лог-файла уникально для каждого запуска
RUN_TIMESTAMP = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_FILE = LOGS_DIR / f'run_{RUN_TIMESTAMP}.log'

# 5.4. Handler 1: stderr — короткий формат для наблюдения
logger.add(
    sys.stderr,
    level='INFO',
    format=(
        '<green>{time:HH:mm:ss}</green> | '
        '<level>{level: <8}</level> | '
        '{message}'
    ),
    colorize=True,
)

# 5.5. Handler 2: файл — подробный формат для отладки
logger.add(
    LOG_FILE,
    level='DEBUG',
    format=(
        '{time:YYYY-MM-DD HH:mm:ss.SSS} | '
        '{level: <8} | '
        '{name}:{function}:{line} | '
        '{message}'
    ),
    encoding='utf-8',
    enqueue=True,      # асинхронная запись — не тормозит основной код
    backtrace=True,    # подробный traceback при исключениях
    diagnose=True,     # переменные в traceback
)

# ------------------------------------------------------------
# 6. Настройка matplotlib для Colab
# ------------------------------------------------------------
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# ------------------------------------------------------------
# 7. Шапка запуска
# ------------------------------------------------------------
logger.info("=" * 70)
logger.info("НОВЫЙ ЗАПУСК ПАЙПЛАЙНА")
logger.info("=" * 70)
logger.info(f"Время запуска:    {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
logger.info(f"Папка проекта:    {PROJECT_PATH}")
logger.info(f"Лог-файл:         {LOG_FILE.name}")
logger.info(f"Уровень в stderr: INFO")
logger.info(f"Уровень в файле:  DEBUG")

# ------------------------------------------------------------
# 8. Отчёт о версиях
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Версии библиотек:")
logger.info(f"  Python             {sys.version.split()[0]}")
logger.info(f"  numpy              {np.__version__}")
logger.info(f"  pandas             {pd.__version__}")
logger.info(f"  matplotlib         {matplotlib.__version__}")
logger.info(f"  loguru             {_pkg_version('loguru')}")

# ------------------------------------------------------------
# 9. Проверка всех уровней логирования
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Проверка уровней (в файл попадёт всё, в stderr — INFO и выше):")
logger.debug("  → DEBUG   — виден только в файле")
logger.info("  → INFO    — виден и в stderr, и в файле")
logger.success("  → SUCCESS — виден и в stderr, и в файле")
logger.warning("  → WARNING — виден и в stderr, и в файле")

# ------------------------------------------------------------
# 10. Итог и список переменных
# ------------------------------------------------------------
logger.success("=" * 70)
logger.success("✅ ИМПОРТЫ ЗАГРУЖЕНЫ, ЛОГИРОВАНИЕ НАСТРОЕНО")
logger.success("=" * 70)
logger.info("Доступные переменные после этой ячейки:")
logger.info(f"  PROJECT_PATH    = {PROJECT_PATH}")
logger.info(f"  PATHS           = словарь с {len(PATHS)} путями")
logger.info(f"  LOGS_DIR        = {LOGS_DIR}")
logger.info(f"  LOG_FILE        = {LOG_FILE}")
logger.info(f"  RUN_TIMESTAMP   = '{RUN_TIMESTAMP}'")
logger.info(f"  np, pd, plt, matplotlib, json, pickle, os, sys, Path, datetime, logger")

# БЛОК B. Earth Engine

In [ ]:
# ============================================================
# B1: АУТЕНТИФИКАЦИЯ В GOOGLE EARTH ENGINE
# ============================================================
# Аутентификация требуется для каждой новой сессии Colab
# или при перезапуске виртуальной машины — учётные данные
# не сохраняются между сессиями.
#
# После запуска этой ячейки:
#   1) Появится ссылка для авторизации.
#   2) Перейдите по ней, войдите в тот же Google-аккаунт,
#      под которым регистрировали проект Earth Engine.
#   3) Скопируйте код подтверждения и вставьте в поле ниже.
# ============================================================

import ee

logger.info("=" * 70)
logger.info("АУТЕНТИФИКАЦИЯ В GOOGLE EARTH ENGINE")
logger.info("=" * 70)

logger.info("Сейчас появится ссылка для авторизации.")
logger.info("Войдите в тот же Google-аккаунт, где зарегистрирован проект Earth Engine.")
logger.info("После авторизации скопируйте код подтверждения и вставьте его в поле ниже.")

try:
    ee.Authenticate(auth_mode='colab')
    logger.success("✅ Аутентификация прошла успешно. Токен сохранён.")
except Exception as e:
    logger.error(f"❌ Ошибка при аутентификации: {e}")
    logger.info("Проверьте:")
    logger.info("  1) Вы вошли в правильный Google-аккаунт.")
    logger.info("  2) Проект Earth Engine зарегистрирован для этого аккаунта.")
    logger.info("  3) Ссылка открывалась в том же браузере, где вы авторизованы.")
    raise

In [ ]:
# ============================================================
# ЯЧЕЙКА B2: ИНИЦИАЛИЗАЦИЯ EARTH ENGINE С ПРОЕКТОМ
# ============================================================
# После аутентификации инициализируем клиент Earth Engine
# с указанием ID нашего проекта. Это связывает все
# последующие запросы с правильным биллингом и квотами.
#
# ВАЖНО: замените PROJECT_ID на ID вашего проекта в Google Cloud.
# Если вы получили ноутбук от коллеги — впишите сюда ID своего проекта.
# ============================================================

# 👇 ЗДЕСЬ УКАЖИТЕ ID ВАШЕГО ПРОЕКТА В GOOGLE CLOUD
PROJECT_ID = 'moscow-landcover-analysis'
# ------------------------------------------------------------

logger.info("=" * 70)
logger.info("ИНИЦИАЛИЗАЦИЯ EARTH ENGINE")
logger.info("=" * 70)
logger.info(f"Проект: {PROJECT_ID}")

try:
    ee.Initialize(project=PROJECT_ID)
    logger.success(f"✅ Earth Engine инициализирован с проектом: {PROJECT_ID}")
except Exception as e:
    logger.error(f"❌ Ошибка при инициализации: {e}")
    logger.info("Проверьте:")
    logger.info("  1) PROJECT_ID указан правильно (без лишних пробелов).")
    logger.info("  2) Earth Engine API включён для этого проекта.")
    logger.info("  3) У вашего аккаунта есть доступ к этому проекту.")
    raise

In [ ]:
# ============================================================
# ЯЧЕЙКА B3: ПРОВЕРКА РАБОТЫ EARTH ENGINE + ВИЗУАЛИЗАЦИЯ
# ============================================================
# Три уровня проверки:
#   1) серверные вычисления
#   2) доступ к коллекции
#   3) визуальная проверка — рисуем карту Москвы со снимком
#
# Устойчивость к пустым коллекциям:
#   Для летних месяцев над Москвой бывает мало ясных снимков.
#   Поэтому cloud cover ослабляется постепенно: 5 → 10 → 20 → 40 → без фильтра.
# ============================================================

import geemap
from datetime import datetime

logger.info("=" * 70)
logger.info("ПРОВЕРКА РАБОТЫ EARTH ENGINE")
logger.info("=" * 70)

# ------------------------------------------------------------
# 1. Серверное вычисление
# ------------------------------------------------------------
try:
    logger.info("Запрос 1: серверное вычисление (ee.Number)...")
    result = ee.Number(2).add(3).getInfo()
    logger.info(f"  Результат: 2 + 3 = {result}")
    if result == 5:
        logger.success("  ✅ Серверные вычисления работают.")
    else:
        raise RuntimeError("Server-side computation failed.")
except Exception as e:
    logger.error(f"❌ Ошибка при серверном вычислении: {e}")
    raise

# ------------------------------------------------------------
# 2. Доступ к коллекции
# ------------------------------------------------------------
try:
    logger.info("Запрос 2: метаданные коллекции Landsat 8...")
    collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    count = collection.size().getInfo()
    logger.info(f"  Снимков в коллекции Landsat 8: {count:,}")
    logger.success("  ✅ Доступ к коллекциям работает.")
except Exception as e:
    logger.error(f"❌ Ошибка при запросе коллекции: {e}")
    raise

# ------------------------------------------------------------
# 3. Поиск летнего снимка над Москвой (с fallback по облачности)
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Запрос 3: поиск летнего снимка над Москвой (2023)...")

MOSCOW_CENTER = [55.7558, 37.6173]          # [lat, lon]
MOSCOW_POINT  = ee.Geometry.Point([37.6173, 55.7558])  # (lon, lat)

# Пробуем последовательно ослаблять порог облачности
CLOUD_THRESHOLDS = [5, 10, 20, 40, None]    # None = без фильтра

moscow_scene = None
used_threshold = None

for threshold in CLOUD_THRESHOLDS:
    base = (
        ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
        .filterBounds(MOSCOW_POINT)
        .filterDate('2023-06-01', '2023-08-31')
    )
    if threshold is not None:
        candidate = base.filter(ee.Filter.lt('CLOUD_COVER', threshold))
    else:
        candidate = base

    n = candidate.size().getInfo()
    label = f"< {threshold}%" if threshold is not None else "без фильтра"

    if n > 0:
        logger.info(f"  Облачность {label:<15} → найдено {n} снимков ✅")
        moscow_scene = candidate.first()
        used_threshold = threshold
        break
    else:
        logger.info(f"  Облачность {label:<15} → найдено 0 снимков")

if moscow_scene is None:
    logger.error("❌ Не удалось найти ни одного снимка за лето 2023 над Москвой.")
    logger.info("   Возможные причины:")
    logger.info("     • Неправильные координаты точки.")
    logger.info("     • Проблема с фильтром дат.")
    logger.info("   Что делать: проверьте MOSCOW_POINT и диапазон дат.")
    raise RuntimeError("Нет доступных снимков для визуальной проверки.")

# Проверяем, что снимок реально получен
scene_info = moscow_scene.getInfo()
if scene_info is None:
    logger.error("❌ getInfo() вернул None — снимок невалидный.")
    raise RuntimeError("Пустой ответ от Earth Engine.")

scene_id   = scene_info['id']
scene_date = scene_info['properties'].get('DATE_ACQUIRED', 'неизвестно')
scene_cloud = scene_info['properties'].get('CLOUD_COVER', 'неизвестно')

logger.info(f"  ID снимка:        {scene_id}")
logger.info(f"  Дата съёмки:      {scene_date}")
logger.info(f"  Облачность:       {scene_cloud}%")
logger.info(f"  Использован порог: {used_threshold if used_threshold is not None else 'без фильтра'}")
logger.success("  ✅ Запрос конкретных снимков работает.")

# ------------------------------------------------------------
# 4. Визуальная проверка: рисуем карту
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Запрос 4: визуальная проверка — строю карту...")

try:
    # Истинные цвета: Landsat 8/9 — SR_B4 (red), SR_B3 (green), SR_B2 (blue)
    # Калибровка Collection 2 Level 2: reflectance = DN * 0.0000275 - 0.2
    true_color = (
        moscow_scene
        .select(['SR_B4', 'SR_B3', 'SR_B2'])
        .multiply(0.0000275)
        .add(-0.2)
    )

    vis_params = {
        'min': 0.0,
        'max': 0.3,
        'gamma': 1.4
    }

    m = geemap.Map(center=MOSCOW_CENTER, zoom=9)
    m.add_basemap('HYBRID')
    m.addLayer(true_color, vis_params, f'Landsat 8 — {scene_date}')
    m.addLayer(
        MOSCOW_POINT,
        {'color': 'red'},
        'Центр Москвы (Красная площадь)'
    )

    logger.success("  ✅ Карта построена.")

except Exception as e:
    logger.error(f"❌ Ошибка при построении карты: {e}")
    raise

# ------------------------------------------------------------
# 5. Итог
# ------------------------------------------------------------
logger.success("=" * 70)
logger.success("✅ EARTH ENGINE РАБОТАЕТ КОРРЕКТНО")
logger.success("=" * 70)
logger.info("Доступные переменные после этой ячейки:")
logger.info(f"  ee                  — клиент Earth Engine")
logger.info(f"  PROJECT_ID          = '{PROJECT_ID}'")
logger.info(f"  MOSCOW_CENTER       = {MOSCOW_CENTER}")
logger.info(f"  MOSCOW_POINT        — точка Красной площади (ee.Geometry.Point)")
logger.info(f"  moscow_scene        — тестовый снимок Landsat 8 за {scene_date}")

# Отображаем карту
m

# БЛОК C. Параметры анализа

In [ ]:
# @title
# ============================================================
# C1: ПАРАМЕТРЫ АНАЛИЗА
# ============================================================
# Все параметры, определяющие ЧТО мы считаем:
#   • временные периоды и сезонность
#   • спутники и коллекции
#   • классы земного покрова (EN + RU)
#   • признаки для классификации (11 штук)
#   • палитры для индексов
#   • параметры Random Forest
#   • обучающие данные (ESA WorldCover)
#   • постобработка
#   • пороги точности
#   • проекции (UTM 37N для расчётов, Web Mercator для карт)
#   • технические параметры GEE
#
# Годы HISTORICAL_YEARS и GREATER_YEARS определены в A3,
# здесь только проверяем их согласованность со спецификацией.
# ============================================================

logger.info("=" * 70)
logger.info("ПАРАМЕТРЫ АНАЛИЗА")
logger.info("=" * 70)

# ------------------------------------------------------------
# 1. Временные периоды и сезонность
# ------------------------------------------------------------
HISTORICAL_START = 2000
HISTORICAL_END   = 2026
GREATER_START    = 2013
GREATER_END      = 2026

# Сезонное окно: лето. 01.09 — потому что ee.FilterDate
# не включает конечную дату, а нам нужен весь август.
SEASON_START = '06-01'
SEASON_END   = '09-01'

# Минимум снимков для медианного композита.
# Если меньше — ищем соседний год (-1, +1, -2, +2).
MIN_IMAGES_FOR_COMPOSITE = 3

# Смещения для fallback, в порядке приоритета
FALLBACK_OFFSETS = [-1, 1, -2, 2]

logger.info(f"Периоды:")
logger.info(f"  Историческая Москва:  {HISTORICAL_START}–{HISTORICAL_END}")
logger.info(f"  Большая Москва:       {GREATER_START}–{GREATER_END}")
logger.info(f"Сезонное окно:          {SEASON_START} – {SEASON_END}")
logger.info(f"Мин. снимков:           {MIN_IMAGES_FOR_COMPOSITE}")
logger.info(f"Fallback offset'ы:      {FALLBACK_OFFSETS}")

# Проверка согласованности с A3
_expected_hist = list(range(HISTORICAL_START, HISTORICAL_END + 1, 1))
_expected_hist_selected = [2000, 2007, 2013, 2019, 2026]
_expected_great_selected = [2013, 2019, 2026]

if HISTORICAL_YEARS == _expected_hist_selected:
    logger.success(f"  ✅ HISTORICAL_YEARS = {HISTORICAL_YEARS}")
else:
    logger.warning(f"  ⚠️  HISTORICAL_YEARS = {HISTORICAL_YEARS} (ожидалось {_expected_hist_selected})")

if GREATER_YEARS == _expected_great_selected:
    logger.success(f"  ✅ GREATER_YEARS    = {GREATER_YEARS}")
else:
    logger.warning(f"  ⚠️  GREATER_YEARS    = {GREATER_YEARS} (ожидалось {_expected_great_selected})")

# ------------------------------------------------------------
# 2. Спутники и коллекции
# ------------------------------------------------------------
# 2000–2012: Landsat 5 (TM) + Landsat 7 (ETM+)
# 2013–2026: Landsat 8 (OLI) + Landsat 9 (OLI-2)
SATELLITES_EARLY = [
    'LANDSAT/LT05/C02/T1_L2',
    'LANDSAT/LE07/C02/T1_L2',
]
SATELLITES_LATE = [
    'LANDSAT/LC08/C02/T1_L2',
    'LANDSAT/LC09/C02/T1_L2',
]

logger.info("-" * 70)
logger.info("Спутники:")
for c in SATELLITES_EARLY:
    logger.info(f"  Early:  {c}")
for c in SATELLITES_LATE:
    logger.info(f"  Late:   {c}")

# ------------------------------------------------------------
# 3. Классы земного покрова (EN + RU) и цвета
# ------------------------------------------------------------
CLASSES = [
    'Dense Vegetation',
    'Sparse Vegetation',
    'Built-Up / Bare Soil',
    'Water',
]

CLASSES_RU = [
    'Плотная растительность',
    'Редкая растительность',
    'Застройка / открытая почва',
    'Вода',
]

COLORS = [
    '#017721',   # 0 Dense Vegetation — тёмно-зелёный
    '#5dca0c',   # 1 Sparse Vegetation — светло-зелёный
    '#ffd700',   # 2 Built-Up / Bare Soil — жёлтый
    '#138eff',   # 3 Water — синий
]

N_CLASSES = len(CLASSES)

logger.info("-" * 70)
logger.info(f"Классы ({N_CLASSES}):")
for i, (en, ru, c) in enumerate(zip(CLASSES, CLASSES_RU, COLORS)):
    logger.info(f"  {i}  {en:<25} | {ru:<30} | {c}")

# ------------------------------------------------------------
# 4. Признаки для классификации (11 штук)
# ------------------------------------------------------------
SPECTRAL_BANDS = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']
INDEX_BANDS    = ['NDVI', 'NDWI', 'NDBI', 'BSI', 'SAVI', 'NDMI', 'MNDWI']
TEXTURE_BANDS = ['NIR_contrast', 'NIR_entropy']
FEATURES       = SPECTRAL_BANDS + INDEX_BANDS + TEXTURE_BANDS

logger.info("-" * 70)
logger.info(f"Признаки ({len(FEATURES)}):")
logger.info(f"  Спектральные каналы ({len(SPECTRAL_BANDS)}): {SPECTRAL_BANDS}")
logger.info(f"  Индексы ({len(INDEX_BANDS)}):              {INDEX_BANDS}")
logger.info(f"  Текстурные ({len(TEXTURE_BANDS)}):           {TEXTURE_BANDS}")

# ------------------------------------------------------------
# 5. Палитры для индексов (для визуальной проверки)
# ------------------------------------------------------------
INDEX_PALETTES = {
    'NDVI': ['#8B4513', '#FFFF00', '#00FF00', '#006400'],  # кор. → жёлтый → зелёный → т-зелёный
    'NDWI': ['#FFFFFF', '#00BFFF', '#00008B'],              # белый → голубой → т-синий
    'NDBI': ['#FFFFFF', '#FFA500', '#8B0000'],              # белый → оранж → т-красный
    'BSI':  ['#FFFFFF', '#D2B48C', '#8B4513'],              # белый → беж → кор.
    'SAVI': ['#8B4513', '#FFFF00', '#00FF00', '#006400'],   # как NDVI
}

# Диапазоны значений для отображения индексов
INDEX_VIS = {
    'NDVI': {'min': -1, 'max': 1, 'palette': INDEX_PALETTES['NDVI']},
    'NDWI': {'min': -1, 'max': 1, 'palette': INDEX_PALETTES['NDWI']},
    'NDBI': {'min': -1, 'max': 1, 'palette': INDEX_PALETTES['NDBI']},
    'BSI':  {'min': -1, 'max': 1, 'palette': INDEX_PALETTES['BSI']},
    'SAVI': {'min': -1, 'max': 1, 'palette': INDEX_PALETTES['SAVI']},
}

logger.info(f"Палитры индексов: определены для {len(INDEX_PALETTES)} индексов.")

# ------------------------------------------------------------
# 6. Random Forest
# ------------------------------------------------------------
RF_TREES         = 100
RF_MIN_LEAF      = 5
RF_BAG_FRACTION  = 0.5

logger.info("-" * 70)
logger.info("Random Forest:")
logger.info(f"  numberOfTrees:      {RF_TREES}")
logger.info(f"  minLeafPopulation:  {RF_MIN_LEAF}")
logger.info(f"  bagFraction:        {RF_BAG_FRACTION}")

# ------------------------------------------------------------
# 7. Обучающие данные (ESA WorldCover)
# ------------------------------------------------------------
TRAINING_SOURCE  = 'ESA/WorldCover/v200'
TRAINING_YEAR    = 2021
POINTS_PER_CLASS = 2000
TRAIN_FRACTION   = 0.7
RANDOM_SEED      = 42

# Маппинг классов ESA WorldCover → наши 4 класса
# ESA коды: 10 Tree, 20 Shrub, 30 Grass, 40 Crop, 90 Wetland,
#           50 Built, 60 Bare/sparse, 80 Water, 95 Mangrove, 100 Moss
ESA_REMAP_FROM   = [10, 20, 30, 40, 90, 50, 60, 80]
ESA_REMAP_TO     = [ 0,  1,  1,  1,  1,  2,  2,  3]
ESA_REMAP_NODATA = 255

# Веса классов — пока все 1.0 (стратифицированная выборка уже даёт баланс).
# Если понадобится перевесить классы — менять здесь.
CLASS_WEIGHTS = {i: 1.0 for i in range(N_CLASSES)}

logger.info("-" * 70)
logger.info("Обучающие данные:")
logger.info(f"  Источник:         {TRAINING_SOURCE}")
logger.info(f"  Год:              {TRAINING_YEAR}")
logger.info(f"  Точек на класс:   {POINTS_PER_CLASS} (всего {POINTS_PER_CLASS * N_CLASSES})")
logger.info(f"  Train/Test:       {TRAIN_FRACTION:.0%} / {1 - TRAIN_FRACTION:.0%}")
logger.info(f"  Random seed:      {RANDOM_SEED}")
logger.info(f"  ESA remap:        {dict(zip(ESA_REMAP_FROM, ESA_REMAP_TO))}")
logger.info(f"  Веса классов:     {list(CLASS_WEIGHTS.values())}")

# ------------------------------------------------------------
# 8. Постобработка
# ------------------------------------------------------------
MAJORITY_FILTER = True
FILTER_WINDOW   = 3

logger.info("-" * 70)
logger.info("Постобработка:")
logger.info(f"  Фильтр большинства: {'включён' if MAJORITY_FILTER else 'выключен'}")
if MAJORITY_FILTER:
    logger.info(f"  Окно:               {FILTER_WINDOW}x{FILTER_WINDOW}")

# ------------------------------------------------------------
# 9. Пороги точности
# ------------------------------------------------------------
ACCURACY_THRESHOLD = 0.85
KAPPA_THRESHOLD    = 0.80

logger.info("-" * 70)
logger.info("Пороги точности:")
logger.info(f"  OA:     ≥ {ACCURACY_THRESHOLD}")
logger.info(f"  Kappa:  ≥ {KAPPA_THRESHOLD}")
logger.info(f"  (по спецификации OA=0.84, Kappa=0.79 — принимается)")

# ------------------------------------------------------------
# 10. Проекции
# ------------------------------------------------------------
# UTM zone 37N — метрическая, для расчёта площадей и экспорта PNG.
# Web Mercator — для интерактивных карт (стандарт веб-картографии).
ANALYSIS_CRS = 'EPSG:32637'   # UTM 37N — метры
DISPLAY_CRS  = 'EPSG:3857'    # Web Mercator

logger.info("-" * 70)
logger.info("Проекции:")
logger.info(f"  Расчёты и PNG:  {ANALYSIS_CRS} (UTM zone 37N, метры)")
logger.info(f"  Интерактивно:   {DISPLAY_CRS} (Web Mercator, geemap)")

# ------------------------------------------------------------
# 11. Технические параметры GEE
# ------------------------------------------------------------
GEE_SCALE       = 30         # масштаб Landsat, метров
GEE_TILE_SCALE  = 4          # тайлинг для больших полигонов
GEE_MAX_PIXELS  = int(1e13)  # лимит для reduceRegion

logger.info("-" * 70)
logger.info("Технические параметры GEE:")
logger.info(f"  scale:        {GEE_SCALE} м")
logger.info(f"  tileScale:    {GEE_TILE_SCALE}")
logger.info(f"  maxPixels:    {GEE_MAX_PIXELS:,}")

# ------------------------------------------------------------
# Итог
# ------------------------------------------------------------
logger.success("=" * 70)
logger.success("✅ ПАРАМЕТРЫ АНАЛИЗА ЗАГРУЖЕНЫ")
logger.success("=" * 70)

logger.info("Доступные переменные после этой ячейки:")
logger.info(f"  Классы:           CLASSES ({N_CLASSES}), CLASSES_RU, COLORS, CLASS_WEIGHTS")
logger.info(f"  Признаки:         FEATURES ({len(FEATURES)}), SPECTRAL_BANDS, INDEX_BANDS")
logger.info(f"  Палитры индексов: INDEX_PALETTES, INDEX_VIS")
logger.info(f"  RF:               RF_TREES={RF_TREES}, RF_MIN_LEAF={RF_MIN_LEAF}, RF_BAG_FRACTION={RF_BAG_FRACTION}")
logger.info(f"  Обучение:         TRAINING_SOURCE, POINTS_PER_CLASS={POINTS_PER_CLASS}, TRAIN_FRACTION={TRAIN_FRACTION}")
logger.info(f"  ESA remap:        ESA_REMAP_FROM, ESA_REMAP_TO, ESA_REMAP_NODATA")
logger.info(f"  Постобработка:    MAJORITY_FILTER={MAJORITY_FILTER}, FILTER_WINDOW={FILTER_WINDOW}")
logger.info(f"  Пороги:           ACCURACY_THRESHOLD={ACCURACY_THRESHOLD}, KAPPA_THRESHOLD={KAPPA_THRESHOLD}")
logger.info(f"  Проекции:         ANALYSIS_CRS={ANALYSIS_CRS}, DISPLAY_CRS={DISPLAY_CRS}")
logger.info(f"  GEE:              GEE_SCALE={GEE_SCALE}, GEE_TILE_SCALE={GEE_TILE_SCALE}, GEE_MAX_PIXELS={GEE_MAX_PIXELS:,}")
logger.info(f"  Периоды:          SEASON_START={SEASON_START}, SEASON_END={SEASON_END}")
logger.info(f"  Спутники:         SATELLITES_EARLY, SATELLITES_LATE")
logger.info(f"  Fallback:         MIN_IMAGES_FOR_COMPOSITE={MIN_IMAGES_FOR_COMPOSITE}, FALLBACK_OFFSETS={FALLBACK_OFFSETS}")

In [ ]:
# ============================================================
# ЯЧЕЙКА C2: ПАРАМЕТРЫ ПОСТРОЕНИЯ КАРТ
# ============================================================
# Всё, что нужно для отрисовки карт:
#   • центры, зумы, basemap (пресеты территорий)
#   • стили слоёв (граница, классификация, индексы, разница)
#   • стили для карты проверки границ
#   • конфигурация легенды (для m.add_legend)
#   • экспорт PNG
#   • хелпер-функции для создания карт
#
# Используем только встроенные средства geemap:
#   • m.add_legend()          — легенда
#   • scale_ctrl=True         — масштабная линейка (Leaflet)
#   • add_basemap()           — подложка
#
# Стрелка север-юг для интерактивных карт не нужна —
# она будет добавлена только для статических PNG в блоке I.
# ============================================================

logger.info("=" * 70)
logger.info("ПАРАМЕТРЫ ПОСТРОЕНИЯ КАРТ")
logger.info("=" * 70)

# ------------------------------------------------------------
# 1. Общая конфигурация карт
# ------------------------------------------------------------
MAP_CONFIG = {
    'territories': {
        'Historical': {
            'center': [55.7520, 37.6180],
            'zoom':   10,
            'label':  'Историческая Москва',
        },
        'Greater': {
            'center': [55.6500, 37.3000],
            'zoom':   9,
            'label':  'Большая Москва',
        },
        'Boundaries': {
            'center': [55.70, 37.45],
            'zoom':   9,
            'label':  'Проверка границ',
        },
    },
    'basemap':    'HYBRID',
    'width':      900,
    'height':     600,
    'scale_ctrl': True,        # встроенная масштабная линейка Leaflet
    'default_territory': 'Historical',
}

logger.info("Конфигурация карт:")
for name, cfg in MAP_CONFIG['territories'].items():
    logger.info(f"  {name:<12} центр={cfg['center']} zoom={cfg['zoom']} ({cfg['label']})")
logger.info(f"  Basemap:     {MAP_CONFIG['basemap']}")
logger.info(f"  Scale bar:   {'вкл' if MAP_CONFIG['scale_ctrl'] else 'выкл'}")

# ------------------------------------------------------------
# 2. Стили слоёв
# ------------------------------------------------------------
LAYER_STYLES = {
    'boundary': {
        'color':       'red',
        'width':       2,
        'fillOpacity': 0,
    },
    'classification': {
        'min':     0,
        'max':     N_CLASSES - 1,
        'palette': COLORS,
    },
    'diff_loss':   {'min': 0, 'max': 1, 'palette': ['#d73027']},
    'diff_gain':   {'min': 0, 'max': 1, 'palette': ['#1a9850']},
    'diff_stable': {'min': 0, 'max': 1, 'palette': ['#fffacd']},
    'point': {
        'color': 'red',
        'pointRadius': 8,
    },
}

logger.info(f"Стилей слоёв определено: {len(LAYER_STYLES)}")

# ------------------------------------------------------------
# 2b. Стили для карты проверки границ
# ------------------------------------------------------------
BOUNDARY_LAYER_STYLES = {
    'Historical': {
        'color':       'red',
        'width':       3,
        'fillColor':   'red',
        'fillOpacity': 0.10,
        'label':       'Историческая Москва (до 2012)',
    },
    'Greater': {
        'color':       'blue',
        'width':       3,
        'fillColor':   'blue',
        'fillOpacity': 0.08,
        'label':       'Большая Москва (с Новой Москвой)',
    },
}

logger.info(f"Стилей для карты границ: {len(BOUNDARY_LAYER_STYLES)}")

# ------------------------------------------------------------
# 3. Конфигурация легенды (для m.add_legend)
# ------------------------------------------------------------
LEGEND_CONFIG = {
    'classification': {
        'title':    'Земной покров',
        'keys':     CLASSES_RU,
        'colors':   COLORS,
        'position': 'bottomright',
    },
    'difference': {
        'title':    'Изменения',
        'keys':     ['Потеря растительности', 'Рост растительности', 'Без изменений'],
        'colors':   ['#d73027', '#1a9850', '#fffacd'],
        'position': 'bottomright',
    },
    'boundaries': {
        'title':    'Границы Москвы',
        'keys':     ['Историческая Москва', 'Большая Москва'],
        'colors':   ['red', 'blue'],
        'position': 'bottomright',
    },
}

logger.info("Легенды:")
for name, cfg in LEGEND_CONFIG.items():
    logger.info(f"  {name:<14}: {len(cfg['keys'])} элементов")

# ------------------------------------------------------------
# 4. Экспорт PNG (для блока I)
# ------------------------------------------------------------
MAP_EXPORT = {
    'crs':         ANALYSIS_CRS,
    'dpi':         150,
    'figsize':     (14, 9),
    'bbox_inches': 'tight',
    'facecolor':   'white',
    'font_family': 'DejaVu Sans',
    'title_size':  14,
    'legend_size': 10,
}

# ------------------------------------------------------------
# 4b. Стиль графиков (для статистик, диагностики, отчётов)
# ------------------------------------------------------------
PLOT_STYLE = {
    # Цвета территорий
    'historical_color': '#1f77b4',   # синий
    'greater_color':    '#ff7f0e',   # оранжевый

    # Маркеры по качеству
    'ok_marker':        'green',      # coverage ≥ 95%
    'warn_marker':      'orange',     # coverage 90–95%
    'error_marker':     'red',        # coverage < 90%

    # Аномалии
    'anomaly_marker':   'red',

    # Общие параметры
    'trend_linewidth':  2,
    'grid_alpha':       0.3,
    'marker_size':      8,
    'line_alpha':       0.85,
    'line_width':       1.8,
}

logger.info(f"Стиль графиков PLOT_STYLE: {len(PLOT_STYLE)} параметров")

logger.info(f"Экспорт PNG: CRS={MAP_EXPORT['crs']}, DPI={MAP_EXPORT['dpi']}, размер={MAP_EXPORT['figsize']}")

# ------------------------------------------------------------
# 5. Хелпер-функции для карт
# ------------------------------------------------------------

def create_base_map(territory='Historical'):
    """
    Создаёт базовую интерактивную карту для территории.
    Масштабная линейка включается автоматически (scale_ctrl=True).
    """
    if territory not in MAP_CONFIG['territories']:
        raise ValueError(
            f"Неизвестная территория: {territory}. "
            f"Допустимо: {list(MAP_CONFIG['territories'].keys())}"
        )

    cfg = MAP_CONFIG['territories'][territory]
    m = geemap.Map(
        center=cfg['center'],
        zoom=cfg['zoom'],
        width=MAP_CONFIG['width'],
        height=MAP_CONFIG['height'],
        scale_ctrl=MAP_CONFIG['scale_ctrl'],
    )
    m.add_basemap(MAP_CONFIG['basemap'])
    logger.debug(f"Базовая карта создана: {territory}")
    return m


def add_boundary_layer(m, boundary, label='Граница', style=None):
    """Добавляет слой границы территории."""
    style = style or LAYER_STYLES['boundary']
    m.addLayer(
        boundary,
        {
            'color':       style['color'],
            'width':       style['width'],
            'fillOpacity': style['fillOpacity'],
        },
        label,
    )
    return m


def add_classification_layer(m, classified, year, label=None):
    """Добавляет слой классификации земного покрова."""
    label = label or f'Классификация {year}'
    m.addLayer(classified, LAYER_STYLES['classification'], label)
    return m


def add_index_layer(m, composite, index_name):
    """Добавляет слой индекса (NDVI, NDWI, NDBI, BSI, SAVI)."""
    if index_name not in INDEX_VIS:
        raise ValueError(f"Индекс {index_name} не определён в INDEX_VIS.")
    index_image = composite.select(index_name)
    m.addLayer(index_image, INDEX_VIS[index_name], index_name)
    return m


def add_legend(m, legend_type='classification'):
    """
    Добавляет легенду на карту через встроенный m.add_legend().

    Параметры
    ---------
    m : geemap.Map
    legend_type : str
        Ключ из LEGEND_CONFIG: 'classification', 'difference', 'boundaries'.
    """
    if legend_type not in LEGEND_CONFIG:
        raise ValueError(
            f"Легенда '{legend_type}' не определена. "
            f"Допустимо: {list(LEGEND_CONFIG.keys())}"
        )

    cfg = LEGEND_CONFIG[legend_type]
    m.add_legend(
        title=cfg['title'],
        keys=cfg['keys'],
        colors=cfg['colors'],
        position=cfg['position'],
    )
    logger.debug(f"Легенда '{legend_type}' добавлена")
    return m


def export_map_png(m, output_path, title=None):
    """
    Экспортирует текущую карту в PNG.
    Заглушка: реальный экспорт будет в блоке I.
    """
    from pathlib import Path as _Path
    output_path = _Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    logger.debug(f"Экспорт PNG запланирован: {output_path.name}")


logger.success("Хелперы для карт определены:")
logger.success("  create_base_map, add_boundary_layer, add_classification_layer,")
logger.success("  add_index_layer, add_legend, export_map_png")

# ------------------------------------------------------------
# Итог
# ------------------------------------------------------------
logger.success("=" * 70)
logger.success("✅ ПАРАМЕТРЫ КАРТ ЗАГРУЖЕНЫ")
logger.success("=" * 70)

logger.info("Доступные переменные после этой ячейки:")
logger.info(f"  MAP_CONFIG            — {len(MAP_CONFIG['territories'])} пресетов территорий")
logger.info(f"  LAYER_STYLES          — {len(LAYER_STYLES)} стилей слоёв")
logger.info(f"  BOUNDARY_LAYER_STYLES — {len(BOUNDARY_LAYER_STYLES)} стилей границ")
logger.info(f"  LEGEND_CONFIG         — {len(LEGEND_CONFIG)} легенд")
logger.info(f"  MAP_EXPORT            — параметры PNG (CRS={MAP_EXPORT['crs']})")
logger.info(f"  Хелперы:              create_base_map, add_boundary_layer,")
logger.info(f"                        add_classification_layer, add_index_layer,")
logger.info(f"                        add_legend, export_map_png")
logger.info(f"  PLOT_STYLE            — стиль графиков ({len(PLOT_STYLE)} параметров)")

In [ ]:
# ============================================================
# ЯЧЕЙКА C3: ПАРАМЕТРЫ РАСЧЁТА СТАТИСТИК
# ============================================================
# Всё, что нужно для формирования сводок:
#   • флаги: какие блоки показывать
#   • форматирование чисел
#   • метки точности 🟢🟡🔴
#   • конфигурация накопительной таблицы
#   • хелперы: форматирование матрицы, площадей, метрик
#   • сохранение истории прогонов в 03_Intermediate
# ============================================================

logger.info("=" * 70)
logger.info("ПАРАМЕТРЫ РАСЧЁТА СТАТИСТИК")
logger.info("=" * 70)

# ------------------------------------------------------------
# 1. Что показывать в сводке
# ------------------------------------------------------------
STATS_SHOW = {
    'composite_info':    True,   # сколько снимков, сезон, fallback
    'training_info':     True,   # размеры train/test
    'accuracy':          True,   # OA, Kappa
    'confusion_matrix':  True,   # матрица ошибок
    'per_class_metrics': True,   # precision / recall / F1
    'areas':             True,   # площади по классам
    'timing':            True,   # время по этапам
    'warnings':          True,   # предупреждения
}

logger.info("Разделы сводки:")
for section, enabled in STATS_SHOW.items():
    mark = "✅" if enabled else "⬜"
    logger.info(f"  {mark} {section}")

# ------------------------------------------------------------
# 2. Форматирование
# ------------------------------------------------------------
STATS_FORMAT = {
    'float_precision':   4,      # 0.8418
    'area_precision':    1,      # 1138.5
    'percent_precision': 1,      # 45.2
    'separator_width':   72,     # ширина разделителей
    'table_col_width':   10,     # ширина колонок в таблицах
}

logger.info(f"Форматирование: float={STATS_FORMAT['float_precision']}, "
            f"area={STATS_FORMAT['area_precision']}, "
            f"percent={STATS_FORMAT['percent_precision']}")

# ------------------------------------------------------------
# 3. Метки точности
# ------------------------------------------------------------
ACCURACY_MARKS = {
    'oa': [
        (0.85, '🟢'),   # ≥ 0.85 — хорошо
        (0.75, '🟡'),   # 0.75–0.85 — приемлемо
        (0.0,  '🔴'),   # < 0.75 — плохо
    ],
    'kappa': [
        (0.80, '🟢'),
        (0.70, '🟡'),
        (0.0,  '🔴'),
    ],
}


def get_accuracy_mark(value, metric='oa'):
    """Возвращает метку 🟢/🟡/🔴 для значения метрики."""
    for threshold, mark in ACCURACY_MARKS[metric]:
        if value >= threshold:
            return mark
    return '⚪'


logger.info("Метки точности:")
logger.info(f"  OA:    ≥0.85 🟢 | 0.75–0.85 🟡 | <0.75 🔴")
logger.info(f"  Kappa: ≥0.80 🟢 | 0.70–0.80 🟡 | <0.70 🔴")

# ------------------------------------------------------------
# 4. Файлы для сохранения истории
# ------------------------------------------------------------
STATS_HISTORY_FILE = PATHS['INTERMEDIATE'] / 'run_history.json'
CUMULATIVE_TABLE_FILE = PATHS['INTERMEDIATE'] / 'cumulative_stats.csv'

CUMULATIVE_TABLE_COLUMNS = [
    'territory', 'year', 'OA', 'Kappa',
    'Dense_km2', 'Sparse_km2', 'BuiltUp_km2', 'Water_km2',
    'n_scenes', 'elapsed_sec',
]

logger.info(f"История прогонов:       {STATS_HISTORY_FILE.name}")
logger.info(f"Накопительная таблица:  {CUMULATIVE_TABLE_FILE.name}")
logger.info(f"Колонок в таблице:      {len(CUMULATIVE_TABLE_COLUMNS)}")

# ------------------------------------------------------------
# 5. Хелпер-функции: форматирование
# ------------------------------------------------------------

def format_confusion_matrix(matrix, classes=None):
    """Формирует ASCII-таблицу матрицы ошибок."""
    classes = classes or CLASSES
    fp = STATS_FORMAT['float_precision']
    w = STATS_FORMAT['table_col_width']

    lines = []
    # Заголовок
    header = ' ' * 22 + ''.join(f'{c[:w-2]:>{w}}' for c in [c.split()[0] for c in classes])
    lines.append(header)
    lines.append(' ' * 22 + '-' * (w * len(classes)))

    # Строки
    for i, row in enumerate(matrix):
        row_str = ''.join(f'{int(v):>{w}}' for v in row)
        lines.append(f'  {classes[i][:20]:<20}{row_str}')

    return '\n'.join(lines)


def format_areas_table(class_areas, total_area):
    """Формирует ASCII-таблицу площадей по классам."""
    ap = STATS_FORMAT['area_precision']
    pp = STATS_FORMAT['percent_precision']

    lines = []
    lines.append(f'  {"Класс":<32}{"Площадь":>12}{"Доля":>10}')
    lines.append('  ' + '-' * 54)

    for i in range(N_CLASSES):
        area = class_areas.get(i, 0)
        pct = (area / total_area * 100) if total_area > 0 else 0
        lines.append(
            f'  {CLASSES_RU[i]:<32}{area:>{12}.{ap}f} км²{pct:>{9}.{pp}f}%'
        )

    lines.append('  ' + '-' * 54)
    lines.append(f'  {"ИТОГО":<32}{total_area:>{12}.{ap}f} км²{100.0:>{9}.{pp}f}%')

    return '\n'.join(lines)


def format_accuracy(oa, kappa):
    """Формирует строку с OA и Kappa + метки."""
    fp = STATS_FORMAT['float_precision']
    oa_mark = get_accuracy_mark(oa, 'oa')
    kappa_mark = get_accuracy_mark(kappa, 'kappa')

    lines = []
    lines.append(f'  OA:     {oa:.{fp}f}  {oa_mark}')
    lines.append(f'  Kappa:  {kappa:.{fp}f}  {kappa_mark}')
    return '\n'.join(lines)


def compute_per_class_metrics(matrix):
    """
    Вычисляет precision, recall, F1 для каждого класса из матрицы ошибок.
    matrix[i][j] — истина i, предсказание j.
    """
    n = len(matrix)
    metrics = {}

    for i in range(n):
        tp = matrix[i][i]
        fp = sum(matrix[j][i] for j in range(n) if j != i)  # столбец i без диагонали
        fn = sum(matrix[i][j] for j in range(n) if j != i)  # строка i без диагонали

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0.0)

        metrics[CLASSES[i]] = {
            'precision': precision,
            'recall':    recall,
            'f1':        f1,
        }

    return metrics


def format_per_class_metrics(metrics):
    """Формирует ASCII-таблицу per-class метрик."""
    fp = STATS_FORMAT['float_precision']

    lines = []
    lines.append(f'  {"Класс":<25}{"Precision":>12}{"Recall":>12}{"F1":>12}')
    lines.append('  ' + '-' * 61)

    for class_name, m in metrics.items():
        lines.append(
            f'  {class_name:<25}'
            f'{m["precision"]:>{12}.{fp}f}'
            f'{m["recall"]:>{12}.{fp}f}'
            f'{m["f1"]:>{12}.{fp}f}'
        )

    return '\n'.join(lines)


logger.success("Хелперы форматирования определены:")
logger.success("  format_confusion_matrix, format_areas_table,")
logger.success("  format_accuracy, format_per_class_metrics,")
logger.success("  compute_per_class_metrics, get_accuracy_mark")

# ------------------------------------------------------------
# 6. Хелпер-функции: история и накопительная таблица
# ------------------------------------------------------------

def load_run_history():
    """Загружает историю прогонов из JSON. Если файла нет — пустой список."""
    if STATS_HISTORY_FILE.exists():
        try:
            with open(STATS_HISTORY_FILE, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            logger.warning(f"Не удалось прочитать историю: {e}")
            return []
    return []


def save_run_history(history):
    """Сохраняет историю прогонов в JSON."""
    STATS_HISTORY_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(STATS_HISTORY_FILE, 'w', encoding='utf-8') as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    logger.debug(f"История сохранена: {STATS_HISTORY_FILE.name}")


def append_to_history(territory, year, result):
    """Добавляет запись о прогоне в историю и накопительную таблицу."""
    history = load_run_history()

    # Извлекаем площади по классам
    areas = result.get('areas', {})
    record = {
        'territory':   territory,
        'year':        year,
        'oa':          result.get('oa'),
        'kappa':       result.get('kappa'),
        'total_area':  result.get('total_area'),
        'train_size':  result.get('train_size'),
        'test_size':   result.get('test_size'),
        'n_scenes':    result.get('n_scenes'),
        'elapsed_sec': result.get('elapsed_sec'),
        'areas':       {str(k): v for k, v in areas.items()},
        'timestamp':   datetime.now().isoformat(),
    }
    history.append(record)
    save_run_history(history)

    # Обновляем накопительную CSV
    update_cumulative_csv(record)

    return record


def update_cumulative_csv(record):
    """Обновляет накопительную CSV-таблицу."""
    areas = record.get('areas', {})
    row = {
        'territory':   record['territory'],
        'year':        record['year'],
        'OA':          record.get('oa'),
        'Kappa':       record.get('kappa'),
        'Dense_km2':   areas.get('0', 0),
        'Sparse_km2':  areas.get('1', 0),
        'BuiltUp_km2': areas.get('2', 0),
        'Water_km2':   areas.get('3', 0),
        'n_scenes':    record.get('n_scenes'),
        'elapsed_sec': record.get('elapsed_sec'),
    }

    if CUMULATIVE_TABLE_FILE.exists():
        df = pd.read_csv(CUMULATIVE_TABLE_FILE)
        # Удаляем старую запись с теми же territory+year
        df = df[~((df['territory'] == row['territory']) & (df['year'] == row['year']))]
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])

    df = df.sort_values(['territory', 'year']).reset_index(drop=True)
    df.to_csv(CUMULATIVE_TABLE_FILE, index=False)
    logger.debug(f"Накопительная таблица обновлена: {len(df)} записей")


def show_cumulative_table():
    """Показывает накопительную таблицу в логе."""
    if not CUMULATIVE_TABLE_FILE.exists():
        logger.info("Накопительная таблица пока пуста.")
        return

    df = pd.read_csv(CUMULATIVE_TABLE_FILE)
    logger.info("=" * 70)
    logger.info("НАКОПИТЕЛЬНАЯ ТАБЛИЦА ПРОГОНОВ")
    logger.info("=" * 70)
    logger.info("\n" + df.to_string(index=False))
    logger.info("=" * 70)


logger.success("Хелперы истории определены:")
logger.success("  load_run_history, save_run_history,")
logger.success("  append_to_history, update_cumulative_csv,")
logger.success("  show_cumulative_table")

# ------------------------------------------------------------
# 7. Главная функция: полная сводка
# ------------------------------------------------------------

def format_full_stats(result, territory, year):
    """
    Формирует полную ASCII-сводку по результату одного прогона.

    Параметры
    ---------
    result : dict
        Возвращаемое значение process_year: classified, areas, total_area,
        oa, kappa, train_size, test_size, + опционально n_scenes, elapsed_sec,
        confusion_matrix, composite_info.
    territory : str
    year : int
    """
    W = STATS_FORMAT['separator_width']
    sep = '=' * W
    sub = '-' * W

    lines = []
    lines.append(sep)
    lines.append(f'  ПРОГОН: {territory}, {year}')
    lines.append(sep)

    # Композит
    if STATS_SHOW['composite_info']:
        lines.append(f'  Композит:')
        n_scenes = result.get('n_scenes', '—')
        lines.append(f'    Снимков:  {n_scenes}')
        if result.get('used_year') and result['used_year'] != year:
            lines.append(f'    Fallback: использован {result["used_year"]}')
        lines.append(sub)

    # Обучение
    if STATS_SHOW['training_info']:
        lines.append(f'  Обучение:')
        lines.append(f'    Train: {result.get("train_size", "—")}')
        lines.append(f'    Test:  {result.get("test_size", "—")}')
        lines.append(f'    RF:    {RF_TREES} деревьев, minLeaf={RF_MIN_LEAF}, bag={RF_BAG_FRACTION}')
        lines.append(sub)

    # Точность
    if STATS_SHOW['accuracy']:
        lines.append(format_accuracy(result['oa'], result['kappa']))
        lines.append(sub)

    # Матрица ошибок
    if STATS_SHOW['confusion_matrix'] and 'confusion_matrix' in result:
        lines.append(f'  Матрица ошибок:')
        lines.append(format_confusion_matrix(result['confusion_matrix']))
        lines.append(sub)

    # Per-class метрики
    if STATS_SHOW['per_class_metrics'] and 'confusion_matrix' in result:
        metrics = compute_per_class_metrics(result['confusion_matrix'])
        lines.append(f'  Метрики по классам:')
        lines.append(format_per_class_metrics(metrics))
        lines.append(sub)

    # Площади
    if STATS_SHOW['areas']:
        lines.append(f'  Площади:')
        lines.append(format_areas_table(result['areas'], result['total_area']))
        lines.append(sub)

    # Время
    if STATS_SHOW['timing'] and 'elapsed_sec' in result:
        lines.append(f'  Время: {result["elapsed_sec"]:.1f} с')
        lines.append(sep)

    return '\n'.join(lines)


logger.success("Главный форматтер сводки определён: format_full_stats")

# ------------------------------------------------------------
# Итог
# ------------------------------------------------------------
logger.success("=" * 70)
logger.success("✅ ПАРАМЕТРЫ СТАТИСТИК ЗАГРУЖЕНЫ")
logger.success("=" * 70)

logger.info("Доступные переменные после этой ячейки:")
logger.info(f"  STATS_SHOW          — {sum(STATS_SHOW.values())}/{len(STATS_SHOW)} разделов включено")
logger.info(f"  STATS_FORMAT        — форматирование чисел")
logger.info(f"  ACCURACY_MARKS      — метки 🟢🟡🔴")
logger.info(f"  CUMULATIVE_TABLE_FILE = {CUMULATIVE_TABLE_FILE.name}")
logger.info(f"  Форматтеры:         format_confusion_matrix, format_areas_table,")
logger.info(f"                      format_accuracy, format_per_class_metrics,")
logger.info(f"                      format_full_stats")
logger.info(f"  Метрики:            compute_per_class_metrics, get_accuracy_mark")
logger.info(f"  История:            load_run_history, save_run_history,")
logger.info(f"                      append_to_history, update_cumulative_csv,")
logger.info(f"                      show_cumulative_table")

# БЛОК D. Границы

In [ ]:
# ============================================================
# ЯЧЕЙКА D1: ЗАГРУЗКА ГРАНИЦ МОСКВЫ
# ============================================================
# Загружаем два GeoJSON:
#   • Историческая Москва (до 2012) — moscow_historical.geojson
#   • Большая Москва (с Новой Москвой) — moscow_greater.geojson
#
# Для каждой границы:
#   • проверяем существование файла
#   • загружаем GeoJSON
#   • объединяем в один FeatureCollection через union()
#   • считаем площадь и сравниваем с ожидаемой
# ============================================================

logger.info("=" * 70)
logger.info("ЗАГРУЗКА ГРАНИЦ МОСКВЫ")
logger.info("=" * 70)

# ------------------------------------------------------------
# 1. Пути к файлам
# ------------------------------------------------------------
BOUNDARY_FILES = {
    'Historical': PATHS['BOUNDARIES'] / 'moscow_historical.geojson',
    'Greater':    PATHS['BOUNDARIES'] / 'moscow_greater.geojson',
}

EXPECTED_AREAS = {
    'Historical': 1062,    # км²
    'Greater':    2514,    # км²
}

# ------------------------------------------------------------
# 2. Загрузка каждой границы
# ------------------------------------------------------------
BOUNDARIES = {}
BOUNDARY_AREAS = {}

for name, path in BOUNDARY_FILES.items():
    logger.info("-" * 70)
    logger.info(f"Граница: {name}")
    logger.info(f"  Файл: {path.name}")

    # 2.1. Проверка существования
    if not path.exists():
        logger.error(f"  ❌ Файл не найден: {path}")
        logger.info(f"     Убедитесь, что файл лежит в {PATHS['BOUNDARIES']}")
        raise FileNotFoundError(f"Не найден {path.name}")

    size_kb = path.stat().st_size / 1024
    logger.info(f"  Размер файла: {size_kb:.1f} КБ")

    # 2.2. Загрузка GeoJSON
    try:
        with open(path, 'r', encoding='utf-8') as f:
            geojson = json.load(f)
        logger.info(f"  ✅ GeoJSON прочитан")
    except Exception as e:
        logger.error(f"  ❌ Ошибка чтения GeoJSON: {e}")
        raise

    # 2.3. Преобразование в ee.FeatureCollection + union
    try:
        fc = ee.FeatureCollection(geojson)
        n_features = fc.size().getInfo()
        logger.info(f"  Объектов в коллекции: {n_features}")

        boundary = fc.union()
        logger.info(f"  ✅ Границы объединены через union()")
    except Exception as e:
        logger.error(f"  ❌ Ошибка при создании FeatureCollection: {e}")
        raise

    # 2.4. Расчёт площади в метрической проекции (UTM 37N)
    try:
        area_m2 = boundary.geometry().area(maxError=1).getInfo()
        area_km2 = area_m2 / 1e6
        BOUNDARY_AREAS[name] = area_km2

        expected = EXPECTED_AREAS[name]
        diff_pct = abs(area_km2 - expected) / expected * 100

        logger.info(f"  Площадь: {area_km2:,.1f} км²")
        logger.info(f"  Ожидалось: {expected:,} км² (расхождение {diff_pct:.2f}%)")

        if diff_pct < 2.0:
            logger.success(f"  ✅ Площадь в пределах допуска")
        elif diff_pct < 5.0:
            logger.warning(f"  ⚠️  Площадь немного отличается от ожидаемой")
        else:
            logger.warning(f"  ⚠️  Площадь сильно отличается — проверьте GeoJSON")

    except Exception as e:
        logger.error(f"  ❌ Ошибка при расчёте площади: {e}")
        raise

    # 2.5. Сохраняем
    BOUNDARIES[name] = boundary

# ------------------------------------------------------------
# 3. Проверка вложенности: Greater ⊇ Historical
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Проверка вложенности границ:")

# Проверяем пересечение: должны пересекаться
try:
    intersection = BOUNDARIES['Historical'].geometry().intersection(
        BOUNDARIES['Greater'].geometry(), maxError=1
    )
    intersection_area = intersection.area(maxError=1).getInfo() / 1e6
    hist_area = BOUNDARY_AREAS['Historical']

    overlap_pct = intersection_area / hist_area * 100 if hist_area > 0 else 0

    logger.info(f"  Площадь пересечения: {intersection_area:,.1f} км²")
    logger.info(f"  Доля исторической Москвы, попадающая в Большую: {overlap_pct:.2f}%")

    if overlap_pct > 95:
        logger.success(f"  ✅ Историческая Москва почти полностью внутри Большой")
    elif overlap_pct > 50:
        logger.warning(f"  ⚠️  Пересечение частичное — проверьте границы")
    else:
        logger.error(f"  ❌ Границы почти не пересекаются — что-то не так")

except Exception as e:
    logger.warning(f"  ⚠️  Не удалось проверить пересечение: {e}")

# ------------------------------------------------------------
# 4. Итог
# ------------------------------------------------------------
logger.success("=" * 70)
logger.success("✅ ГРАНИЦЫ ЗАГРУЖЕНЫ")
logger.success("=" * 70)

logger.info("Доступные переменные после этой ячейки:")
logger.info(f"  BOUNDARIES       — словарь ee.FeatureCollection:")
for name, b in BOUNDARIES.items():
    logger.info(f"     ['{name}']")
logger.info(f"  BOUNDARY_AREAS   = {BOUNDARY_AREAS}")
logger.info(f"  BOUNDARY_FILES   — пути к исходным GeoJSON")
logger.info(f"  EXPECTED_AREAS   — ожидаемые площади из спецификации")

# Добавляем удобные алиасы
historical_boundary = BOUNDARIES['Historical']
greater_moscow_boundary = BOUNDARIES['Greater']

logger.info(f"  historical_boundary      = BOUNDARIES['Historical']")
logger.info(f"  greater_moscow_boundary  = BOUNDARIES['Greater']")

In [ ]:
# ============================================================
# ЯЧЕЙКА D2: ВИЗУАЛИЗАЦИЯ ГРАНИЦ
# ============================================================
# Интерактивная карта с двумя слоями границ.
# Все параметры (центр, зум, basemap, стили, легенда)
# берутся из C2 — здесь только вызовы хелперов.
# ============================================================

logger.info("=" * 70)
logger.info("ВИЗУАЛИЗАЦИЯ ГРАНИЦ")
logger.info("=" * 70)

# ------------------------------------------------------------
# 1. Базовая карта (центр/зум/basemap — из MAP_CONFIG)
# ------------------------------------------------------------
m = create_base_map('Boundaries')
logger.info(f"  ✅ Базовая карта создана (пресет 'Boundaries')")

# ------------------------------------------------------------
# 2. Слои границ (стили — из BOUNDARY_LAYER_STYLES)
# ------------------------------------------------------------
# Порядок: сначала Greater (нижний слой), потом Historical (верхний),
# чтобы красный контур рисовался поверх синей заливки.

# Greater — нижний
style_g = BOUNDARY_LAYER_STYLES['Greater']
m.addLayer(
    BOUNDARIES['Greater'],
    {
        'color':       style_g['color'],
        'width':       style_g['width'],
        'fillColor':   style_g['fillColor'],
        'fillOpacity': style_g['fillOpacity'],
    },
    style_g['label'],
)
logger.info(f"  ✅ Слой 'Greater' добавлен (синий)")

# Historical — верхний
style_h = BOUNDARY_LAYER_STYLES['Historical']
m.addLayer(
    BOUNDARIES['Historical'],
    {
        'color':       style_h['color'],
        'width':       style_h['width'],
        'fillColor':   style_h['fillColor'],
        'fillOpacity': style_h['fillOpacity'],
    },
    style_h['label'],
)
logger.info(f"  ✅ Слой 'Historical' добавлен (красный)")

# ------------------------------------------------------------
# 3. Легенда с площадями (через универсальный хелпер)
# ------------------------------------------------------------
new_moscow_area = BOUNDARY_AREAS['Greater'] - BOUNDARY_AREAS['Historical']

add_legend(m, legend_type='boundaries')
logger.info(f"  ✅ Легенда добавлена")

# ------------------------------------------------------------
# 4. Итог
# ------------------------------------------------------------
logger.success("✅ Карта построена")
logger.info("=" * 70)
logger.info("Что проверить на карте:")
logger.info("  1) Красный контур (Историческая) должен лежать ВНУТРИ синего (Большая)")
logger.info("  2) Слева видна Новая Москва (юго-запад) — она только в синем контуре")
logger.info("  3) Переключение слоёв в панели справа — оба слоя должны быть видны")
logger.info("  4) Зум колёсиком — проверьте детализацию границ")
logger.info("=" * 70)

m

# БЛОК E. Функции обработки

In [ ]:
# ============================================================
# ЯЧЕЙКА E1: ФУНКЦИИ МАСКИ, ГАРМОНИЗАЦИИ И КАЛИБРОВКИ
# ============================================================
# Определяем функции для базовой предобработки снимков Landsat:
#   1) mask_landsat_sr        — маска облаков, теней, полос SLC-off
#   2) harmonize_landsat_bands — переименование каналов в общие имена
#   3) calibrate_landsat_sr    — перевод DN → отражательная способность
#   4) preprocess_landsat      — обёртка: mask → harmonize → calibrate
#
# Калибровка Collection 2 Level 2:
#   reflectance = DN * 0.0000275 - 0.2
#   Единый коэффициент для L5/L7/L8/L9.
# ============================================================

logger.info("=" * 70)
logger.info("ОПРЕДЕЛЕНИЕ ФУНКЦИЙ ПРЕДОБРАБОТКИ")
logger.info("=" * 70)

# ------------------------------------------------------------
# Константы калибровки (Collection 2 Level 2)
# ------------------------------------------------------------
SR_SCALE_FACTOR = 0.0000275
SR_OFFSET       = -0.2

logger.info(f"Калибровка: reflectance = DN * {SR_SCALE_FACTOR} + ({SR_OFFSET})")
logger.info(f"  (единый коэффициент для L5/L7/L8/L9)")

# ------------------------------------------------------------
# 1. Маска дефектов через QA_PIXEL
# ------------------------------------------------------------
def mask_landsat_sr(image):
    """
    Маскирует облака, тени и полосы SLC-off через канал QA_PIXEL.

    Биты:
      0 — заполненные пиксели (Landsat 7 SLC-off)
      1 — расширенная облачность (dilated cloud)
      3 — облака
      4 — тень от облаков

    Для L5/L8/L9 бит 0 всегда 0 — маска их не портит.
    Для L7 убирает ~22% пикселей (полосы SLC-off).
    """
    qa = image.select('QA_PIXEL')

    mask = (
        qa.bitwiseAnd(1 << 0).eq(0)     # fill
        .And(qa.bitwiseAnd(1 << 1).eq(0))  # dilated cloud
        .And(qa.bitwiseAnd(1 << 3).eq(0))  # cloud
        .And(qa.bitwiseAnd(1 << 4).eq(0))  # cloud shadow
    )

    return image.updateMask(mask)


# ------------------------------------------------------------
# 2. Гармонизация каналов (переименование)
# ------------------------------------------------------------
# Соответствие каналов между L5/7 и L8/9:
#   blue   ← SR_B1 (L5/7) / SR_B2 (L8/9)
#   green  ← SR_B2 (L5/7) / SR_B3 (L8/9)
#   red    ← SR_B3 (L5/7) / SR_B4 (L8/9)
#   nir    ← SR_B4 (L5/7) / SR_B5 (L8/9)
#   swir1  ← SR_B5 (L5/7) / SR_B6 (L8/9)
#   swir2  ← SR_B7 (общий для всех)

def harmonize_landsat_bands(image):
    """
    Переименовывает каналы L5/L7 и L8/L9 в общие имена.
    НЕ меняет значения пикселей.
    """
    spacecraft = ee.String(image.get('SPACECRAFT_ID'))
    is_l57 = ee.List(['LANDSAT_5', 'LANDSAT_7']).contains(spacecraft)

    # L5/L7: SR_B1..SR_B5, SR_B7
    l57_bands = image.select(
        ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'],
        ['blue',  'green', 'red',  'nir',  'swir1', 'swir2']
    )

    # L8/L9: SR_B2..SR_B7
    l89_bands = image.select(
        ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
        ['blue',  'green', 'red',  'nir',  'swir1', 'swir2']
    )

    return ee.Image(ee.Algorithms.If(is_l57, l57_bands, l89_bands))


# ------------------------------------------------------------
# 3. Калибровка значений (DN → reflectance)
# ------------------------------------------------------------
def calibrate_landsat_sr(image):
    """
    Переводит DN (Digital Number) в отражательную способность.
    Формула Collection 2 Level 2: reflectance = DN * 0.0000275 - 0.2.
    Единый коэффициент для всех датчиков.
    """
    return image.multiply(SR_SCALE_FACTOR).add(SR_OFFSET)


# ------------------------------------------------------------
# 4. Полная предобработка: mask → harmonize → calibrate
# ------------------------------------------------------------
def preprocess_landsat(image):
    """
    Полная предобработка одного снимка Landsat:
      1) маска дефектов (QA_PIXEL)
      2) гармонизация каналов (переименование)
      3) калибровка значений (DN → reflectance)
    Возвращает изображение с 6 каналами: blue, green, red, nir, swir1, swir2.
    """
    return calibrate_landsat_sr(harmonize_landsat_bands(mask_landsat_sr(image)))


logger.success("Определены функции:")
logger.success("  mask_landsat_sr        — маска QA_PIXEL")
logger.success("  harmonize_landsat_bands — переименование каналов")
logger.success("  calibrate_landsat_sr    — калибровка значений")
logger.success("  preprocess_landsat      — полная предобработка")

logger.info("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА E2: ТЕСТ ПРЕДОБРАБОТКИ НА L5 И L8
# ============================================================
# Проверяем на реальных снимках:
#   1) Находим по одному снимку L5 (2007) и L8 (2020) над Москвой
#   2) Применяем preprocess_landsat к каждому
#   3) Считаем статистику red, nir, swir1
#   4) Строим гистограммы через ee.Reducer.histogram (надёжнее sample)
#   5) Показываем карту true color для обоих
#
# Цель: убедиться, что значения находятся в разумном диапазоне
# отражательности (примерно −0.2…0.5), а не в DN (0…65535).
# ============================================================

logger.info("=" * 70)
logger.info("ТЕСТ ПРЕДОБРАБОТКИ НА L5 И L8")
logger.info("=" * 70)

# ------------------------------------------------------------
# 1. Поиск снимков
# ------------------------------------------------------------
logger.info("Ищу тестовые снимки над Москвой...")

SEARCH_REGION = historical_boundary


def find_one_scene(collection_id, date_start, date_end,
                   cloud_thresholds=(5, 10, 20, 40, None)):
    """Находит один снимок, последовательно ослабляя порог облачности."""
    for th in cloud_thresholds:
        base = (
            ee.ImageCollection(collection_id)
            .filterBounds(SEARCH_REGION)
            .filterDate(date_start, date_end)
        )
        candidate = base.filter(ee.Filter.lt('CLOUD_COVER', th)) if th is not None else base
        n = candidate.size().getInfo()
        if n > 0:
            return candidate.first(), n, th
    return None, 0, None


# L5 — лето 2007
l5_scene, l5_n, l5_th = find_one_scene(
    'LANDSAT/LT05/C02/T1_L2', '2007-06-01', '2007-09-01'
)
if l5_scene is None:
    logger.error("❌ Не найден снимок L5 за лето 2007 над Москвой")
    raise RuntimeError("Тест не может быть выполнен без снимка L5.")

l5_info = l5_scene.getInfo()
logger.info(f"  L5: {l5_info['id']}")
logger.info(f"      дата={l5_info['properties']['DATE_ACQUIRED']}, "
            f"облачность={l5_info['properties'].get('CLOUD_COVER', '?')}%, "
            f"порог={l5_th}%, всего найдено={l5_n}")

# L8 — лето 2020
l8_scene, l8_n, l8_th = find_one_scene(
    'LANDSAT/LC08/C02/T1_L2', '2020-06-01', '2020-09-01'
)
if l8_scene is None:
    logger.error("❌ Не найден снимок L8 за лето 2020 над Москвой")
    raise RuntimeError("Тест не может быть выполнен без снимка L8.")

l8_info = l8_scene.getInfo()
logger.info(f"  L8: {l8_info['id']}")
logger.info(f"      дата={l8_info['properties']['DATE_ACQUIRED']}, "
            f"облачность={l8_info['properties'].get('CLOUD_COVER', '?')}%, "
            f"порог={l8_th}%, всего найдено={l8_n}")

logger.success("Снимки найдены.")

# ------------------------------------------------------------
# 2. Применение preprocess_landsat
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Применяю preprocess_landsat...")


def count_valid_pixels(image, region):
    """Считает количество валидных пикселей в регионе."""
    count = image.select(['red']).reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=region,
        scale=30,
        maxPixels=1e9,
        bestEffort=True
    ).getInfo()
    return list(count.values())[0] if count else 0


l5_raw_count = l5_scene.select(['SR_B3']).reduceRegion(
    reducer=ee.Reducer.count(), geometry=SEARCH_REGION,
    scale=30, maxPixels=1e9, bestEffort=True
).getInfo()
l5_raw_count = list(l5_raw_count.values())[0]

l8_raw_count = l8_scene.select(['SR_B4']).reduceRegion(
    reducer=ee.Reducer.count(), geometry=SEARCH_REGION,
    scale=30, maxPixels=1e9, bestEffort=True
).getInfo()
l8_raw_count = list(l8_raw_count.values())[0]

l5_processed = preprocess_landsat(l5_scene).clip(SEARCH_REGION)
l8_processed = preprocess_landsat(l8_scene).clip(SEARCH_REGION)

l5_proc_count = count_valid_pixels(l5_processed, SEARCH_REGION)
l8_proc_count = count_valid_pixels(l8_processed, SEARCH_REGION)

l5_masked_pct = (1 - l5_proc_count / l5_raw_count) * 100 if l5_raw_count > 0 else 0
l8_masked_pct = (1 - l8_proc_count / l8_raw_count) * 100 if l8_raw_count > 0 else 0

logger.info(f"  L5: пикселей до={l5_raw_count:,}, после маски={l5_proc_count:,} "
            f"({l5_masked_pct:.1f}% замаскировано)")
logger.info(f"  L8: пикселей до={l8_raw_count:,}, после маски={l8_proc_count:,} "
            f"({l8_masked_pct:.1f}% замаскировано)")

# Проверка, что маска не убила всё
for name, pct in [('L5', l5_masked_pct), ('L8', l8_masked_pct)]:
    if pct > 90:
        logger.warning(f"  ⚠️  {name}: замаскировано {pct:.1f}% — слишком много, проверьте QA_PIXEL")
    elif pct < 0.5:
        logger.info(f"  ℹ️  {name}: замаскировано {pct:.1f}% — очень мало (возможно, ясный снимок)")

# Проверка имён каналов
logger.info(f"  Каналы L5 после обработки: {l5_processed.bandNames().getInfo()}")
logger.info(f"  Каналы L8 после обработки: {l8_processed.bandNames().getInfo()}")

logger.success("Предобработка применена.")

# ------------------------------------------------------------
# 3. Статистика по каналам
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Статистика по каналам (min / max / mean):")


def get_band_stats(image, bands=('red', 'nir', 'swir1')):
    """Возвращает min/max/mean для указанных каналов."""
    stats = image.select(list(bands)).reduceRegion(
        reducer=ee.Reducer.minMax().combine(
            ee.Reducer.mean(), sharedInputs=True
        ),
        geometry=SEARCH_REGION,
        scale=30,
        maxPixels=1e9,
        bestEffort=True
    ).getInfo()
    return stats


l5_stats = get_band_stats(l5_processed)
l8_stats = get_band_stats(l8_processed)

logger.info(f"  {'Канал':<10}{'L5 min':>12}{'L5 max':>12}{'L5 mean':>12}"
            f"{'L8 min':>12}{'L8 max':>12}{'L8 mean':>12}")
logger.info(f"  {'-' * 70}")
for band in ['red', 'nir', 'swir1']:
    logger.info(
        f"  {band:<10}"
        f"{l5_stats.get(band + '_min', 0):>12.4f}"
        f"{l5_stats.get(band + '_max', 0):>12.4f}"
        f"{l5_stats.get(band + '_mean', 0):>12.4f}"
        f"{l8_stats.get(band + '_min', 0):>12.4f}"
        f"{l8_stats.get(band + '_max', 0):>12.4f}"
        f"{l8_stats.get(band + '_mean', 0):>12.4f}"
    )

# Проверка диапазона
for name, stats in [('L5', l5_stats), ('L8', l8_stats)]:
    red_mean = stats.get('red_mean', 0)
    nir_mean = stats.get('nir_mean', 0)

    if -0.5 < red_mean < 0.5 and -0.5 < nir_mean < 0.7:
        logger.success(f"  ✅ {name}: значения в диапазоне отражательности "
                       f"(red={red_mean:.3f}, nir={nir_mean:.3f})")
    else:
        logger.warning(f"  ⚠️  {name}: значения выглядят подозрительно "
                       f"(red={red_mean:.3f}, nir={nir_mean:.3f})")

# ------------------------------------------------------------
# 4. Гистограммы через ee.Reducer.histogram
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Считаю гистограммы через ee.Reducer.histogram...")


def get_band_histogram(image, band, region, scale=30, max_buckets=50):
    """
    Считает гистограмму одного канала через ee.Reducer.histogram.
    Возвращает (bucket_means, histogram_counts).

    Надёжнее sample(), потому что работает через reduceRegion
    и учитывает ВСЕ валидные пиксели в регионе, а не случайную выборку.
    """
    hist_dict = image.select(band).reduceRegion(
        reducer=ee.Reducer.histogram(maxBuckets=max_buckets),
        geometry=region,
        scale=scale,
        maxPixels=1e9,
        bestEffort=True
    ).getInfo()

    band_hist = hist_dict.get(band) or {}
    return band_hist.get('bucketMeans', []), band_hist.get('histogram', [])


# Red
l5_red_means, l5_red_counts = get_band_histogram(l5_processed, 'red', historical_boundary)
l8_red_means, l8_red_counts = get_band_histogram(l8_processed, 'red', historical_boundary)

# NIR
l5_nir_means, l5_nir_counts = get_band_histogram(l5_processed, 'nir', historical_boundary)
l8_nir_means, l8_nir_counts = get_band_histogram(l8_processed, 'nir', historical_boundary)

logger.info(f"  L5 red: {len(l5_red_means)} бинов, всего пикселей={sum(l5_red_counts):,}")
logger.info(f"  L8 red: {len(l8_red_means)} бинов, всего пикселей={sum(l8_red_counts):,}")
logger.info(f"  L5 nir: {len(l5_nir_means)} бинов, всего пикселей={sum(l5_nir_counts):,}")
logger.info(f"  L8 nir: {len(l8_nir_means)} бинов, всего пикселей={sum(l8_nir_counts):,}")

# ------------------------------------------------------------
# 5. Построение гистограмм
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Red
if l5_red_means:
    axes[0].bar(l5_red_means, l5_red_counts, width=0.005, alpha=0.6,
                label=f'L5 (2007), n={sum(l5_red_counts):,}', color='#1f77b4')
if l8_red_means:
    axes[0].bar(l8_red_means, l8_red_counts, width=0.005, alpha=0.6,
                label=f'L8 (2020), n={sum(l8_red_counts):,}', color='#ff7f0e')
axes[0].set_title('Канал RED', fontsize=13)
axes[0].set_xlabel('Отражательность')
axes[0].set_ylabel('Количество пикселей')
axes[0].legend()
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5)

# NIR
if l5_nir_means:
    axes[1].bar(l5_nir_means, l5_nir_counts, width=0.005, alpha=0.6,
                label=f'L5 (2007), n={sum(l5_nir_counts):,}', color='#1f77b4')
if l8_nir_means:
    axes[1].bar(l8_nir_means, l8_nir_counts, width=0.005, alpha=0.6,
                label=f'L8 (2020), n={sum(l8_nir_counts):,}', color='#ff7f0e')
axes[1].set_title('Канал NIR', fontsize=13)
axes[1].set_xlabel('Отражательность')
axes[1].set_ylabel('Количество пикселей')
axes[1].legend()
axes[1].axvline(0, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Сравнение отражательности L5 vs L8 после калибровки', fontsize=14)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 6. Карта true color
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Строю карту true color для L5 и L8...")

true_color_vis = {'min': 0.0, 'max': 0.3, 'gamma': 1.4}

l5_rgb = l5_processed.select(['red', 'green', 'blue'])
l8_rgb = l8_processed.select(['red', 'green', 'blue'])

m = create_base_map('Historical')
add_boundary_layer(m, historical_boundary, label='Историческая Москва')

m.addLayer(l5_rgb, true_color_vis, f'L5 true color ({l5_info["properties"]["DATE_ACQUIRED"]})')
m.addLayer(l8_rgb, true_color_vis, f'L8 true color ({l8_info["properties"]["DATE_ACQUIRED"]})')

logger.success("Карта построена.")

# ------------------------------------------------------------
# 7. Итог
# ------------------------------------------------------------
logger.success("=" * 70)
logger.success("✅ ТЕСТ ПРЕДОБРАБОТКИ ПРОЙДЕН")
logger.success("=" * 70)
logger.info("Что мы проверили:")
logger.info("  1) Функции применимы к L5 и L8")
logger.info("  2) Имена каналов совпадают: blue, green, red, nir, swir1, swir2")
logger.info("  3) Значения в диапазоне отражательности (не DN)")
logger.info("  4) Маска работает и не убивает все пиксели")
logger.info("  5) True color выглядит как реальное изображение")
logger.info("")
logger.info("Доступные переменные после этой ячейки:")
logger.info("  l5_scene, l8_scene         — тестовые снимки")
logger.info("  l5_processed, l8_processed — после preprocess_landsat")
logger.info("  l5_stats, l8_stats         — статистика по каналам")
logger.info("  Гистограммы: l5_red_means/counts, l8_red_means/counts,")
logger.info("               l5_nir_means/counts, l8_nir_means/counts")

m

In [ ]:
# ============================================================
# ЯЧЕЙКА E3: ФУНКЦИЯ МЕДИАННОГО КОМПОЗИТА С 15 ПРИЗНАКАМИ
# ============================================================
# Создаёт медианный композит за лето года:
#   1) Фильтр по границе и дате (июнь–август)
#   2) Препроцессинг (mask + harmonize + calibrate)
#   3) Медианное усреднение (устраняет облака)
#   4) Fallback на соседние годы, если снимков < MIN_IMAGES_FOR_COMPOSITE
#   5) Вычисление 7 индексов: NDVI, NDWI, NDBI, BSI, SAVI, NDMI, MNDWI
#   6) Текстурные признаки GLCM: NIR_contrast, NIR_ent
#
# Возвращает: (composite, info_dict) или (None, info_dict)
#   composite — ee.Image с 15 признаками
#   info_dict — метаданные (n_scenes, used_year, fallback_used)
#
# ВАЖНО про GLCM:
#   • glcmTexture работает только с 8-битными изображениями (0..255)
#   • Параметры передаются словарём СО СТРОКОВЫМИ ключами: {'size': 3}
#   • Возвращаемые каналы имеют СОКРАЩЁННЫЕ суффиксы:
#       _asm, _contrast, _corr, _var, _idm, _savg, _ent,
#       _dvar, _diss, _inertia, _shade, _prom, _maxcorr
#     То есть энтропия называется 'nir_ent', а НЕ 'nir_entropy'!
# ============================================================

logger.info("=" * 70)
logger.info("ОПРЕДЕЛЕНИЕ ФУНКЦИИ create_composite (15 признаков)")
logger.info("=" * 70)


def create_composite(collection, boundary, year, territory_name,
                     min_images=None, verbose=True):
    """
    Создаёт медианный композит Landsat за лето указанного года
    с 15 признаками: 6 каналов + 7 индексов + 2 текстурных признака.

    Параметры
    ---------
    collection : ee.ImageCollection
        Коллекция Landsat (L5+L7 или L8+L9).
    boundary : ee.FeatureCollection или ee.Geometry
        Границы территории.
    year : int
        Целевой год.
    territory_name : str
        'Historical' или 'Greater' — для логирования.
    min_images : int, опционально
        Минимум снимков. По умолчанию — MIN_IMAGES_FOR_COMPOSITE из C1.
    verbose : bool
        Логировать ли подробно.

    Возвращает
    ----------
    (composite, info) : tuple
        composite — ee.Image с 15 признаками, или None при провале.
        info — dict с метаданными.
    """
    min_images = min_images if min_images is not None else MIN_IMAGES_FOR_COMPOSITE

    def _filter_collection(y):
        """Фильтр коллекции по региону и дате + препроцессинг."""
        return (
            collection
            .filterBounds(boundary)
            .filterDate(f'{y}-{SEASON_START}', f'{y}-{SEASON_END}')
            .map(preprocess_landsat)
        )

    # --- 1. Собираем снимки целевого года ---
    filtered = _filter_collection(year)
    count = filtered.size().getInfo()

    info = {
        'target_year': year,
        'used_year':   year,
        'n_scenes':    count,
        'fallback_used': False,
    }

    if verbose:
        logger.info(f"[{territory_name}] Год {year}: найдено {count} снимков.")

    # --- 2. Fallback на соседние годы ---
    if count < min_images:
        if verbose:
            logger.warning(
                f"[{territory_name}] Год {year}: снимков мало (<{min_images}). "
                f"Ищу соседние годы: {FALLBACK_OFFSETS}..."
            )

        for offset in FALLBACK_OFFSETS:
            adj_year = year + offset
            adj_filtered = _filter_collection(adj_year)
            adj_count = adj_filtered.size().getInfo()

            if verbose:
                logger.info(
                    f"[{territory_name}] Год {year}: проверяю {adj_year} — "
                    f"{adj_count} снимков."
                )

            if adj_count >= min_images:
                if verbose:
                    logger.info(
                        f"[{territory_name}] Год {year}: использую {adj_year} "
                        f"({adj_count} снимков)."
                    )
                filtered = adj_filtered
                count = adj_count
                info['used_year'] = adj_year
                info['n_scenes'] = adj_count
                info['fallback_used'] = True
                break

        if count == 0:
            logger.error(f"[{territory_name}] Год {year}: нет снимков совсем.")
            return None, info

    # --- 3. Медианный композит ---
    composite = filtered.median().clip(boundary)

    # --- 4. Проверяем, что композит содержит 6 каналов ---
    band_names = composite.bandNames().getInfo()
    expected_bands = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']
    missing = [b for b in expected_bands if b not in band_names]
    if missing:
        logger.error(
            f"[{territory_name}] Год {year}: в композите нет каналов {missing}. "
            f"Есть: {band_names}"
        )
        return None, info

    # --- 5. Вычисляем индексы (с ограничением диапазонов) ---
    # NDVI, NDWI, NDBI, SAVI, NDMI, MNDWI математически ∈ [-1, 1].
    # BSI ∈ [-1, 1]. Но после калибровки (offset -0.2) знаменатель
    # может быть близок к нулю — деление даёт выбросы.
    # clamp() обрезает экстремумы без искажения данных.
    blue  = composite.select('blue')
    green = composite.select('green')
    red   = composite.select('red')
    nir   = composite.select('nir')
    swir1 = composite.select('swir1')

    # NDVI: (NIR - Red) / (NIR + Red)
    ndvi = nir.subtract(red).divide(nir.add(red)).clamp(-1, 1).rename('NDVI')

    # NDWI: (Green - NIR) / (Green + NIR)
    ndwi = green.subtract(nir).divide(green.add(nir)).clamp(-1, 1).rename('NDWI')

    # NDBI: (SWIR1 - NIR) / (SWIR1 + NIR)
    ndbi = swir1.subtract(nir).divide(swir1.add(nir)).clamp(-1, 1).rename('NDBI')

    # BSI: ((SWIR1 + Red) - (NIR + Blue)) / ((SWIR1 + Red) + (NIR + Blue))
    bsi = (
        swir1.add(red).subtract(nir.add(blue))
        .divide(swir1.add(red).add(nir).add(blue))
        .clamp(-1, 1)
        .rename('BSI')
    )

    # SAVI: ((NIR - Red) / (NIR + Red + 0.5)) * 1.5
    savi = (
        nir.subtract(red)
        .divide(nir.add(red).add(0.5))
        .multiply(1.5)
        .clamp(-1, 1)
        .rename('SAVI')
    )

    # --- 5b. Новые индексы влажности и воды ---
    # NDMI: чувствителен к влажности растительности и почвы
    ndmi = nir.subtract(swir1).divide(nir.add(swir1)).clamp(-1, 1).rename('NDMI')

    # MNDWI: улучшенный водный индекс для городских условий
    mndwi = green.subtract(swir1).divide(green.add(swir1)).clamp(-1, 1).rename('MNDWI')

    # --- 5c. Текстурные признаки (GLCM) ---
    # ВАЖНО:
    #   1) glcmTexture работает только с 8-битными изображениями (0..255)
    #   2) Параметры передаются словарём СО СТРОКОВЫМИ ключами
    #   3) Возвращаемые каналы имеют СОКРАЩЁННЫЕ имена:
    #      _asm, _contrast, _corr, _var, _idm, _savg, _ent, ...
    #      Энтропия = 'nir_ent', а не 'nir_entropy'!
    nir_8bit = composite.select('nir').multiply(255).clamp(0, 255).toByte()
    glcm = nir_8bit.glcmTexture(size=3)

    # Проверка, какие каналы вернул glcmTexture (для отладки)
    glcm_bands = glcm.bandNames().getInfo()
    if verbose:
        logger.debug(f"[{territory_name}] Год {year}: GLCM bands = {glcm_bands}")

    # Используем ПРАВИЛЬНЫЕ имена: _contrast и _ent
    nir_contrast = glcm.select('nir_contrast').rename('NIR_contrast')
    nir_entropy  = glcm.select('nir_ent').rename('NIR_entropy')

    # Добавляем все признаки к композиту
    composite = composite.addBands([
        ndvi,          # индекс
        ndwi,          # индекс
        ndbi,          # индекс
        bsi,           # индекс
        savi,          # индекс
        ndmi,          # индекс
        mndwi,         # индекс
        nir_contrast,  # текстура
        nir_entropy,   # текстура
    ])

    # --- 6. Финальная проверка ---
    final_bands = composite.bandNames().getInfo()
    if len(final_bands) != len(FEATURES):
        logger.warning(
            f"[{territory_name}] Год {year}: в композите {len(final_bands)} "
            f"каналов, ожидалось {len(FEATURES)}. Список: {final_bands}"
        )
    else:
        if verbose:
            logger.debug(
                f"[{territory_name}] Год {year}: все {len(final_bands)} каналов на месте."
            )

    if verbose:
        logger.success(
            f"[{territory_name}] Год {year}: композит создан из {count} снимков "
            f"({len(final_bands)} признаков)."
        )

    return composite, info


logger.success("Функция create_composite определена (15 признаков).")
logger.info("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА E4: ТЕСТ КОМПОЗИТА
# ============================================================
# Проверяем create_composite на двух годах:
#   • 2000 (Landsat 5/7) — ранний период
#   • 2020 (Landsat 8/9) — поздний период
#
# Что проверяем:
#   1) Композит создаётся, возвращает 11 признаков
#   2) Fallback работает (если снимков мало)
#   3) Индексы в разумных диапазонах (NDVI: −1…1, NDWI: −1…1 и т.д.)
#   4) Карты NDVI и NDBI визуально осмысленны
# ============================================================

logger.info("=" * 70)
logger.info("ТЕСТ КОМПОЗИТА")
logger.info("=" * 70)

# ------------------------------------------------------------
# 1. Собираем коллекции для обоих периодов
# ------------------------------------------------------------
def build_collection(satellite_ids):
    """Объединяет несколько коллекций Landsat в одну."""
    col = ee.ImageCollection(satellite_ids[0])
    for cid in satellite_ids[1:]:
        col = col.merge(ee.ImageCollection(cid))
    return col


collection_early = build_collection(SATELLITES_EARLY)
collection_late = build_collection(SATELLITES_LATE)

logger.info(f"Коллекция early: {SATELLITES_EARLY}")
logger.info(f"Коллекция late:  {SATELLITES_LATE}")

# ------------------------------------------------------------
# 2. Композит 2000 (Historical, L5/7)
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Тест 1: композит за 2000 (Историческая Москва, L5/7)")

composite_2000, info_2000 = create_composite(
    collection=collection_early,
    boundary=historical_boundary,
    year=2000,
    territory_name='Historical',
)

if composite_2000 is None:
    logger.error("❌ Не удалось создать композит за 2000")
else:
    logger.success(f"✅ Композит 2000 создан: {info_2000}")
    bands_2000 = composite_2000.bandNames().getInfo()
    logger.info(f"  Каналы: {bands_2000}")

# ------------------------------------------------------------
# 3. Композит 2020 (Greater, L8/9)
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Тест 2: композит за 2020 (Большая Москва, L8/9)")

composite_2020, info_2020 = create_composite(
    collection=collection_late,
    boundary=greater_moscow_boundary,
    year=2020,
    territory_name='Greater',
)

if composite_2020 is None:
    logger.error("❌ Не удалось создать композит за 2020")
else:
    logger.success(f"✅ Композит 2020 создан: {info_2020}")
    bands_2020 = composite_2020.bandNames().getInfo()
    logger.info(f"  Каналы: {bands_2020}")

# ------------------------------------------------------------
# 4. Статистика индексов
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Статистика индексов (min/max/mean):")


def get_index_stats(composite, indexes, region):
    """Возвращает min/max/mean для указанных индексов."""
    stats = composite.select(list(indexes)).reduceRegion(
        reducer=ee.Reducer.minMax().combine(
            ee.Reducer.mean(), sharedInputs=True
        ),
        geometry=region,
        scale=30,
        maxPixels=1e9,
        bestEffort=True
    ).getInfo()
    return stats


if composite_2000 is not None:
    stats_2000 = get_index_stats(
        composite_2000, INDEX_BANDS, historical_boundary
    )
    logger.info(f"  2000 (Historical):")
    for idx in INDEX_BANDS:
        mn = stats_2000.get(f'{idx}_min', 0)
        mx = stats_2000.get(f'{idx}_max', 0)
        mean = stats_2000.get(f'{idx}_mean', 0)
        logger.info(f"    {idx:<6}  min={mn:>7.3f}  max={mx:>7.3f}  mean={mean:>7.3f}")

    # Проверка диапазона
    ndvi_mean = stats_2000.get('NDVI_mean', 0)
    if -1 <= ndvi_mean <= 1:
        logger.success(f"    ✅ NDVI mean={ndvi_mean:.3f} в диапазоне [-1, 1]")
    else:
        logger.warning(f"    ⚠️  NDVI mean={ndvi_mean:.3f} вне [-1, 1]")

if composite_2020 is not None:
    stats_2020 = get_index_stats(
        composite_2020, INDEX_BANDS, greater_moscow_boundary
    )
    logger.info(f"  2020 (Greater):")
    for idx in INDEX_BANDS:
        mn = stats_2020.get(f'{idx}_min', 0)
        mx = stats_2020.get(f'{idx}_max', 0)
        mean = stats_2020.get(f'{idx}_mean', 0)
        logger.info(f"    {idx:<6}  min={mn:>7.3f}  max={mx:>7.3f}  mean={mean:>7.3f}")

    ndvi_mean = stats_2020.get('NDVI_mean', 0)
    if -1 <= ndvi_mean <= 1:
        logger.success(f"    ✅ NDVI mean={ndvi_mean:.3f} в диапазоне [-1, 1]")
    else:
        logger.warning(f"    ⚠️  NDVI mean={ndvi_mean:.3f} вне [-1, 1]")

# ------------------------------------------------------------
# 5. Карты NDVI для 2000 и 2020
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Строю карты NDVI...")

m = create_base_map('Boundaries')

if composite_2000 is not None:
    add_index_layer(m, composite_2000, 'NDVI')
    logger.info("  ✅ NDVI 2000 добавлен")

if composite_2020 is not None:
    # Для 2020 добавляем NDVI с тем же названием слоя, но разными периодами
    ndvi_2020 = composite_2020.select('NDVI')
    m.addLayer(ndvi_2020, INDEX_VIS['NDVI'], 'NDVI 2020')

add_boundary_layer(m, historical_boundary, label='Историческая Москва')
add_boundary_layer(m, greater_moscow_boundary, label='Большая Москва')

logger.success("Карта NDVI построена.")

# ------------------------------------------------------------
# 6. Итог
# ------------------------------------------------------------
logger.success("=" * 70)
logger.success("✅ ТЕСТ КОМПОЗИТА ПРОЙДЕН")
logger.success("=" * 70)
logger.info("Доступные переменные после этой ячейки:")
logger.info("  composite_2000, info_2000  — композит L5/7 за 2000")
logger.info("  composite_2020, info_2020  — композит L8/9 за 2020")
logger.info("  stats_2000, stats_2020     — статистика индексов")
logger.info("  collection_early, collection_late — объединённые коллекции")

m

In [ ]:
# ============================================================
# ЯЧЕЙКА E5: ФУНКЦИЯ ГЕНЕРАЦИИ ОБУЧАЮЩИХ ТОЧЕК (ESA WorldCover)
# ============================================================
# Генерирует 8000 обучающих точек на основе ESA WorldCover 2021:
#   1) Загрузка ESA WorldCover v200 за 2021
#   2) Clip по границе
#   3) Remap 8 → 4 наших классов (ESA_REMAP_FROM/TO из C1)
#   4) Маскирование nodata (255)
#   5) Проверка площадей ESA в границах
#   6) StratifiedSample: по POINTS_PER_CLASS на класс, scale=10
#   7) Логирование распределения
#
# Возвращает: (points, info) или (None, info)
#   points — ee.FeatureCollection с колонкой 'class' и геометрией
#   info — dict с метаданными
# ============================================================

logger.info("=" * 70)
logger.info("ОПРЕДЕЛЕНИЕ ФУНКЦИИ generate_training_data")
logger.info("=" * 70)


def get_class_areas(image, region, scale=10):
    """
    Возвращает dict: class_id → площадь в км², для всех классов в image.
    Работает через pixelArea + group.
    """
    area_image = ee.Image.pixelArea().divide(1e6).addBands(image)
    areas = area_image.reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='class'),
        geometry=region,
        scale=scale,
        maxPixels=1e13,
        bestEffort=True
    ).getInfo()

    result = {}
    for item in areas.get('groups', []):
        result[int(item['class'])] = item['sum']
    return result


def generate_training_data(boundary, year, boundary_name='',
                            points_per_class=None, verbose=True):
    """
    Генерирует обучающие точки на основе ESA WorldCover 2021.

    Параметры
    ---------
    boundary : ee.FeatureCollection или ee.Geometry
        Границы территории.
    year : int
        Целевой год (для логирования; ESA всегда 2021).
    boundary_name : str
        Имя территории для логов ('Historical' или 'Greater').
    points_per_class : int, опционально
        По умолчанию POINTS_PER_CLASS из C1.
    verbose : bool
        Логировать ли подробно.

    Возвращает
    ----------
    (points, info) : tuple
        points — ee.FeatureCollection с точками, или None.
        info — dict с метаданными.
    """
    points_per_class = points_per_class or POINTS_PER_CLASS
    tag = f"[TrainingData:{boundary_name}]" if boundary_name else "[TrainingData]"

    info = {
        'boundary_name':    boundary_name,
        'year':             year,
        'training_year':    TRAINING_YEAR,
        'points_per_class': points_per_class,
        'total_points':     0,
        'class_distribution': {},
        'esa_areas_mapped':   {},
        'esa_total_area':     0.0,
        'boundary_area':      0.0,
        'area_discrepancy_pct': 0.0,
    }

    # --- 1. Загрузка ESA ---
    if verbose:
        logger.info(f"{tag} Год {year}: эталон — ESA WorldCover {TRAINING_YEAR}.")

    try:
        worldcover = (
            ee.ImageCollection(TRAINING_SOURCE)
            .filterDate(f'{TRAINING_YEAR}-01-01', f'{TRAINING_YEAR}-12-31')
            .first()
            .clip(boundary)
        )
    except Exception as e:
        logger.error(f"{tag} Ошибка загрузки ESA: {e}")
        return None, info

    # --- 2. Remap: 8 классов ESA → 4 наших ---
    try:
        remapped = worldcover.remap(
            ESA_REMAP_FROM,
            ESA_REMAP_TO,
            ESA_REMAP_NODATA
        ).rename('class')

        # Маскируем nodata
        remapped = remapped.updateMask(remapped.neq(ESA_REMAP_NODATA))
    except Exception as e:
        logger.error(f"{tag} Ошибка remap: {e}")
        return None, info

    # --- 3. Проверка площадей ESA в границах ---
    try:
        areas_mapped = get_class_areas(remapped, boundary, scale=10)
        info['esa_areas_mapped'] = areas_mapped
        info['esa_total_area'] = sum(areas_mapped.values())

        # Площадь границы
        boundary_area_m2 = boundary.geometry().area(maxError=1).getInfo()
        boundary_area_km2 = boundary_area_m2 / 1e6
        info['boundary_area'] = boundary_area_km2

        if boundary_area_km2 > 0:
            info['area_discrepancy_pct'] = abs(
                info['esa_total_area'] - boundary_area_km2
            ) / boundary_area_km2 * 100

        if verbose:
            logger.info(f"{tag} Площади ESA по классам (в границах):")
            for cls_id in sorted(areas_mapped.keys()):
                area = areas_mapped[cls_id]
                pct = area / info['esa_total_area'] * 100 if info['esa_total_area'] > 0 else 0
                cls_name = CLASSES[cls_id] if 0 <= cls_id < len(CLASSES) else f'?{cls_id}'
                logger.info(f"{tag}   класс {cls_id} ({cls_name:<25}): "
                            f"{area:>7.1f} км² ({pct:>4.1f}%)")
            logger.info(f"{tag}   ИТОГО ESA:              {info['esa_total_area']:>7.1f} км²")
            logger.info(f"{tag}   Площадь границы:        {boundary_area_km2:>7.1f} км² "
                        f"(расхождение {info['area_discrepancy_pct']:.2f}%)")

            # Проверка clip
            if info['area_discrepancy_pct'] < 5:
                logger.success(f"{tag} ✅ ESA корректно обрезан по границе")
            elif info['area_discrepancy_pct'] < 15:
                logger.warning(f"{tag} ⚠️  Расхождение {info['area_discrepancy_pct']:.1f}% "
                               f"— ESA может быть в другой проекции")
            else:
                logger.error(f"{tag} ❌ Большое расхождение {info['area_discrepancy_pct']:.1f}% "
                             f"— ESA не обрезан или границы не совпадают")

    except Exception as e:
        logger.error(f"{tag} Ошибка проверки площадей: {e}")
        # Не фатально, продолжаем

    # --- 4. Стратифицированная выборка ---
    try:
        training_points = remapped.stratifiedSample(
            numPoints=points_per_class,
            classBand='class',
            region=boundary,
            scale=10,
            geometries=True,
            dropNulls=True,
            seed=RANDOM_SEED
        )
    except Exception as e:
        logger.error(f"{tag} Ошибка stratifiedSample: {e}")
        return None, info

    # --- 5. Проверка результата ---
    try:
        count = training_points.size().getInfo()
        dist_raw = training_points.aggregate_histogram('class').getInfo()

        # Приводим ключи к int (GEE может вернуть '0' или '0.0')
        dist = {}
        for k, v in dist_raw.items():
            try:
                cls_id = int(float(k))
                dist[cls_id] = int(v)
            except (ValueError, TypeError):
                logger.warning(f"{tag} Некорректный ключ класса: {k!r}")

        info['total_points'] = count
        info['class_distribution'] = dist

        if verbose:
            logger.info(f"{tag} Сгенерировано {count} точек (запрошено {points_per_class * N_CLASSES}):")
            for cls_id in sorted(dist.keys()):
                cls_name = CLASSES[cls_id] if 0 <= cls_id < len(CLASSES) else f'?{cls_id}'
                logger.info(f"{tag}   класс {cls_id} ({cls_name:<25}): {dist[cls_id]:>5} точек")

            # Проверка на недобор
            expected = points_per_class * N_CLASSES
            if count < expected * 0.9:
                logger.warning(f"{tag} ⚠️  Получено {count} из {expected} — "
                               f"некоторые классы могут занимать мало места")

        if count == 0:
            logger.error(f"{tag} stratifiedSample вернул 0 точек.")
            return None, info

    except Exception as e:
        logger.error(f"{tag} Ошибка при подсчёте точек: {e}")
        return None, info

    if verbose:
        logger.success(f"{tag} Год {year}: обучающие точки готовы.")

    return training_points, info


logger.success("Функция generate_training_data определена.")
logger.info("=" * 70)

In [ ]:
# ============================================================
# ЯЧЕЙКА E6: ТЕСТ ГЕНЕРАЦИИ ТОЧЕК
# ============================================================
# Что проверяем:
#   1) Точки для Historical создаются
#   2) Точки для Greater создаются
#   3) Распределение по классам корректное
#   4) ESA обрезан по границе (площади совпадают)
#   5) Дубликаты в пикселях Landsat 30м
#   6) Визуализация варианта Г: Historical заливкой, Greater контуром
# ============================================================

logger.info("=" * 70)
logger.info("ТЕСТ ГЕНЕРАЦИИ ТОЧЕК (ESA WorldCover)")
logger.info("=" * 70)

# ------------------------------------------------------------
# 1. Генерация точек для обеих территорий
# ------------------------------------------------------------
logger.info("Генерация точек для Historical...")
points_hist, info_hist = generate_training_data(
    boundary=historical_boundary,
    year=2020,
    boundary_name='Historical',
    verbose=True
)

if points_hist is None:
    logger.error("❌ Не удалось сгенерировать точки для Historical")
    raise RuntimeError("Тест не может быть продолжен.")

logger.info("")
logger.info("Генерация точек для Greater...")
points_greater, info_greater = generate_training_data(
    boundary=greater_moscow_boundary,
    year=2020,
    boundary_name='Greater',
    verbose=True
)

if points_greater is None:
    logger.error("❌ Не удалось сгенерировать точки для Greater")
    raise RuntimeError("Тест не может быть продолжен.")

# ------------------------------------------------------------
# 2. Сводная таблица распределения
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("СВОДНАЯ ТАБЛИЦА РАСПРЕДЕЛЕНИЯ ТОЧЕК")

logger.info(f"  {'Класс':<32}{'Historical':>15}{'Greater':>15}")
logger.info(f"  {'-' * 62}")

for cls_id in range(N_CLASSES):
    n_h = info_hist['class_distribution'].get(cls_id, 0)
    n_g = info_greater['class_distribution'].get(cls_id, 0)
    logger.info(f"  {CLASSES_RU[cls_id]:<32}{n_h:>15,}{n_g:>15,}")

logger.info(f"  {'-' * 62}")
logger.info(f"  {'ИТОГО':<32}{info_hist['total_points']:>15,}"
            f"{info_greater['total_points']:>15,}")

# ------------------------------------------------------------
# 3. Проверка дубликатов в пикселях Landsat 30м
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("ПРОВЕРКА ДУБЛИКАТОВ (Landsat 30м)")
logger.info("Точки сэмплированы при scale=10м. Один пиксель Landsat 30м")
logger.info("содержит ~9 таких точек. Проверяем, сколько дубликатов.")


def check_duplicates(points, composite, region, boundary_name='',
                     sample_n=2000, verbose=True):
    """
    Проверяет, сколько обучающих точек попадает в один пиксель Landsat.

    Сэмплирует ПОДВЫБОРКУ точек (по умолчанию 2000) через sampleRegions,
    считает уникальные комбинации всех 11 признаков + class.

    getInfo() имеет лимит 5000 элементов, поэтому берём подвыборку.
    """
    tag = f"[Dup:{boundary_name}]" if boundary_name else "[Dup]"

    try:
        # Стратифицированная подвыборка: по sample_n/N_CLASSES на класс
        per_class = sample_n // N_CLASSES

        subsets = []
        for cls_id in range(N_CLASSES):
            cls_pts = points.filter(ee.Filter.eq('class', cls_id))
            n_avail = cls_pts.size().getInfo()
            if n_avail == 0:
                continue
            subset = (
                cls_pts.randomColumn('r', seed=RANDOM_SEED)
                .sort('r')
                .limit(min(per_class, n_avail))
            )
            subsets.append(subset)

        subset_points = ee.FeatureCollection(subsets).flatten()
        n_subset = subset_points.size().getInfo()

        if verbose:
            logger.info(f"{tag} Подвыборка для проверки: {n_subset} точек "
                        f"(~{per_class}/класс)")

        # sampleRegions на подвыборке
        samples = composite.select(FEATURES).sampleRegions(
            collection=subset_points,
            properties=['class'],
            scale=30,
            geometries=False,
            tileScale=4
        )

        fc_info = samples.getInfo()
        features = fc_info.get('features', [])

        if not features:
            logger.warning(f"{tag} Не удалось получить образцы")
            return {'total': 0, 'unique': 0, 'dup_pct': 0.0, 'error': True}

        rows = [feat['properties'] for feat in features]
        df = pd.DataFrame(rows)

        total = len(df)
        # Уникальные комбинации: все признаки + class
        cols = FEATURES + ['class']
        # Округляем признаки до 4 знаков — иначе float-шум даёт ложную уникальность
        df_rounded = df[cols].round(4)
        df_unique = df_rounded.drop_duplicates()
        unique = len(df_unique)

        dup_pct = (1 - unique / total) * 100 if total > 0 else 0.0

        if verbose:
            logger.info(f"{tag} Всего образцов:     {total:,}")
            logger.info(f"{tag} Уникальных пар:      {unique:,}")
            logger.info(f"{tag} Дубликатов:          {dup_pct:.1f}%")

            if dup_pct < 20:
                logger.success(f"{tag} ✅ Дубликатов мало")
            elif dup_pct < 50:
                logger.warning(f"{tag} ⚠️  Умеренное количество дубликатов")
            else:
                logger.warning(f"{tag} ⚠️  Много дубликатов ({dup_pct:.1f}%)")

        return {
            'total': total,
            'unique': unique,
            'dup_pct': dup_pct,
            'error': False
        }

    except Exception as e:
        logger.error(f"{tag} Ошибка при проверке: {e}")
        return {'total': 0, 'unique': 0, 'dup_pct': 0.0, 'error': True}


# Проверяем только если composite_2000 и composite_2020 доступны
if composite_2000 is not None and composite_2020 is not None:
    dup_hist = check_duplicates(
        points_hist, composite_2000, historical_boundary, 'Historical'
    )
    dup_greater = check_duplicates(
        points_greater, composite_2020, greater_moscow_boundary, 'Greater'
    )
else:
    logger.warning("⚠️  Композиты недоступны — пропускаю проверку дубликатов")
    dup_hist = dup_greater = None

# ------------------------------------------------------------
# 4. Подвыборка для визуализации (200 точек/класс)
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Подвыборка точек для визуализации (200/класс)")


def subsample_points(points, n_per_class=200):
    """Сэмплирует по n_per_class точек из каждого класса."""
    subsets = []
    for cls_id in range(N_CLASSES):
        cls_pts = points.filter(ee.Filter.eq('class', cls_id))
        n_avail = cls_pts.size().getInfo()
        if n_avail == 0:
            continue
        subset = cls_pts.randomColumn('r', seed=RANDOM_SEED) \
                       .sort('r') \
                       .limit(min(n_per_class, n_avail))
        subsets.append(subset)
    return ee.FeatureCollection(subsets).flatten()


vis_hist = subsample_points(points_hist, 200)
vis_greater = subsample_points(points_greater, 200)

n_vis_hist = vis_hist.size().getInfo()
n_vis_greater = vis_greater.size().getInfo()

logger.info(f"  Historical: {n_vis_hist} точек для визуализации")
logger.info(f"  Greater:    {n_vis_greater} точек для визуализации")

# ------------------------------------------------------------
# 5. Карта: Historical заливкой, Greater контуром
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Строю карту (вариант Г)...")
logger.info("  Historical — заливка (radius 4)")
logger.info("  Greater    — контур (radius 7, без заливки)")

m = create_base_map('Boundaries')

# Границы
add_boundary_layer(m, historical_boundary, label='Историческая Москва')
add_boundary_layer(m, greater_moscow_boundary, label='Большая Москва')

# Точки Historical — по классам, с заливкой
for cls_id in range(N_CLASSES):
    cls_pts = vis_hist.filter(ee.Filter.eq('class', cls_id))
    color_hex = COLORS[cls_id].replace('#', '')
    style = {
        'color':            color_hex,
        'pointRadius':      4,
        'pointFillColor':   color_hex,
        'pointFillOpacity': 0.9,
    }
    m.addLayer(cls_pts, style, f'Historical: {CLASSES_RU[cls_id]}')

# Точки Greater — по классам, только контуром (без заливки)
for cls_id in range(N_CLASSES):
    cls_pts = vis_greater.filter(ee.Filter.eq('class', cls_id))
    color_hex = COLORS[cls_id].replace('#', '')
    style = {
        'color':            color_hex,
        'pointRadius':      7,
        'pointFillColor':   color_hex,
        'pointFillOpacity': 0.0,   # прозрачная заливка = только контур
        'width':            2,
    }
    m.addLayer(cls_pts, style, f'Greater: {CLASSES_RU[cls_id]}')

# Легенда
add_legend(m, 'classification')

logger.success("Карта построена.")

# ------------------------------------------------------------
# 6. Итог
# ------------------------------------------------------------
logger.success("=" * 70)
logger.success("✅ ТЕСТ ГЕНЕРАЦИИ ТОЧЕК ПРОЙДЕН")
logger.success("=" * 70)
logger.info("Что мы проверили:")
logger.info(f"  1) Historical: {info_hist['total_points']} точек, "
            f"ESA площадь {info_hist['esa_total_area']:.0f} км²")
logger.info(f"  2) Greater:    {info_greater['total_points']} точек, "
            f"ESA площадь {info_greater['esa_total_area']:.0f} км²")
logger.info(f"  3) ESA корректно обрезан по границам")
if dup_hist and dup_greater:
    logger.info(f"  4) Дубликаты: Historical {dup_hist['dup_pct']:.1f}%, "
                f"Greater {dup_greater['dup_pct']:.1f}%")
logger.info("")
logger.info("На карте:")
logger.info("  • Historical — заливка в цвет класса (radius 4)")
logger.info("  • Greater    — контур в цвет класса (radius 7, без заливки)")
logger.info("")
logger.info("Доступные переменные после этой ячейки:")
logger.info("  points_hist, info_hist         — точки и метаданные Historical")
logger.info("  points_greater, info_greater   — точки и метаданные Greater")
logger.info("  vis_hist, vis_greater          — подвыборки для визуализации")
logger.info("  dup_hist, dup_greater          — результаты проверки дубликатов")

m

In [ ]:
# ============================================================
# ЯЧЕЙКА E7 (ФИНАЛ): process_year с precomputed_points
# ============================================================
# Содержит две функции:
#   1) build_collection_for_year — выбор коллекции Landsat по году
#   2) parse_feature_importance — безопасный парсер explain()
#   3) process_year             — полный цикл обработки одного года
#
# ФОРМАТ explain() В GEE (проверено эмпирически):
#   {'importance': {'NDVI': 79.15, 'NIR_contrast': 125.86, ...}}
#   — готовый словарь {имя_признака: значение}, без 'names'.
# ============================================================

logger.info("=" * 70)
logger.info("E7: определение process_year (финальная версия)")
logger.info("=" * 70)


# ------------------------------------------------------------
# 1. Выбор коллекции Landsat по году
# ------------------------------------------------------------
def build_collection_for_year(year):
    """Возвращает объединённую коллекцию Landsat для указанного года."""
    sat_ids = SATELLITES_EARLY if year <= 2012 else SATELLITES_LATE
    col = ee.ImageCollection(sat_ids[0])
    for cid in sat_ids[1:]:
        col = col.merge(ee.ImageCollection(cid))
    return col, sat_ids


# ------------------------------------------------------------
# 2. Безопасный парсер feature importance
# ------------------------------------------------------------
def parse_feature_importance(classifier, territory_name='', year=None):
    """
    Извлекает feature importance из classifier.explain().

    Возвращает dict {feature_name: importance} или None.

    ФОРМАТЫ (по убыванию вероятности):
      1. {'importance': {feature: value}}      ← актуальный GEE
      2. {'names': [...], 'importance': [...]}
      3. {'names': [], 'importance': [value...]} с order = FEATURES
    """
    try:
        explain_info = classifier.explain().getInfo()
    except Exception as e:
        logger.warning(f"[{territory_name} {year}] explain().getInfo() упал: {e}")
        return None

    if not isinstance(explain_info, dict):
        return None

    importance = explain_info.get('importance')
    names = explain_info.get('names')

    # --- Вариант A: importance — это готовый dict {feature: value} ---
    if isinstance(importance, dict) and importance:
        # Проверим, что это действительно числа
        try:
            return {k: float(v) for k, v in importance.items()}
        except (ValueError, TypeError):
            logger.warning(f"[{territory_name} {year}] importance dict "
                           f"содержит нечисловые значения")
            return None

    # --- Вариант B: names и importance — списки одинаковой длины ---
    if (isinstance(names, list) and isinstance(importance, list)
        and len(names) == len(importance) > 0):
        try:
            return {str(n): float(v) for n, v in zip(names, importance)}
        except (ValueError, TypeError):
            pass

    # --- Вариант C: names пустой, importance — список чисел длиной FEATURES ---
    if (not names and isinstance(importance, list)
        and len(importance) == len(FEATURES)):
        try:
            return {f: float(v) for f, v in zip(FEATURES, importance)}
        except (ValueError, TypeError):
            pass

    logger.warning(f"[{territory_name} {year}] feature importance: "
                   f"неизвестная структура. names={names!r}, "
                   f"importance type={type(importance).__name__}")
    return None


# ------------------------------------------------------------
# 3. Полный цикл обработки одного года
# ------------------------------------------------------------
def process_year(boundary, year, territory_name,
                 precomputed_points=None,
                 precomputed_training_info=None,
                 points_per_class=None, verbose=True):
    """Полный цикл обработки одного года."""
    points_per_class = points_per_class or POINTS_PER_CLASS

    logger.info("=" * 70)
    logger.info(f"[{territory_name}] ГОД {year}: СТАРТ ОБРАБОТКИ")
    logger.info("=" * 70)
    t_start = time.time()

    # --- ЭТАП 0: коллекция ---
    collection, sat_ids = build_collection_for_year(year)
    logger.info(f"[{territory_name}] Год {year}: ЭТАП 0/6 — коллекция выбрана.")
    logger.info(f"[{territory_name}] Год {year}: спутники: {sat_ids}")

    # --- ЭТАП 1: композит ---
    logger.info(f"[{territory_name}] Год {year}: ЭТАП 1/6 — создание композита.")
    t1 = time.time()
    composite, comp_info = create_composite(
        collection=collection, boundary=boundary, year=year,
        territory_name=territory_name, verbose=verbose
    )
    if composite is None:
        logger.error(f"[{territory_name}] Год {year}: композит не создан.")
        return None
    logger.info(f"[{territory_name}] Год {year}: ЭТАП 1 завершён за {time.time()-t1:.1f} с.")

    # --- ЭТАП 2: обучающие точки ---
    t2 = time.time()
    if precomputed_points is not None:
        logger.info(f"[{territory_name}] Год {year}: ЭТАП 2/6 — "
                    f"использую готовые точки.")
        training_points = precomputed_points
        train_info = precomputed_training_info or {
            'boundary_name': territory_name, 'year': year,
            'source': 'precomputed'
        }
        n_points = training_points.size().getInfo()
        logger.info(f"[{territory_name}] Год {year}: точек в наборе = {n_points}.")
    else:
        logger.info(f"[{territory_name}] Год {year}: ЭТАП 2/6 — генерация точек.")
        training_points, train_info = generate_training_data(
            boundary=boundary, year=year, boundary_name=territory_name,
            points_per_class=points_per_class, verbose=verbose
        )
        if training_points is None:
            logger.error(f"[{territory_name}] Год {year}: точки не сгенерированы.")
            return None
    logger.info(f"[{territory_name}] Год {year}: ЭТАП 2 завершён за {time.time()-t2:.1f} с.")

    # --- ЭТАП 3: извлечение спектров ---
    logger.info(f"[{territory_name}] Год {year}: ЭТАП 3/6 — извлечение спектров.")
    t3 = time.time()

    training_samples = composite.select(FEATURES).sampleRegions(
        collection=training_points, properties=['class'],
        scale=GEE_SCALE, projection=ee.Projection(ANALYSIS_CRS),
        geometries=False, tileScale=GEE_TILE_SCALE
    )

    total_samples = training_samples.size().getInfo()
    logger.info(f"[{territory_name}] Год {year}: получено {total_samples} "
                f"спектральных образцов.")

    with_random = training_samples.randomColumn('random', seed=RANDOM_SEED)
    train = with_random.filter(ee.Filter.lt('random', TRAIN_FRACTION))
    test  = with_random.filter(ee.Filter.gte('random', TRAIN_FRACTION))

    train_size = train.size().getInfo()
    test_size = test.size().getInfo()
    logger.info(f"[{territory_name}] Год {year}: train = {train_size}, test = {test_size}.")
    logger.info(f"[{territory_name}] Год {year}: ЭТАП 3 завершён за {time.time()-t3:.1f} с.")

    # --- ЭТАП 4: обучение RF ---
    logger.info(f"[{territory_name}] Год {year}: ЭТАП 4/6 — обучение Random Forest.")
    t4 = time.time()
    classifier = ee.Classifier.smileRandomForest(
        numberOfTrees=RF_TREES,
        minLeafPopulation=RF_MIN_LEAF,
        bagFraction=RF_BAG_FRACTION
    ).train(features=train, classProperty='class', inputProperties=FEATURES)

    feature_importance = parse_feature_importance(classifier, territory_name, year)
    if feature_importance and verbose:
        top5 = sorted(feature_importance.items(),
                      key=lambda x: x[1], reverse=True)[:5]
        logger.info(f"[{territory_name}] Год {year}: топ-5 важных признаков:")
        for feat, imp in top5:
            logger.info(f"    {feat:<16} {imp:.4f}")

    logger.info(f"[{territory_name}] Год {year}: ЭТАП 4 завершён за {time.time()-t4:.1f} с.")

    # --- ЭТАП 5: валидация + классификация ---
    logger.info(f"[{territory_name}] Год {year}: ЭТАП 5/6 — валидация и классификация.")
    t5 = time.time()

    test_classified = test.classify(classifier)
    accuracy = test_classified.errorMatrix('class', 'classification')
    oa = accuracy.accuracy().getInfo()
    kappa = accuracy.kappa().getInfo()
    matrix = accuracy.array().getInfo()

    logger.info(f"[{territory_name}] Год {year}: ВАЛИДАЦИЯ: OA = {oa:.4f}, "
                f"Kappa = {kappa:.4f}.")

    if oa < ACCURACY_THRESHOLD:
        logger.warning(f"[{territory_name}] Год {year}: OA ниже порога {ACCURACY_THRESHOLD}.")
    if kappa < KAPPA_THRESHOLD:
        logger.warning(f"[{territory_name}] Год {year}: Kappa ниже порога {KAPPA_THRESHOLD}.")

    logger.info(f"[{territory_name}] Год {year}: матрица ошибок:")
    for i, row in enumerate(matrix):
        row_str = "  ".join([f"{int(v):5d}" for v in row])
        logger.info(f"    {CLASSES[i]:25s}: {row_str}")

    classified = composite.select(FEATURES).classify(classifier)

    if MAJORITY_FILTER:
        classified = classified.focalMode(FILTER_WINDOW, 'square', 'pixels')
        logger.info(f"[{territory_name}] Год {year}: фильтр большинства "
                    f"{FILTER_WINDOW}x{FILTER_WINDOW}.")

    logger.info(f"[{territory_name}] Год {year}: ЭТАП 5 завершён за {time.time()-t5:.1f} с.")

    # --- ЭТАП 6: площади ---
    logger.info(f"[{territory_name}] Год {year}: ЭТАП 6/6 — расчёт площадей.")
    t6 = time.time()

    area_image = ee.Image.pixelArea().divide(1e6).addBands(classified)
    areas_dict = area_image.reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='class'),
        geometry=boundary.geometry(), scale=GEE_SCALE,
        crs=ANALYSIS_CRS, maxPixels=GEE_MAX_PIXELS,
    ).getInfo()

    class_areas = {}
    total_area = 0.0
    for item in areas_dict.get('groups', []):
        class_id = int(item['class'])
        class_areas[class_id] = item['sum']
        total_area += item['sum']

    logger.info(f"[{territory_name}] Год {year}: ПЛОЩАДИ:")
    for cls_id in sorted(class_areas.keys()):
        pct = class_areas[cls_id] / total_area * 100 if total_area > 0 else 0
        logger.info(f"    {CLASSES[cls_id]:25s}: {class_areas[cls_id]:8.1f} км² "
                    f"({pct:5.1f}%)")
    logger.info(f"    {'ИТОГО':25s}: {total_area:8.1f} км²")

    logger.info(f"[{territory_name}] Год {year}: ЭТАП 6 завершён за {time.time()-t6:.1f} с.")

    elapsed_sec = time.time() - t_start
    logger.success(f"[{territory_name}] Год {year}: ОБРАБОТКА ЗАВЕРШЕНА "
                   f"за {elapsed_sec:.1f} с.")
    logger.info("=" * 70)

    return {
        'classified':         classified,
        'areas':              class_areas,
        'total_area':         total_area,
        'oa':                 oa,
        'kappa':              kappa,
        'confusion_matrix':   matrix,
        'train_size':         train_size,
        'test_size':          test_size,
        'territory':          territory_name,
        'year':               year,
        'sat_ids':            sat_ids,
        'n_scenes':           comp_info['n_scenes'],
        'used_year':          comp_info['used_year'],
        'fallback_used':      comp_info['fallback_used'],
        'elapsed_sec':        elapsed_sec,
        'composite_info':     comp_info,
        'training_info':      train_info,
        'feature_importance': feature_importance,
    }


logger.success("Функция process_year обновлена (финальная версия).")
logger.info("=" * 70)

# БЛОК F. Диагностика

In [ ]:
# ============================================================
# ЯЧЕЙКА F1: ДОСТУПНОСТЬ СНИМКОВ 2000-2026
# ============================================================
# Проверяем для КАЖДОГО года 2000-2026:
#   • сколько снимков за лето (июнь-август) в границах территории
#   • медиана облачности
#   • нужен ли fallback (n < MIN_IMAGES_FOR_COMPOSITE)
#   • разбивка по сенсорам (L5, L7, L8, L9)
#
# Один запрос на территорию: забираем свойства всех снимков сразу,
# агрегируем локально в pandas. Быстро (~10-20 сек на территорию).
# ============================================================

logger.info("=" * 70)
logger.info("F1: ДОСТУПНОСТЬ СНИМКОВ 2000-2026")
logger.info("=" * 70)

ALL_YEARS = list(range(2000, 2027))


def get_all_scenes_metadata(boundary, year_range, season_months=(6, 7, 8)):
    """
    Возвращает список метаданных всех летних снимков в границах.
    Один запрос к GEE.
    """
    # Объединяем все коллекции Landsat
    col = ee.ImageCollection(SATELLITES_EARLY[0])
    for cid in SATELLITES_EARLY[1:] + SATELLITES_LATE:
        col = col.merge(ee.ImageCollection(cid))

    # Фильтр по дате (грубо) — за весь период
    start = f'{year_range[0]}-06-01'
    end   = f'{year_range[-1]}-09-01'
    col = col.filterDate(start, end).filterBounds(boundary)

    # Создаём FeatureCollection с нужными свойствами
    def make_feature(img):
        date = ee.Date(img.get('DATE_ACQUIRED'))
        return ee.Feature(None, {
            'year':    date.get('year'),
            'month':   date.get('month'),
            'cloud':   img.get('CLOUD_COVER'),
            'sensor':  img.get('SPACECRAFT_ID'),
        })

    features = col.map(make_feature)

    # Фильтр по месяцам (июнь-август)
    months_filter = ee.Filter.inList('month', list(season_months))
    features = features.filter(months_filter)

    # Один запрос
    info = features.getInfo()
    return [f['properties'] for f in info.get('features', [])]


def build_scene_summary(metadata_list):
    """Агрегирует метаданные по годам."""
    if not metadata_list:
        return pd.DataFrame(columns=['year', 'n_scenes', 'median_cloud',
                                      'min_cloud', 'max_cloud', 'sensors'])

    df = pd.DataFrame(metadata_list)
    df['year']  = df['year'].astype(int)
    df['cloud'] = df['cloud'].astype(float)

    def agg_sensors(series):
        return ','.join(sorted(set(series.dropna().astype(str))))

    summary = df.groupby('year').agg(
        n_scenes=('cloud', 'count'),
        median_cloud=('cloud', 'median'),
        min_cloud=('cloud', 'min'),
        max_cloud=('cloud', 'max'),
        sensors=('sensor', agg_sensors),
    ).reset_index()

    return summary


# ------------------------------------------------------------
# Сбор данных по обеим территориям
# ------------------------------------------------------------
f1_data = {}

for territory_name, boundary in [('Historical', historical_boundary),
                                  ('Greater',    greater_moscow_boundary)]:
    logger.info(f"[F1] Собираю метаданные снимков: {territory_name}...")
    t0 = time.time()

    metadata = get_all_scenes_metadata(boundary, ALL_YEARS)
    summary = build_scene_summary(metadata)

    # Убеждаемся, что все годы присутствуют
    all_years_df = pd.DataFrame({'year': ALL_YEARS})
    summary = all_years_df.merge(summary, on='year', how='left')

    # Заполняем только те колонки, где нужны конкретные значения.
    # median_cloud, min_cloud, max_cloud остаются как NaN — так и надо.
    summary['n_scenes'] = summary['n_scenes'].fillna(0).astype(int)
    summary['sensors']  = summary['sensors'].fillna('')
    summary['needs_fallback'] = summary['n_scenes'] < MIN_IMAGES_FOR_COMPOSITE

    f1_data[territory_name] = summary
    logger.info(f"[F1] {territory_name}: собрано за {time.time()-t0:.1f} с, "
                f"всего снимков за период: {summary['n_scenes'].sum()}")


# ------------------------------------------------------------
# Печать сводной таблицы
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("СВОДКА ПО ГОДАМ (все 2000-2026)")
logger.info("-" * 70)

for territory_name, df in f1_data.items():
    logger.info(f"\n[{territory_name}]")
    logger.info(f"  {'Год':<6}{'Снимков':>10}{'Медиана':>10}{'Мин':>8}{'Макс':>8}  {'Fallback':<10}{'Сенсоры'}")
    logger.info(f"  {'-' * 80}")

    for _, row in df.iterrows():
        year = int(row['year'])
        n = int(row['n_scenes'])
        med = row['median_cloud']
        mn = row['min_cloud']
        mx = row['max_cloud']
        sensors = row['sensors']
        fallback = '⚠️ ДА' if row['needs_fallback'] else 'нет'

        # Форматирование
        med_str = f"{med:.1f}" if pd.notna(med) else "—"
        mn_str = f"{mn:.1f}" if pd.notna(mn) else "—"
        mx_str = f"{mx:.1f}" if pd.notna(mx) else "—"

        # Подсветка проблемных лет
        marker = ''
        if n == 0:
            marker = ' ❌'
        elif row['needs_fallback']:
            marker = ' ⚠️'

        logger.info(f"  {year:<6}{n:>10}{med_str:>10}{mn_str:>8}{mx_str:>8}  "
                    f"{fallback:<10}{sensors}{marker}")


# ------------------------------------------------------------
# Анализ: где нужен fallback
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("АНАЛИЗ FALLBACK")

for territory_name, df in f1_data.items():
    problem_years = df[df['needs_fallback']]['year'].tolist()
    zero_years = df[df['n_scenes'] == 0]['year'].tolist()

    if not problem_years:
        logger.success(f"[{territory_name}] ✅ Все годы имеют ≥{MIN_IMAGES_FOR_COMPOSITE} снимков")
    else:
        logger.warning(f"[{territory_name}] ⚠️  Годы с недостатком снимков (<{MIN_IMAGES_FOR_COMPOSITE}): "
                       f"{problem_years}")
        if zero_years:
            logger.error(f"[{territory_name}] ❌ Годы БЕЗ снимков совсем: {zero_years}")

# ------------------------------------------------------------
# Проверка целевых годов
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("ПРОВЕРКА ЦЕЛЕВЫХ ГОДОВ")

for territory_name, df in f1_data.items():
    target_years = HISTORICAL_YEARS if territory_name == 'Historical' else GREATER_YEARS
    logger.info(f"\n[{territory_name}] целевые годы: {target_years}")

    for year in target_years:
        row = df[df['year'] == year]
        if row.empty:
            logger.error(f"  {year}: ❌ нет данных")
            continue
        row = row.iloc[0]
        n = int(row['n_scenes'])
        med = row['median_cloud']
        med_str = f"{med:.1f}%" if pd.notna(med) else "—"

        if n >= MIN_IMAGES_FOR_COMPOSITE:
            logger.success(f"  {year}: ✅ {n} снимков, медиана облачности {med_str}")
        elif n > 0:
            logger.warning(f"  {year}: ⚠️  только {n} снимков (нужно {MIN_IMAGES_FOR_COMPOSITE}), "
                           f"будет fallback")
        else:
            logger.error(f"  {year}: ❌ нет снимков")


# ------------------------------------------------------------
# Графики
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Строю графики...")

fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)

# График 1: количество снимков
ax1 = axes[0]
for territory_name, df in f1_data.items():
    color = 'steelblue' if territory_name == 'Historical' else 'darkorange'
    ax1.bar(df['year'] + (0.2 if territory_name == 'Greater' else -0.2),
            df['n_scenes'], width=0.4, alpha=0.75,
            label=territory_name, color=color)

ax1.axhline(y=MIN_IMAGES_FOR_COMPOSITE, color='red', linestyle='--',
            alpha=0.7, label=f'Минимум ({MIN_IMAGES_FOR_COMPOSITE})')
ax1.set_ylabel('Количество снимков', fontsize=12)
ax1.set_title('Доступность снимков Landsat за лето (2000–2026)', fontsize=13)
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# График 2: медиана облачности
ax2 = axes[1]
for territory_name, df in f1_data.items():
    color = 'steelblue' if territory_name == 'Historical' else 'darkorange'
    valid_df = df.dropna(subset=['median_cloud'])
    ax2.plot(valid_df['year'], valid_df['median_cloud'],
             marker='o', linewidth=2, markersize=5,
             label=territory_name, color=color)

ax2.axhline(y=20, color='red', linestyle='--', alpha=0.7, label='20% (ориентир)')
ax2.set_xlabel('Год', fontsize=12)
ax2.set_ylabel('Медиана облачности (%)', fontsize=12)
ax2.set_title('Медиана облачности летних снимков', fontsize=13)
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(2000, 2027, 2))
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Сохранение в JSON
# ------------------------------------------------------------
f1_json = {
    'run_timestamp': RUN_TIMESTAMP,
    'years': ALL_YEARS,
    'min_images_for_composite': MIN_IMAGES_FOR_COMPOSITE,
    'data': {
        territory: df.to_dict(orient='records')
        for territory, df in f1_data.items()
    },
}

F1_FILE = PATHS['INTERMEDIATE'] / 'diagnostics_f1_scenes.json'
with open(F1_FILE, 'w', encoding='utf-8') as f:
    json.dump(f1_json, f, indent=2, ensure_ascii=False, default=str)

logger.success(f"Результаты F1 сохранены: {F1_FILE.name}")


# ------------------------------------------------------------
# Итог
# ------------------------------------------------------------
logger.success("=" * 70)
logger.success("✅ F1 ЗАВЕРШЁН: доступность снимков проверена")
logger.success("=" * 70)
logger.info("Доступные переменные после этой ячейки:")
logger.info("  f1_data       — dict {territory: DataFrame}")
logger.info("  F1_FILE       — путь к JSON с результатами")

In [ ]:
# ============================================================
# ЯЧЕЙКА F2: ДИАГНОСТИКА КОМПОЗИТОВ И ОБУЧАЮЩИХ ТОЧЕК
# ============================================================
# ЕДИНЫЙ ПРОХОД по всем 27 годам × 2 территориям:
#   • create_composite — один раз на итерацию
#   • Базовые статистики композита — всегда
#   • Расширенные метрики (percentiles, валидность точек) — только
#     для ЦЕЛЕВЫХ лет (5 Historical + 3 Greater)
#
# ИСТОРИЯ ИСПРАВЛЕНИЙ:
#   1. Проверка ключа mean-значений: `if f'{band}_mean' in stats_ms`.
#   2. boundary.geometry() — явное приведение к геометрии.
#   3. Reducer.combine с sharedInputs=True.
#   4. Count и cloud_residue — отдельными запросами.
#   5. crs=ANALYSIS_CRS во всех reduceRegion. Без этого в EPSG:4326
#      scale=30 даёт пиксель ~17×30 м, count завышается в ~1.78 раз.
#   6. Убран bestEffort=True (он мог незаметно менять масштаб).
#   7. sampleRegions принимает projection=ee.Projection(...), а НЕ crs=.
#   8. total_pixels считается через reduceRegion на константном
#      изображении (в той же сетке, что и valid_pixels), чтобы
#      coverage был строго ≤ 100%.
# ============================================================

import numpy as np

try:
    from scipy import stats as scipy_stats
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    logger.warning("scipy не установлен — p-value не будет рассчитан")

logger.info("=" * 70)
logger.info("F2: ДИАГНОСТИКА КОМПОЗИТОВ И ОБУЧАЮЩИХ ТОЧЕК")
logger.info("=" * 70)

# ------------------------------------------------------------
# 0. Настройки tiered-диагностики
# ------------------------------------------------------------
TARGET_YEARS_BY_TERRITORY = {
    'Historical': set(HISTORICAL_YEARS),
    'Greater':    set(GREATER_YEARS),
}

SAMPLE_POINTS_PER_CLASS = 300
VALIDATION_SEED = 42

logger.info(f"Целевые годы Historical: {sorted(TARGET_YEARS_BY_TERRITORY['Historical'])}")
logger.info(f"Целевые годы Greater:    {sorted(TARGET_YEARS_BY_TERRITORY['Greater'])}")
logger.info(f"Точек на класс для проверки: {SAMPLE_POINTS_PER_CLASS}")

# ------------------------------------------------------------
# 1. Площади и границы
# ------------------------------------------------------------
BOUNDARY_AREAS_KM2 = {
    'Historical': BOUNDARY_AREAS['Historical'],
    'Greater':    BOUNDARY_AREAS['Greater'],
}
BOUNDARIES_MAP = {
    'Historical': historical_boundary,
    'Greater':    greater_moscow_boundary,
}

# ------------------------------------------------------------
# 2. Подготовка обучающих точек для валидации (один раз)
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("Подготовка подвыборок обучающих точек для валидации...")


def subsample_points_tiered(points, n_per_class):
    """Берёт n_per_class точек из каждого класса, рандомизированно."""
    subsets = []
    for cls_id in range(N_CLASSES):
        cls_pts = points.filter(ee.Filter.eq('class', cls_id))
        subset = (
            cls_pts.randomColumn('r', seed=VALIDATION_SEED)
            .sort('r')
            .limit(n_per_class)
        )
        subsets.append(subset)
    return ee.FeatureCollection(subsets).flatten()


points_hist_small = subsample_points_tiered(points_hist, SAMPLE_POINTS_PER_CLASS)
points_greater_small = subsample_points_tiered(points_greater, SAMPLE_POINTS_PER_CLASS)

n_hist_small = points_hist_small.size().getInfo()
n_greater_small = points_greater_small.size().getInfo()

logger.info(f"  Historical: {n_hist_small} точек (по {SAMPLE_POINTS_PER_CLASS}/класс)")
logger.info(f"  Greater:    {n_greater_small} точек (по {SAMPLE_POINTS_PER_CLASS}/класс)")

VALIDATION_POINTS = {
    'Historical': points_hist_small,
    'Greater':    points_greater_small,
}


# ------------------------------------------------------------
# 3. Функции
# ------------------------------------------------------------

def compute_composite_stats_light(composite, boundary, area_km2):
    """
    Базовые метрики композита:
      • coverage (valid_pixels / total_pixels)
      • NDVI_mean, NDVI_std
      • NDWI_mean, NDBI_mean, BSI_mean, SAVI_mean
      • red_mean, nir_mean, swir1_mean
      • cloud_residue_pct

    КЛЮЧЕВЫЕ МОМЕНТЫ:
      • geom = boundary.geometry() — FC может обрабатываться как набор features.
      • crs=ANALYSIS_CRS (EPSG:32637) — метрическая проекция UTM 37N.
        Без неё scale=30 в градусах даёт пиксель 17×30 м и count ×1.78.
      • total_pixels — через тот же reduceRegion с константным изображением.
        Это гарантирует, что сетка совпадает с сеткой valid_pixels,
        и coverage ≤ 100%.
    """
    try:
        geom = boundary.geometry()

        # cloud_flag: облака = высокий blue + низкий nir
        cloud_flag = (
            composite.select('blue').gt(0.25)
            .And(composite.select('nir').lt(0.15))
            .rename('cloud_flag')
        )
        img = composite.addBands(cloud_flag)

        # --- 1. Mean + stdDev ---
        stats_ms = img.reduceRegion(
            reducer=ee.Reducer.mean().combine(
                ee.Reducer.stdDev(), sharedInputs=True
            ),
            geometry=geom,
            scale=GEE_SCALE,
            crs=ANALYSIS_CRS,
            maxPixels=GEE_MAX_PIXELS,
            tileScale=8,
        ).getInfo() or {}

        # --- 2. Count валидных пикселей (только NDVI) ---
        stats_count = img.select(['NDVI']).reduceRegion(
            reducer=ee.Reducer.count(),
            geometry=geom,
            scale=GEE_SCALE,
            crs=ANALYSIS_CRS,
            maxPixels=GEE_MAX_PIXELS,
            tileScale=8,
        ).getInfo() or {}

        # --- 3. Cloud residue ---
        cloud_count = img.select('cloud_flag').reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=geom,
            scale=GEE_SCALE,
            crs=ANALYSIS_CRS,
            maxPixels=GEE_MAX_PIXELS,
            tileScale=8,
        ).getInfo().get('cloud_flag', 0) or 0

        # --- 4. Total pixels ТЕМ ЖЕ методом ---
        # Считаем константное изображение 1 через reduceRegion.count
        # в той же сетке (scale=30, crs=EPSG:32637). Это даёт ровно то
        # количество пикселей, которое GEE «видит» внутри геометрии,
        # и гарантирует coverage ≤ 100%.
        total_pixels = ee.Image.constant(1).rename('total').reduceRegion(
            reducer=ee.Reducer.count(),
            geometry=geom,
            scale=GEE_SCALE,
            crs=ANALYSIS_CRS,
            maxPixels=GEE_MAX_PIXELS,
            tileScale=8,
        ).getInfo().get('total', 0) or 0

        # --- Собираем результат ---
        result = {}

        # ИСПРАВЛЕНИЕ: проверяем ключ 'NDVI_mean', а не 'NDVI'
        for band in FEATURES + ['red', 'green', 'blue', 'nir', 'swir1', 'swir2']:
            key_mean = f'{band}_mean'
            if key_mean in stats_ms:
                result[key_mean] = stats_ms[key_mean]

        result['NDVI_std'] = stats_ms.get('NDVI_stdDev')

        valid_pixels = stats_count.get('NDVI', 0) or 0
        result['valid_pixels'] = valid_pixels
        result['total_pixels'] = total_pixels
        result['coverage_pct'] = (
            valid_pixels / total_pixels * 100 if total_pixels > 0 else 0
        )

        result['cloud_residue_pct'] = (
            cloud_count / valid_pixels * 100 if valid_pixels > 0 else 0
        )

        return result

    except Exception as e:
        logger.error(f"    compute_composite_stats_light: {e}")
        return None


def compute_composite_percentiles(composite, boundary):
    """NDVI percentiles (только для целевых лет)."""
    try:
        geom = boundary.geometry()

        stats = composite.select('NDVI').reduceRegion(
            reducer=ee.Reducer.percentile([10, 50, 90]),
            geometry=geom,
            scale=GEE_SCALE,
            crs=ANALYSIS_CRS,
            maxPixels=GEE_MAX_PIXELS,
            tileScale=8,
        ).getInfo()

        if stats is None:
            return None

        p10 = stats.get('NDVI_p10')
        p50 = stats.get('NDVI_p50')
        p90 = stats.get('NDVI_p90')

        result = {
            'NDVI_p10': p10,
            'NDVI_p50': p50,
            'NDVI_p90': p90,
        }

        if p10 is not None and p50 is not None and p90 is not None:
            result['NDVI_bimodality'] = p50 - (p10 + p90) / 2
        else:
            result['NDVI_bimodality'] = None

        return result

    except Exception as e:
        logger.error(f"    compute_composite_percentiles: {e}")
        return None


def compute_training_validation(composite, validation_points, boundary_name=''):
    """
    Проверяет, сколько обучающих точек попадает на валидные пиксели
    композита, и считает per-class метрики.

    ИСПРАВЛЕНИЕ: sampleRegions принимает параметр `projection`,
    а НЕ `crs`. Передаём ee.Projection(ANALYSIS_CRS).
    """
    tag = f"[Val:{boundary_name}]" if boundary_name else "[Val]"

    try:
        samples = composite.select(FEATURES).sampleRegions(
            collection=validation_points,
            properties=['class'],
            scale=GEE_SCALE,
            projection=ee.Projection(ANALYSIS_CRS),   # ← ИСПРАВЛЕНО
            geometries=False,
            tileScale=4
        )

        fc_info = samples.getInfo()
        features = fc_info.get('features', [])

        if not features:
            logger.warning(f"{tag} 0 валидных точек")
            return None

        df = pd.DataFrame([f['properties'] for f in features])
        df['class'] = df['class'].astype(int)

        requested_per_class = SAMPLE_POINTS_PER_CLASS
        requested_total = requested_per_class * N_CLASSES

        result = {
            'valid_points_total': len(df),
            'requested_total': requested_total,
            'valid_pct': len(df) / requested_total * 100 if requested_total > 0 else 0,
            'valid_points_per_class': {},
            'NDVI_mean_per_class': {},
            'NDVI_std_per_class': {},
        }

        for cls_id in range(N_CLASSES):
            cls_df = df[df['class'] == cls_id]
            n = len(cls_df)
            result['valid_points_per_class'][cls_id] = n
            if n > 0:
                result['NDVI_mean_per_class'][cls_id] = float(cls_df['NDVI'].mean())
                result['NDVI_std_per_class'][cls_id] = float(cls_df['NDVI'].std())
            else:
                result['NDVI_mean_per_class'][cls_id] = None
                result['NDVI_std_per_class'][cls_id] = None

        ndvi_dense = result['NDVI_mean_per_class'].get(0)
        ndvi_sparse = result['NDVI_mean_per_class'].get(1)
        if ndvi_dense is not None and ndvi_sparse is not None:
            result['separability'] = ndvi_dense - ndvi_sparse
        else:
            result['separability'] = None

        return result

    except Exception as e:
        logger.error(f"{tag} Ошибка: {e}")
        return None


# ------------------------------------------------------------
# 4. Основной цикл
# ------------------------------------------------------------
logger.info("-" * 70)
logger.info("СБОР ДАННЫХ: 27 лет × 2 территории")
logger.info("-" * 70)

# Объединяем коллекции один раз
COLLECTION_ALL = ee.ImageCollection(SATELLITES_EARLY[0])
for cid in SATELLITES_EARLY[1:] + SATELLITES_LATE:
    COLLECTION_ALL = COLLECTION_ALL.merge(ee.ImageCollection(cid))

f2_data = {'Historical': [], 'Greater': []}
f2_failed = []

total_iterations = len(ALL_YEARS) * 2
iteration = 0
t_f2_start = time.time()

for territory_name in ['Historical', 'Greater']:
    boundary = BOUNDARIES_MAP[territory_name]
    area_km2 = BOUNDARY_AREAS_KM2[territory_name]
    target_years = TARGET_YEARS_BY_TERRITORY[territory_name]
    validation_points = VALIDATION_POINTS[territory_name]

    logger.info("")
    logger.info(f"[F2:{territory_name}] Целевые годы: {sorted(target_years)}")

    for year in ALL_YEARS:
        iteration += 1
        is_target = year in target_years
        t0 = time.time()
        tag = 'FULL' if is_target else 'light'

        composite, comp_info = create_composite(
            collection=COLLECTION_ALL,
            boundary=boundary,
            year=year,
            territory_name=territory_name,
            verbose=False
        )

        if composite is None:
            logger.error(f"  [{iteration:>2}/{total_iterations}] "
                         f"{territory_name} {year}: композит не создан")
            f2_failed.append({'territory': territory_name, 'year': year,
                              'reason': 'composite_failed'})
            continue

        stats = compute_composite_stats_light(composite, boundary, area_km2)
        if stats is None:
            logger.error(f"  [{iteration:>2}/{total_iterations}] "
                         f"{territory_name} {year}: reduceRegion failed")
            f2_failed.append({'territory': territory_name, 'year': year,
                              'reason': 'reduce_failed'})
            continue

        if is_target:
            perc = compute_composite_percentiles(composite, boundary)
            if perc:
                stats.update(perc)

            val = compute_training_validation(
                composite, validation_points, territory_name
            )
            if val:
                stats['training_validation'] = val

        stats['year'] = year
        stats['territory'] = territory_name
        stats['n_scenes'] = comp_info['n_scenes']
        stats['used_year'] = comp_info['used_year']
        stats['fallback_used'] = comp_info['fallback_used']
        stats['is_target'] = is_target

        f2_data[territory_name].append(stats)

        elapsed = time.time() - t0
        ndvi_m = stats.get('NDVI_mean', 0) or 0
        cov = stats.get('coverage_pct', 0)

        extra = ''
        if is_target and 'training_validation' in stats:
            v = stats['training_validation']
            extra = f", val={v['valid_pct']:.0f}%"

        logger.info(
            f"  [{iteration:>2}/{total_iterations}] {territory_name} {year} "
            f"[{tag:>5}]: n={comp_info['n_scenes']:>2}, "
            f"cov={cov:>5.1f}%, NDVI={ndvi_m:>5.3f}{extra}, {elapsed:.1f}с"
        )

elapsed_min = (time.time() - t_f2_start) / 60
n_success = sum(len(v) for v in f2_data.values())
logger.success(f"Сбор завершён за {elapsed_min:.1f} мин. "
               f"Успешно: {n_success}/{total_iterations}")

if f2_failed:
    logger.warning(f"Неудачных итераций: {len(f2_failed)}")
    for item in f2_failed:
        logger.warning(f"  {item['territory']} {item['year']}: {item['reason']}")


# ------------------------------------------------------------
# 5. Сохранение JSON
# ------------------------------------------------------------
f2_json = {
    'run_timestamp': RUN_TIMESTAMP,
    'years': ALL_YEARS,
    'target_years': {k: sorted(v) for k, v in TARGET_YEARS_BY_TERRITORY.items()},
    'sample_points_per_class': SAMPLE_POINTS_PER_CLASS,
    'failed': f2_failed,
    'data': f2_data,
}

F2_FILE = PATHS['INTERMEDIATE'] / 'diagnostics_f2.json'
with open(F2_FILE, 'w', encoding='utf-8') as f:
    json.dump(f2_json, f, indent=2, ensure_ascii=False, default=str)

logger.success(f"Результаты F2 сохранены: {F2_FILE.name}")


# ------------------------------------------------------------
# 6. Общий DataFrame
# ------------------------------------------------------------
df_f2 = pd.DataFrame(f2_data['Historical'] + f2_data['Greater'])
df_f2 = df_f2.sort_values(['territory', 'year']).reset_index(drop=True)


# ============================================================
# ТАБЛИЦА 1: Спектральная сводка
# ============================================================
logger.info("")
logger.info("=" * 70)
logger.info("ТАБЛИЦА 1: СПЕКТРАЛЬНАЯ СВОДКА (все годы)")
logger.info("=" * 70)

for territory_name in ['Historical', 'Greater']:
    logger.info(f"\n[{territory_name}]")
    logger.info(f"  {'Год':<6}{'n':>4}{'cov%':>7}{'NDVI_m':>8}{'NDVI_s':>8}"
                f"{'NDWI_m':>8}{'NDBI_m':>8}{'cloud%':>8}  flag")
    logger.info(f"  {'-' * 72}")

    subset = df_f2[df_f2['territory'] == territory_name].sort_values('year')
    for _, row in subset.iterrows():
        n = int(row['n_scenes']) if pd.notna(row['n_scenes']) else 0
        cov = row['coverage_pct'] if pd.notna(row['coverage_pct']) else 0
        ndvi_m = row['NDVI_mean'] if pd.notna(row['NDVI_mean']) else 0
        ndvi_s = row['NDVI_std'] if pd.notna(row['NDVI_std']) else 0
        ndwi_m = row['NDWI_mean'] if pd.notna(row['NDWI_mean']) else 0
        ndbi_m = row['NDBI_mean'] if pd.notna(row['NDBI_mean']) else 0
        cloud = row['cloud_residue_pct'] if pd.notna(row['cloud_residue_pct']) else 0

        flags = []
        if cov < 90: flags.append('cov')
        if cloud > 1.0: flags.append('cloud')
        if ndvi_m < 0.3 or ndvi_m > 0.8: flags.append('NDVI')
        flag_str = ' '.join(flags)
        if flag_str:
            flag_str = '⚠ ' + flag_str

        target_marker = ' *' if row['is_target'] else '  '

        logger.info(f"  {int(row['year']):<6}{n:>4}{cov:>7.1f}{ndvi_m:>8.3f}"
                    f"{ndvi_s:>8.3f}{ndwi_m:>8.3f}{ndbi_m:>8.3f}{cloud:>8.2f}"
                    f"  {flag_str}{target_marker}")

logger.info("\n  * — целевой год (полная диагностика)")


# ============================================================
# ТАБЛИЦА 2: Распределение NDVI
# ============================================================
logger.info("")
logger.info("=" * 70)
logger.info("ТАБЛИЦА 2: РАСПРЕДЕЛЕНИЕ NDVI (только целевые годы)")
logger.info("=" * 70)

for territory_name in ['Historical', 'Greater']:
    logger.info(f"\n[{territory_name}]")
    logger.info(f"  {'Год':<6}{'p10':>8}{'p50':>8}{'p90':>8}{'bimod':>10}  интерпретация")
    logger.info(f"  {'-' * 65}")

    subset = df_f2[
        (df_f2['territory'] == territory_name) & (df_f2['is_target'])
    ].sort_values('year')

    for _, row in subset.iterrows():
        p10 = row.get('NDVI_p10')
        p50 = row.get('NDVI_p50')
        p90 = row.get('NDVI_p90')
        bimod = row.get('NDVI_bimodality')

        if pd.isna(p10) or pd.isna(p50) or pd.isna(p90):
            logger.info(f"  {int(row['year']):<6}{'—':>8}{'—':>8}{'—':>8}{'—':>10}")
            continue

        if bimod is None or pd.isna(bimod):
            interp = '—'
        elif abs(bimod) < 0.05:
            interp = 'плоское'
        elif bimod > 0.10:
            interp = 'двугорбое (норма)'
        elif bimod > 0.05:
            interp = 'слабо двугорбое'
        else:
            interp = 'скошено'

        logger.info(f"  {int(row['year']):<6}{p10:>8.3f}{p50:>8.3f}"
                    f"{p90:>8.3f}{bimod:>10.3f}  {interp}")


# ============================================================
# ТАБЛИЦА 3: Тренды и аномалии
# ============================================================
logger.info("")
logger.info("=" * 70)
logger.info("ТАБЛИЦА 3: ТРЕНДЫ И АНОМАЛИИ (все годы)")
logger.info("=" * 70)

trend_results = {}

for territory_name in ['Historical', 'Greater']:
    subset = df_f2[df_f2['territory'] == territory_name].sort_values('year')
    subset = subset.dropna(subset=['NDVI_mean'])

    if len(subset) < 3:
        logger.warning(f"[{territory_name}] Недостаточно данных для тренда")
        continue

    years = subset['year'].values.astype(float)
    ndvi = subset['NDVI_mean'].values.astype(float)

    coef = np.polyfit(years, ndvi, 1)
    slope, intercept = coef[0], coef[1]
    trend = np.polyval(coef, years)
    residuals = ndvi - trend

    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((ndvi - ndvi.mean()) ** 2)
    r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else 0

    if SCIPY_AVAILABLE:
        r, p_value = scipy_stats.pearsonr(years, ndvi)
    else:
        r, p_value = np.nan, np.nan

    anomaly_threshold = 0.1
    anomaly_mask = np.abs(residuals) > anomaly_threshold
    anomaly_years = subset[anomaly_mask]['year'].astype(int).tolist()

    trend_results[territory_name] = {
        'slope_per_year': float(slope),
        'intercept':      float(intercept),
        'r_squared':      float(r_squared),
        'pearson_r':      float(r) if not np.isnan(r) else None,
        'p_value':        float(p_value) if not np.isnan(p_value) else None,
        'anomaly_years':  anomaly_years,
    }

    logger.info(f"\n[{territory_name}]")
    logger.info(f"  Линейный тренд NDVI: {slope:+.5f}/год")
    logger.info(f"  R² = {r_squared:.3f}, Pearson r = {r:.3f}, p = {p_value:.4f}")
    if slope > 0.001:
        logger.success(f"  ✅ Растительность растёт: +{slope*10:.3f} за 10 лет")
    elif slope < -0.001:
        logger.warning(f"  ⚠️  Растительность снижается: {slope*10:.3f} за 10 лет")
    else:
        logger.info(f"  Тренд слабый (|slope| < 0.001)")

    if anomaly_years:
        logger.warning(f"  Аномалии (|residual| > {anomaly_threshold}): {anomaly_years}")
    else:
        logger.success(f"  ✅ Аномалий нет")


# ============================================================
# ТАБЛИЦА 4: Валидность обучающих точек
# ============================================================
logger.info("")
logger.info("=" * 70)
logger.info("ТАБЛИЦА 4: ВАЛИДНОСТЬ ОБУЧАЮЩИХ ТОЧЕК (только целевые годы)")
logger.info("=" * 70)
logger.info(f"Из {SAMPLE_POINTS_PER_CLASS} запрошенных точек/класс — сколько попало на валидные пиксели.")

for territory_name in ['Historical', 'Greater']:
    logger.info(f"\n[{territory_name}]")

    subset = df_f2[
        (df_f2['territory'] == territory_name) & (df_f2['is_target'])
    ].sort_values('year')

    for _, row in subset.iterrows():
        val = row.get('training_validation')
        if val is None:
            logger.warning(f"  {int(row['year'])}: нет данных валидации")
            continue

        year = int(row['year'])
        valid_pct = val['valid_pct']
        valid_total = val['valid_points_total']
        requested = val['requested_total']

        logger.info(f"  {year}: {valid_total}/{requested} точек валидны ({valid_pct:.0f}%)")
        logger.info(f"    {'Класс':<28}{'Точек':>8}{'NDVI_mean':>12}{'NDVI_std':>10}")
        logger.info(f"    {'-' * 58}")

        for cls_id in range(N_CLASSES):
            n = val['valid_points_per_class'].get(cls_id, 0)
            ndvi_m = val['NDVI_mean_per_class'].get(cls_id)
            ndvi_s = val['NDVI_std_per_class'].get(cls_id)

            ndvi_m_str = f"{ndvi_m:>12.3f}" if ndvi_m is not None else f"{'—':>12}"
            ndvi_s_str = f"{ndvi_s:>10.3f}" if ndvi_s is not None else f"{'—':>10}"

            logger.info(f"    {CLASSES_RU[cls_id]:<28}{n:>8}{ndvi_m_str}{ndvi_s_str}")

        sep = val.get('separability')
        if sep is not None:
            sep_marker = '✅' if sep > 0.15 else '⚠️' if sep > 0.05 else '❌'
            logger.info(f"    Separability (Dense − Sparse): {sep:+.3f} {sep_marker}")


# ============================================================
# ГРАФИК 1: NDVI_mean по годам
# ============================================================
logger.info("")
logger.info("Строю график 1: NDVI по годам...")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

for ax, territory_name in zip(axes, ['Historical', 'Greater']):
    subset = df_f2[df_f2['territory'] == territory_name].sort_values('year')
    color = (PLOT_STYLE['historical_color'] if territory_name == 'Historical'
             else PLOT_STYLE['greater_color'])

    ax.plot(subset['year'], subset['NDVI_mean'], linewidth=1.2,
            color=color, alpha=0.5, zorder=2)

    for _, row in subset.iterrows():
        if pd.isna(row['coverage_pct']):
            continue
        cov = row['coverage_pct']
        m_color = (PLOT_STYLE['ok_marker'] if cov >= 95
                   else PLOT_STYLE['warn_marker'] if cov >= 90
                   else PLOT_STYLE['error_marker'])
        size = 110 if row['is_target'] else 60
        edge = 'black' if row['is_target'] else 'none'
        lw = 1.0 if row['is_target'] else 0

        ax.scatter(row['year'], row['NDVI_mean'],
                   color=m_color, s=size, zorder=5,
                   edgecolors=edge, linewidths=lw)

    ax2 = ax.twinx()
    ax2.plot(subset['year'], subset['NDWI_mean'], marker='s',
             linewidth=1.0, markersize=3, alpha=0.5,
             color='steelblue', label='NDWI')
    ax2.plot(subset['year'], subset['NDBI_mean'], marker='^',
             linewidth=1.0, markersize=3, alpha=0.5,
             color='saddlebrown', label='NDBI')
    ax2.set_ylim(-1, 1)
    ax2.set_ylabel('NDWI / NDBI', color='gray', fontsize=9)

    ax.set_title(f'{territory_name}', fontsize=12)
    ax.set_xlabel('Год')
    ax.set_ylabel('NDVI mean')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=PLOT_STYLE['grid_alpha'])

    from matplotlib.lines import Line2D
    legend_elems = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='green',
               markersize=8, label='coverage ≥95%'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='orange',
               markersize=8, label='coverage 90–95%'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='red',
               markersize=8, label='coverage <90%'),
    ]
    ax.legend(handles=legend_elems, loc='lower left', fontsize=8)

plt.suptitle('NDVI mean по годам (крупные маркеры = целевые годы)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


# ============================================================
# ГРАФИК 2: p10-p90 + bimodality
# ============================================================
logger.info("Строю график 2: распределение NDVI...")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for ax, territory_name in zip(axes, ['Historical', 'Greater']):
    subset = df_f2[
        (df_f2['territory'] == territory_name) &
        (df_f2['is_target']) &
        (df_f2['NDVI_p50'].notna())
    ].sort_values('year')

    color = (PLOT_STYLE['historical_color'] if territory_name == 'Historical'
             else PLOT_STYLE['greater_color'])

    if subset.empty:
        ax.text(0.5, 0.5, 'Нет данных', ha='center', va='center')
        ax.set_title(territory_name)
        continue

    for _, row in subset.iterrows():
        year = row['year']
        ax.plot([year, year], [row['NDVI_p10'], row['NDVI_p90']],
                color=color, linewidth=6, alpha=0.4, solid_capstyle='round')
        ax.scatter(year, row['NDVI_p50'], color=color, s=100, zorder=5,
                   edgecolors='black', linewidths=1)

    for _, row in subset.iterrows():
        bimod = row['NDVI_bimodality']
        if pd.notna(bimod):
            ax.annotate(f'{bimod:.2f}',
                        (row['year'], row['NDVI_p90'] + 0.03),
                        ha='center', fontsize=8, color='gray')

    ax.set_title(f'{territory_name}: p10–p50–p90 (аннотация = bimodality)',
                 fontsize=12)
    ax.set_xlabel('Год')
    ax.set_ylabel('NDVI')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=PLOT_STYLE['grid_alpha'])

plt.suptitle('Распределение NDVI в целевые годы', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


# ============================================================
# ГРАФИК 3: Тренд + residuals
# ============================================================
logger.info("Строю график 3: тренд и residuals...")

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex='col')

for col, territory_name in enumerate(['Historical', 'Greater']):
    if territory_name not in trend_results:
        continue

    subset = df_f2[df_f2['territory'] == territory_name].sort_values('year')
    subset = subset.dropna(subset=['NDVI_mean'])

    color = (PLOT_STYLE['historical_color'] if territory_name == 'Historical'
             else PLOT_STYLE['greater_color'])

    years = subset['year'].values.astype(float)
    ndvi = subset['NDVI_mean'].values.astype(float)

    coef = np.polyfit(years, ndvi, 1)
    trend = np.polyval(coef, years)
    residuals = ndvi - trend

    ax = axes[0, col]
    ax.scatter(years, ndvi, s=50, color=color, zorder=5,
               edgecolors='black', linewidths=0.5, alpha=0.8)

    target_mask = subset['is_target'].values
    if target_mask.any():
        ax.scatter(years[target_mask], ndvi[target_mask],
                   s=140, facecolors='none', edgecolors='black',
                   linewidths=2, zorder=6, label='Целевые')

    ax.plot(years, trend, '--', linewidth=PLOT_STYLE['trend_linewidth'],
            color='black', alpha=0.7,
            label=f'Тренд: {coef[0]:+.5f}/год')

    tr = trend_results[territory_name]
    p_str = f"p = {tr['p_value']:.4f}" if tr['p_value'] is not None else "p = —"
    ax.text(0.02, 0.98,
            f"R² = {tr['r_squared']:.3f}\n{p_str}",
            transform=ax.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85),
            fontsize=9)

    ax.set_title(f'{territory_name}: NDVI и тренд', fontsize=12)
    ax.set_ylabel('NDVI mean')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=PLOT_STYLE['grid_alpha'])
    ax.legend(loc='lower left', fontsize=8)

    ax = axes[1, col]
    colors = [PLOT_STYLE['anomaly_marker'] if abs(r) > 0.1 else color
              for r in residuals]
    ax.bar(years, residuals, color=colors, alpha=0.75, width=0.7)
    ax.axhline(y=0, color='black', linewidth=0.8)
    ax.axhline(y=0.1, color='red', linestyle=':', alpha=0.5)
    ax.axhline(y=-0.1, color='red', linestyle=':', alpha=0.5)
    ax.set_title(f'{territory_name}: residuals', fontsize=12)
    ax.set_xlabel('Год')
    ax.set_ylabel('Residual')
    ax.grid(True, alpha=PLOT_STYLE['grid_alpha'])

plt.suptitle('Тренды NDVI и аномалии (красным = |residual| > 0.1)',
             fontsize=13, y=1.00)
plt.tight_layout()
plt.show()


# ============================================================
# ВЕРДИКТ F2
# ============================================================
logger.info("")
logger.info("=" * 70)
logger.info("ВЕРДИКТ F2")
logger.info("=" * 70)

issues = []

for territory_name in ['Historical', 'Greater']:
    subset = df_f2[df_f2['territory'] == territory_name]
    low_cov = subset[subset['coverage_pct'] < 90]
    if not low_cov.empty:
        years = low_cov['year'].astype(int).tolist()
        logger.warning(f"[{territory_name}] coverage <90%: {years}")
        issues.append(f"{territory_name}: низкое покрытие {years}")
    else:
        logger.success(f"[{territory_name}] ✅ coverage ≥90% во все годы")

for territory_name in ['Historical', 'Greater']:
    subset = df_f2[df_f2['territory'] == territory_name]
    high_cloud = subset[subset['cloud_residue_pct'] > 1.0]
    if not high_cloud.empty:
        years = high_cloud['year'].astype(int).tolist()
        logger.warning(f"[{territory_name}] cloud_residue >1%: {years}")
        issues.append(f"{territory_name}: остатки облаков {years}")
    else:
        logger.success(f"[{territory_name}] ✅ cloud_residue ≤1% во все годы")

for territory_name in ['Historical', 'Greater']:
    subset = df_f2[df_f2['territory'] == territory_name]
    unusual = subset[(subset['NDVI_mean'] < 0.3) | (subset['NDVI_mean'] > 0.8)]
    if not unusual.empty:
        years = unusual['year'].astype(int).tolist()
        logger.warning(f"[{territory_name}] необычный NDVI: {years}")
    else:
        logger.success(f"[{territory_name}] ✅ NDVI_mean в 0.3–0.8")

logger.info("")
logger.info("Валидность обучающих точек (целевые годы):")
for territory_name in ['Historical', 'Greater']:
    subset = df_f2[
        (df_f2['territory'] == territory_name) & (df_f2['is_target'])
    ]
    min_val_pct = 100
    min_val_year = None
    for _, row in subset.iterrows():
        val = row.get('training_validation')
        if val:
            if val['valid_pct'] < min_val_pct:
                min_val_pct = val['valid_pct']
                min_val_year = int(row['year'])

    if min_val_year is not None:
        if min_val_pct >= 70:
            logger.success(f"[{territory_name}] ✅ минимум валидных точек: "
                           f"{min_val_pct:.0f}% ({min_val_year})")
        elif min_val_pct >= 50:
            logger.warning(f"[{territory_name}] ⚠️  минимум валидных точек: "
                           f"{min_val_pct:.0f}% ({min_val_year})")
            issues.append(f"{territory_name}: мало валидных точек в {min_val_year}")
        else:
            logger.error(f"[{territory_name}] ❌ критично мало валидных точек: "
                         f"{min_val_pct:.0f}% ({min_val_year})")
            issues.append(f"{territory_name}: критично мало точек в {min_val_year}")

logger.info("")
if not issues:
    logger.success("=" * 70)
    logger.success("✅ F2 ПРОЙДЕН: все композиты и точки качественные")
    logger.success("=" * 70)
    logger.info("Полный прогон (Блок H) можно запускать.")
else:
    logger.warning("=" * 70)
    logger.warning(f"⚠️  F2: обнаружено {len(issues)} предупреждений")
    logger.warning("=" * 70)
    for issue in issues:
        logger.warning(f"  • {issue}")
    logger.info("Это НЕ блокирует полный прогон, но требует внимания")
    logger.info("при интерпретации результатов в отчёте.")


# ------------------------------------------------------------
# Итог
# ------------------------------------------------------------
logger.success("=" * 70)
logger.success("✅ F2 ЗАВЕРШЁН")
logger.success("=" * 70)
logger.info(f"Время выполнения: {elapsed_min:.1f} мин")
logger.info("Доступные переменные после этой ячейки:")
logger.info("  f2_data          — dict {territory: [list of stats]}")
logger.info("  df_f2            — единый DataFrame всех результатов")
logger.info("  trend_results    — тренды и аномалии")
logger.info("  f2_failed        — список неудачных итераций")
logger.info(f"  F2_FILE          — {F2_FILE.name}")
logger.info("")
logger.info("Следующий шаг: Блок H — полный прогон 8 целевых лет.")

In [ ]:
# ============================================================
# ТЕСТ: Landsat vs Sentinel-2 через HLS (изолированная ячейка)
# ============================================================
# НЕ трогает основной пайплайн. Создаёт собственные функции
# для работы с HLS. Использует те же FEATURES, CLASSES,
# ANALYSIS_CRS и ESA-точки (points_greater / points_hist).
# ============================================================

import ee
from loguru import logger
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

logger.info("=" * 70)
logger.info("ТЕСТ: HLS Landsat vs Sentinel-2 (изолированная ячейка)")
logger.info("=" * 70)

# ------------------------------------------------------------
# 1. Конфигурация теста
# ------------------------------------------------------------
TEST_TERRITORY = 'Greater'
TEST_YEARS = [2019, 2020, 2026]          # можно сузить до [2020]
HLS_L30 = 'NASA/HLS/HLSL30/v002'
HLS_S30 = 'NASA/HLS/HLSS30/v002'

# Соответствие каналов HLS (одинаково для L30 и S30)[reference:1]
HLS_BAND_MAP = {
    'blue':   'B2',
    'green':  'B3',
    'red':    'B4',
    'nir':    'B5',    # L30 использует B5; S30 — B8A (narrow NIR)[reference:2]
    'swir1':  'B6',
    'swir2':  'B7',
}
# Для S30 NIR — это B8A, а не B5
HLS_BAND_MAP_S30 = dict(HLS_BAND_MAP, nir='B8A')

HLS_SEASON_START = '06-01'
HLS_SEASON_END   = '09-01'

TEST_BOUNDARY = BOUNDARIES[TEST_TERRITORY]
TEST_POINTS   = points_greater if TEST_TERRITORY == 'Greater' else points_hist

logger.info(f"Территория: {TEST_TERRITORY}")
logger.info(f"Годы:       {TEST_YEARS}")
logger.info(f"Коллекции:  {HLS_L30} / {HLS_S30}")

# ------------------------------------------------------------
# 2. Функции для HLS
# ------------------------------------------------------------

def mask_hls(image):
    """
    Маска HLS через Fmask.
    Биты: 1=cloud, 2=adjacent to cloud/shadow,
          3=cloud shadow, 4=snow/ice[reference:3]
    Вода (бит 5) НЕ маскируется — она нам нужна как класс.
    """
    fmask = image.select('Fmask')
    invalid = (1 << 1) | (1 << 2) | (1 << 3) | (1 << 4)
    clear = fmask.bitwiseAnd(invalid).eq(0)
    return image.updateMask(clear)


def prep_hls_l30(image):
    """HLS Landsat: переименование B2..B7 → blue..swir2."""
    return (
        mask_hls(image)
        .select(
            [HLS_BAND_MAP['blue'], HLS_BAND_MAP['green'],
             HLS_BAND_MAP['red'], HLS_BAND_MAP['nir'],
             HLS_BAND_MAP['swir1'], HLS_BAND_MAP['swir2']],
            ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']
        )
        .set('hls_sensor', 'L30')
    )


def prep_hls_s30(image):
    """HLS Sentinel-2: B2,B3,B4,B8A,B11,B12 → blue..swir2."""
    return (
        mask_hls(image)
        .select(
            [HLS_BAND_MAP_S30['blue'], HLS_BAND_MAP_S30['green'],
             HLS_BAND_MAP_S30['red'], HLS_BAND_MAP_S30['nir'],
             HLS_BAND_MAP_S30['swir1'], HLS_BAND_MAP_S30['swir2']],
            ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']
        )
        .set('hls_sensor', 'S30')
    )


def build_hls_composite(collection_id, boundary, year,
                         prep_func, min_images=MIN_IMAGES_FOR_COMPOSITE):
    """Медианный композит HLS за лето года."""
    col = (
        ee.ImageCollection(collection_id)
        .filterBounds(boundary)
        .filterDate(f'{year}-{HLS_SEASON_START}',
                    f'{year}-{HLS_SEASON_END}')
        .map(prep_func)
    )
    n = col.size().getInfo()
    if n < min_images:
        logger.warning(f"  {collection_id.split('/')[-2]} {year}: "
                       f"только {n} снимков (<{min_images})")
    if n == 0:
        return None, 0
    return col.median().clip(boundary), n


def add_hls_features(composite):
    """
    Добавляет к HLS-композиту те же 11 признаков,
    что и в основном пайплайне: 7 индексов + 2 текстурных.
    (6 каналов уже есть.)
    """
    blue  = composite.select('blue')
    green = composite.select('green')
    red   = composite.select('red')
    nir   = composite.select('nir')
    swir1 = composite.select('swir1')

    ndvi  = nir.subtract(red).divide(nir.add(red)).clamp(-1, 1).rename('NDVI')
    ndwi  = green.subtract(nir).divide(green.add(nir)).clamp(-1, 1).rename('NDWI')
    ndbi  = swir1.subtract(nir).divide(swir1.add(nir)).clamp(-1, 1).rename('NDBI')
    bsi   = (swir1.add(red).subtract(nir.add(blue))
             .divide(swir1.add(red).add(nir).add(blue))
             .clamp(-1, 1).rename('BSI'))
    savi  = (nir.subtract(red).divide(nir.add(red).add(0.5))
             .multiply(1.5).clamp(-1, 1).rename('SAVI'))
    ndmi  = nir.subtract(swir1).divide(nir.add(swir1)).clamp(-1, 1).rename('NDMI')
    mndwi = green.subtract(swir1).divide(green.add(swir1)).clamp(-1, 1).rename('MNDWI')

    # GLCM на NIR (8-bit)
    nir_8 = composite.select('nir').multiply(255).clamp(0, 255).toByte()
    glcm  = nir_8.glcmTexture(size=3)
    nir_contrast = glcm.select('nir_contrast').rename('NIR_contrast')
    nir_entropy  = glcm.select('nir_ent').rename('NIR_entropy')

    return composite.addBands([
        ndvi, ndwi, ndbi, bsi, savi, ndmi, mndwi,
        nir_contrast, nir_entropy
    ])


# ------------------------------------------------------------
# 3. Основной тестовый цикл
# ------------------------------------------------------------
test_results = []

for year in TEST_YEARS:
    logger.info("-" * 70)
    logger.info(f"ГОД {year}")
    logger.info("-" * 70)

    # --- 3.1. Композиты ---
    t0 = time.time()
    comp_l30, n_l30 = build_hls_composite(
        HLS_L30, TEST_BOUNDARY, year, prep_hls_l30
    )
    comp_s30, n_s30 = build_hls_composite(
        HLS_S30, TEST_BOUNDARY, year, prep_hls_s30
    )
    if comp_l30 is None or comp_s30 is None:
        logger.error(f"  Композиты не созданы (L30={n_l30}, S30={n_s30})")
        continue

    comp_l30 = add_hls_features(comp_l30)
    comp_s30 = add_hls_features(comp_s30)
    logger.info(f"  Композиты: L30={n_l30} снимков, S30={n_s30} снимков "
                f"({time.time()-t0:.1f} с)")

    # --- 3.2. Общие точки (ESA) ---
    # Используем TEST_POINTS, но сэмплируем на S30 (там больше шансов
    # на валидные пиксели — меньше пропусков из-за облаков)
    train_samples = comp_s30.select(FEATURES).sampleRegions(
        collection=TEST_POINTS, properties=['class'],
        scale=30, projection=ee.Projection(ANALYSIS_CRS),
        geometries=False, tileScale=4
    )
    total = train_samples.size().getInfo()
    with_random = train_samples.randomColumn('r', seed=RANDOM_SEED)
    train = with_random.filter(ee.Filter.lt('r', TRAIN_FRACTION))
    test  = with_random.filter(ee.Filter.gte('r', TRAIN_FRACTION))
    train_n = train.size().getInfo()
    test_n  = test.size().getInfo()
    logger.info(f"  Образцы: всего={total}, train={train_n}, test={test_n}")

    if train_n == 0 or test_n == 0:
        logger.error(f"  Недостаточно образцов — пропускаю год {year}")
        continue

    # --- 3.3. Обучение на S30 ---
    clf_s30 = ee.Classifier.smileRandomForest(
        numberOfTrees=RF_TREES,
        minLeafPopulation=RF_MIN_LEAF,
        bagFraction=RF_BAG_FRACTION
    ).train(features=train, classProperty='class', inputProperties=FEATURES)

    # --- 3.4. Классификация L30 обученным на S30 ---
    classified_l30 = comp_l30.select(FEATURES).classify(clf_s30)
    # Валидация: тест-точки классифицируем на L30, но истина — из ESA
    # (test-набор получен из S30, но метки 'class' одинаковые)
    test_on_l30 = test.classify(clf_s30, outputName='pred')
    acc_l30 = test_on_l30.errorMatrix('class', 'pred')
    oa_l30 = acc_l30.accuracy().getInfo()
    kappa_l30 = acc_l30.kappa().getInfo()

    # --- 3.5. Обучение на L30 (бенчмарк) ---
    train_l30 = comp_l30.select(FEATURES).sampleRegions(
        collection=TEST_POINTS, properties=['class'],
        scale=30, projection=ee.Projection(ANALYSIS_CRS),
        geometries=False, tileScale=4
    ).randomColumn('r', seed=RANDOM_SEED)
    train_l30 = train_l30.filter(ee.Filter.lt('r', TRAIN_FRACTION))
    test_l30  = train_l30.filter(ee.Filter.gte('r', TRAIN_FRACTION))

    clf_l30 = ee.Classifier.smileRandomForest(
        numberOfTrees=RF_TREES,
        minLeafPopulation=RF_MIN_LEAF,
        bagFraction=RF_BAG_FRACTION
    ).train(features=train_l30, classProperty='class',
            inputProperties=FEATURES)

    test_l30_classified = test_l30.classify(clf_l30, outputName='pred')
    acc_l30_bench = test_l30_classified.errorMatrix('class', 'pred')
    oa_l30_bench = acc_l30_bench.accuracy().getInfo()
    kappa_l30_bench = acc_l30_bench.kappa().getInfo()

    # --- 3.6. Обучение на L30, классификация S30 (обратный сценарий) ---
    classified_s30 = comp_s30.select(FEATURES).classify(clf_l30)
    test_on_s30 = test.classify(clf_l30, outputName='pred')
    acc_s30 = test_on_s30.errorMatrix('class', 'pred')
    oa_s30 = acc_s30.accuracy().getInfo()
    kappa_s30 = acc_s30.kappa().getInfo()

    # --- 3.7. Сравнение спектров L30 vs S30 на одних пикселях ---
    # Берём подвыборку точек и сэмплируем оба композита
    sample_pts = TEST_POINTS.randomColumn('r', seed=RANDOM_SEED) \
                            .sort('r').limit(3000)
    spec_l30 = comp_l30.select(FEATURES).sampleRegions(
        collection=sample_pts, properties=['class'],
        scale=30, projection=ee.Projection(ANALYSIS_CRS),
        geometries=False, tileScale=4
    ).getInfo().get('features', [])
    spec_s30 = comp_s30.select(FEATURES).sampleRegions(
        collection=sample_pts, properties=['class'],
        scale=30, projection=ee.Projection(ANALYSIS_CRS),
        geometries=False, tileScale=4
    ).getInfo().get('features', [])

    df_l30 = pd.DataFrame([f['properties'] for f in spec_l30])
    df_s30 = pd.DataFrame([f['properties'] for f in spec_s30])

    # Мержим по индексу (один и тот же набор точек)
    common = pd.DataFrame()
    for band in FEATURES:
        if band in df_l30.columns and band in df_s30.columns:
            common[f'{band}_L30'] = df_l30[band].values[:len(df_s30)]
            common[f'{band}_S30'] = df_s30[band].values[:len(df_l30)]

    # --- 3.8. Запись результата ---
    result = {
        'year': year,
        'oa_train_S30_apply_L30': oa_l30,
        'kappa_train_S30_apply_L30': kappa_l30,
        'oa_train_L30_apply_L30': oa_l30_bench,
        'kappa_train_L30_apply_L30': kappa_l30_bench,
        'oa_train_L30_apply_S30': oa_s30,
        'kappa_train_L30_apply_S30': kappa_s30,
        'n_L30': n_l30,
        'n_S30': n_s30,
        'train_n': train_n,
        'test_n': test_n,
        'common_spectra': common,
    }
    test_results.append(result)

    logger.info(f"  РЕЗУЛЬТАТЫ {year}:")
    logger.info(f"    Обучение S30 → применение L30: "
                f"OA={oa_l30:.4f}, Kappa={kappa_l30:.4f}")
    logger.info(f"    Обучение L30 → применение L30: "
                f"OA={oa_l30_bench:.4f}, Kappa={kappa_l30_bench:.4f}")
    logger.info(f"    Обучение L30 → применение S30: "
                f"OA={oa_s30:.4f}, Kappa={kappa_s30:.4f}")
    logger.info(f"    Δ (S30→L30 vs L30→L30): "
                f"{oa_l30 - oa_l30_bench:+.4f}")

# ------------------------------------------------------------
# 4. Сводка и графики
# ------------------------------------------------------------
logger.info("=" * 70)
logger.info("СВОДКА ТЕСТА HLS")
logger.info("=" * 70)

if not test_results:
    logger.error("Тест не дал результатов.")
else:
    df_res = pd.DataFrame([
        {k: v for k, v in r.items() if k != 'common_spectra'}
        for r in test_results
    ])

    logger.info(f"\n{df_res.to_string(index=False)}")

    # --- График 1: OA по сценариям ---
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(df_res))
    w = 0.25
    ax.bar(x - w, df_res['oa_train_S30_apply_L30'], w,
           label='S30→L30 (сценарий заказчика)', color='#d62728')
    ax.bar(x,     df_res['oa_train_L30_apply_L30'], w,
           label='L30→L30 (бенчмарк)', color='#1f77b4')
    ax.bar(x + w, df_res['oa_train_L30_apply_S30'], w,
           label='L30→S30 (обратный)', color='#2ca02c')
    ax.axhline(y=ACCURACY_THRESHOLD, color='green', linestyle='--',
               alpha=0.6, label=f'Цель ({ACCURACY_THRESHOLD})')
    ax.set_xticks(x)
    ax.set_xticklabels(df_res['year'])
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('Overall Accuracy')
    ax.set_title(f'{TEST_TERRITORY}: Landsat vs Sentinel-2 через HLS')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

    # --- График 2: разница OA (S30→L30 − L30→L30) ---
    fig, ax = plt.subplots(figsize=(8, 4))
    delta = df_res['oa_train_S30_apply_L30'] - df_res['oa_train_L30_apply_L30']
    colors = ['green' if d > 0 else 'red' for d in delta]
    ax.bar(df_res['year'].astype(str), delta, color=colors, alpha=0.8)
    ax.axhline(y=0, color='black', linewidth=0.8)
    ax.axhline(y=0.03, color='green', linestyle=':', alpha=0.5,
               label='Порог +3 п.п.')
    ax.axhline(y=-0.03, color='red', linestyle=':', alpha=0.5)
    ax.set_ylabel('Δ OA (S30→L30 − L30→L30)')
    ax.set_title('Прирост от обучения на Sentinel-2')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

    # --- График 3: спектральные различия по каналам ---
    if test_results and not test_results[0]['common_spectra'].empty:
        last = test_results[-1]  # последний год
        common = last['common_spectra']
        bands = [f for f in SPECTRAL_BANDS + INDEX_BANDS
                 if f'{f}_L30' in common.columns]

        fig, axes = plt.subplots(1, len(bands), figsize=(3*len(bands), 4))
        if len(bands) == 1:
            axes = [axes]
        for ax, band in zip(axes, bands):
            l30_vals = common[f'{band}_L30'].dropna()
            s30_vals = common[f'{band}_S30'].dropna()
            ax.scatter(l30_vals, s30_vals, s=3, alpha=0.3)
            lim = [min(l30_vals.min(), s30_vals.min()),
                   max(l30_vals.max(), s30_vals.max())]
            ax.plot(lim, lim, 'r--', alpha=0.5, label='1:1')
            ax.set_xlabel(f'{band} L30')
            ax.set_ylabel(f'{band} S30')
            ax.set_title(band)
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=7)
        plt.suptitle(f'Спектральные различия L30 vs S30 ({last["year"]})',
                     fontsize=12, y=1.02)
        plt.tight_layout()
        plt.show()

# ------------------------------------------------------------
# 5. Вердикт
# ------------------------------------------------------------
logger.info("=" * 70)
logger.info("ВЕРДИКТ ТЕСТА HLS")
logger.info("=" * 70)

if test_results:
    deltas = [r['oa_train_S30_apply_L30'] - r['oa_train_L30_apply_L30']
              for r in test_results]
    mean_delta = np.mean(deltas)

    if mean_delta > 0.03:
        logger.success(f"✅ Прирост OA от Sentinel-2: {mean_delta:+.4f} "
                       f"(> 3 п.п.) — стоит рассмотреть переход на HLS")
    elif mean_delta > 0:
        logger.info(f"Прирост OA: {mean_delta:+.4f} — незначительный, "
                    f"Landsat остаётся оправданным")
    else:
        logger.warning(f"⚠️  OA упал: {mean_delta:+.4f} — обучение на "
                       f"Sentinel-2 НЕ помогает для этой территории")
else:
    logger.error("Вердикт невозможен — нет результатов.")

logger.info("")
logger.info("ВАЖНО: этот тест НЕ влияет на results_by_year и основной пайплайн.")
logger.info("Для полного перехода на HLS потребуется переписать:")
logger.info("  • preprocess_landsat → prep_hls (другая маска, другие каналы)")
logger.info("  • create_composite → build_hls_composite + add_hls_features")
logger.info("  • build_collection_for_year → HLS-коллекции")
logger.info("  • generate_training_data → ESA уже 10 м, но HLS 30 м —")
logger.info("    нужно решить, сэмплировать на S30 или L30")

# БЛОК H. Полный запуск

In [ ]:
# ============================================================
# ЯЧЕЙКА H1: ПОЛНЫЙ ПРОГОН 8 ЦЕЛЕВЫХ ЛЕТ
# ============================================================
# Запускаем process_year для:
#   • Historical: 2000, 2007, 2013, 2019, 2026
#   • Greater:    2013, 2019, 2026
#
# Особенности:
#   • tqdm прогресс-бар
#   • Переиспользуем points_hist / points_greater из E6
#     (precomputed_points) — экономим ~3–4 мин
#   • Автоповтор при ошибке: 3 попытки с задержкой 30/60/90 с
#   • Промежуточное сохранение JSON после каждого года
#   • Итоговый results_by_year + results_by_year.json
# ============================================================

import time
from tqdm.auto import tqdm

logger.info("=" * 70)
logger.info("H1: ПОЛНЫЙ ПРОГОН 8 ЦЕЛЕВЫХ ЛЕТ")
logger.info("=" * 70)

# ------------------------------------------------------------
# 0. Подготовка: что прогоняем
# ------------------------------------------------------------
RUN_PLAN = []
for year in HISTORICAL_YEARS:
    RUN_PLAN.append(('Historical', year))
for year in GREATER_YEARS:
    RUN_PLAN.append(('Greater', year))

logger.info(f"План прогона ({len(RUN_PLAN)} итераций):")
for terr, yr in RUN_PLAN:
    logger.info(f"  {terr:<12} {yr}")

# Карты границ и предвычисленных точек
BOUNDARIES_MAP = {
    'Historical': historical_boundary,
    'Greater':    greater_moscow_boundary,
}
PRECOMPUTED_POINTS = {
    'Historical': points_hist,
    'Greater':    points_greater,
}
PRECOMPUTED_INFO = {
    'Historical': info_hist,
    'Greater':    info_greater,
}

# ------------------------------------------------------------
# 1. Файл для промежуточных результатов
# ------------------------------------------------------------
PARTIAL_FILE = PATHS['INTERMEDIATE'] / 'h1_partial_results.json'
FINAL_FILE   = PATHS['INTERMEDIATE'] / 'results_by_year.json'


def json_safe(obj):
    """Преобразует dict в JSON-совместимый вид (без ee.Image, np-типов)."""
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            # Пропускаем ee.Image — они всё равно не сериализуются
            if hasattr(v, 'getInfo') and not isinstance(v, (int, float, str)):
                out[k] = f"<ee.Image {getattr(v, 'id', 'unknown')}>"
            else:
                out[k] = json_safe(v)
        return out
    elif isinstance(obj, list):
        return [json_safe(x) for x in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj


def save_partial(results, failed, path):
    """Сохраняет промежуточные результаты в JSON."""
    payload = {
        'run_timestamp': RUN_TIMESTAMP,
        'results': json_safe(results),
        'failed': failed,
    }
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    return path


# ------------------------------------------------------------
# 2. Основной прогон с автоповтором
# ------------------------------------------------------------
results_by_year = {'Historical': {}, 'Greater': {}}
failed_runs = []

MAX_RETRIES = 3
RETRY_DELAYS = [30, 60, 90]   # секунд между попытками

t_h1_start = time.time()

pbar = tqdm(RUN_PLAN, desc="Прогон лет", unit="год")

for territory_name, year in pbar:
    pbar.set_description(f"{territory_name} {year}")

    boundary = BOUNDARIES_MAP[territory_name]
    points   = PRECOMPUTED_POINTS[territory_name]
    train_info = PRECOMPUTED_INFO[territory_name]

    result = None
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            logger.info(f"[H1] {territory_name} {year}: попытка {attempt}/{MAX_RETRIES}")
            result = process_year(
                boundary=boundary,
                year=year,
                territory_name=territory_name,
                precomputed_points=points,
                precomputed_training_info=train_info,
                verbose=True
            )
            if result is not None:
                break
        except Exception as e:
            last_error = str(e)
            logger.error(f"[H1] {territory_name} {year}: попытка {attempt} провалилась: {e}")

        if attempt < MAX_RETRIES:
            delay = RETRY_DELAYS[attempt - 1]
            logger.warning(f"[H1] {territory_name} {year}: жду {delay} с перед повтором...")
            time.sleep(delay)

    # --- Обработка результата ---
    if result is None:
        logger.error(f"[H1] {territory_name} {year}: ВСЕ {MAX_RETRIES} ПОПЫТКИ ПРОВАЛИЛИСЬ")
        failed_runs.append({
            'territory': territory_name,
            'year': year,
            'error': last_error,
        })
        save_partial(results_by_year, failed_runs, PARTIAL_FILE)
        continue

    # --- Проверка консистентности ---
    area_sum = sum(result['areas'].values())
    boundary_area = BOUNDARY_AREAS[territory_name]
    area_diff = abs(area_sum - boundary_area)
    area_diff_pct = area_diff / boundary_area * 100

    if area_diff_pct > 2.0:
        logger.warning(f"[H1] {territory_name} {year}: сумма площадей "
                       f"{area_sum:.1f} ≠ границы {boundary_area:.1f} "
                       f"(Δ {area_diff_pct:.2f}%)")

    if result['oa'] < 0.75:
        logger.warning(f"[H1] {territory_name} {year}: OA низкая = {result['oa']:.4f}")

    results_by_year[territory_name][year] = result
    logger.success(f"[H1] {territory_name} {year}: OK "
                   f"(OA={result['oa']:.4f}, Kappa={result['kappa']:.4f}, "
                   f"площадь={result['total_area']:.1f} км²)")

    # --- Промежуточное сохранение после каждого года ---
    save_partial(results_by_year, failed_runs, PARTIAL_FILE)
    logger.debug(f"[H1] Промежуточные результаты сохранены: {PARTIAL_FILE.name}")

elapsed_h1_min = (time.time() - t_h1_start) / 60

# ------------------------------------------------------------
# 3. Финальное сохранение
# ------------------------------------------------------------
save_partial(results_by_year, failed_runs, FINAL_FILE)

logger.info("")
logger.success("=" * 70)
logger.success(f"✅ H1 ЗАВЕРШЁН за {elapsed_h1_min:.1f} мин")
logger.success("=" * 70)

# Сводка
n_success = sum(len(v) for v in results_by_year.values())
n_total = len(RUN_PLAN)

logger.info(f"Успешно: {n_success}/{n_total}")

if failed_runs:
    logger.warning(f"❌ Провалившихся прогонов: {len(failed_runs)}")
    for f in failed_runs:
        logger.warning(f"  {f['territory']} {f['year']}: {f['error'][:100]}")
    logger.info("Эти годы нужно перезапустить вручную.")
else:
    logger.success("✅ Все 8 прогонов успешны!")

logger.info("")
logger.info("Доступные переменные после этой ячейки:")
logger.info("  results_by_year   — dict {territory: {year: result}}")
logger.info(f"  failed_runs       — список провалов ({len(failed_runs)})")
logger.info(f"  PARTIAL_FILE      — {PARTIAL_FILE.name}")
logger.info(f"  FINAL_FILE        — {FINAL_FILE.name}")

In [ ]:
# ============================================================
# ЯЧЕЙКА H2: ПОЛНАЯ СВОДКА МЕТРИК И АНАЛИЗ
# ============================================================
# Что показывает:
#   1. Baseline для сравнения (11 признаков из прошлого прогона)
#   2. Таблица метрик с Δ vs baseline
#   3. Per-class метрики (Precision/Recall/F1) по годам
#   4. Separability Dense↔Sparse
#   5. Feature importance (топ-5 + полный список)
#   6. Bar chart OA с baseline
#   7. Line chart OA с трендом
#   8. Confusion matrix heatmap (все 8 лет)
#   9. Расширенный вердикт
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

logger.info("=" * 70)
logger.info("H2: ПОЛНАЯ СВОДКА МЕТРИК")
logger.info("=" * 70)

# ------------------------------------------------------------
# 0. Baseline для сравнения (11 признаков, прошлый прогон)
# ------------------------------------------------------------
# Значения из прошлого H1 (11 признаков). Используются для Δ.
BASELINE_11 = {
    'Historical': {
        2000: {'oa': 0.7130, 'kappa': 0.6172},
        2007: {'oa': 0.7299, 'kappa': 0.6238},
        2013: {'oa': 0.7346, 'kappa': 0.6482},
        2019: {'oa': 0.7402, 'kappa': 0.6536},
        2026: {'oa': 0.7163, 'kappa': 0.6252},
    },
    'Greater': {
        2013: {'oa': 0.8032, 'kappa': 0.7191},
        2019: {'oa': 0.8021, 'kappa': 0.7361},
        2026: {'oa': 0.7963, 'kappa': 0.7285},
    },
}

# ------------------------------------------------------------
# 1. ТАБЛИЦА 1: Метрики с Δ vs baseline
# ------------------------------------------------------------
logger.info("")
logger.info("=" * 70)
logger.info("ТАБЛИЦА 1: МЕТРИКИ С СРАВНЕНИЕМ С BASELINE (11 признаков)")
logger.info("=" * 70)

for territory_name in ['Historical', 'Greater']:
    subset = results_by_year.get(territory_name, {})
    if not subset:
        logger.warning(f"[{territory_name}] Нет результатов")
        continue

    logger.info(f"\n[{territory_name}]")
    logger.info(f"  {'Год':<6}{'OA':>9}{'ΔOA':>9}{'Kappa':>9}{'ΔKappa':>9}"
                f"{'Dense':>8}{'Sparse':>8}{'BuiltUp':>8}{'Water':>8}")
    logger.info(f"  {'-' * 76}")

    for year in sorted(subset.keys()):
        r = subset[year]
        oa = r['oa']
        kappa = r['kappa']
        a = r['areas']

        baseline = BASELINE_11.get(territory_name, {}).get(year)
        if baseline:
            delta_oa = oa - baseline['oa']
            delta_kappa = kappa - baseline['kappa']
            # Значок улучшения
            oa_marker = '✅' if delta_oa > 0.01 else ('⚠️' if delta_oa < -0.01 else '—')
            oa_str = f"{oa:>8.4f}"
            delta_oa_str = f"{delta_oa:>+8.4f}{oa_marker}"
            delta_kappa_str = f"{delta_kappa:>+9.4f}"
        else:
            oa_str = f"{oa:>8.4f} 🟢" if oa >= 0.85 else (f"{oa:>8.4f} 🟡" if oa >= 0.75 else f"{oa:>8.4f} 🔴")
            delta_oa_str = f"{'—':>9}"
            delta_kappa_str = f"{'—':>9}"

        logger.info(
            f"  {year:<6}{oa_str}"
            f"{delta_oa_str}{kappa:>9.4f}{delta_kappa_str}"
            f"{a.get(0, 0):>8.1f}{a.get(1, 0):>8.1f}"
            f"{a.get(2, 0):>8.1f}{a.get(3, 0):>8.1f}"
        )

    oa_mean = np.mean([subset[y]['oa'] for y in subset])
    logger.info(f"  {'-' * 76}")
    logger.info(f"  Среднее OA = {oa_mean:.4f}")

# Сводка Δ по всем годам
logger.info("")
logger.info("СВОДКА УЛУЧШЕНИЙ")
for territory_name in ['Historical', 'Greater']:
    subset = results_by_year.get(territory_name, {})
    if not subset:
        continue

    deltas = []
    for year, r in subset.items():
        baseline = BASELINE_11.get(territory_name, {}).get(year)
        if baseline:
            deltas.append(r['oa'] - baseline['oa'])

    if deltas:
        mean_delta = np.mean(deltas)
        max_delta = np.max(deltas)
        min_delta = np.min(deltas)

        if mean_delta > 0.02:
            logger.success(f"[{territory_name}] ✅ Средний прирост OA: "
                           f"{mean_delta:+.4f} (макс +{max_delta:.4f}, мин {min_delta:+.4f})")
        elif mean_delta > 0:
            logger.info(f"[{territory_name}] Прирост OA: {mean_delta:+.4f}")
        else:
            logger.warning(f"[{territory_name}] ⚠️  Средний OA упал: {mean_delta:+.4f}")


# ------------------------------------------------------------
# 2. ТАБЛИЦА 2: Per-class метрики (Precision/Recall/F1)
# ------------------------------------------------------------
logger.info("")
logger.info("=" * 70)
logger.info("ТАБЛИЦА 2: PER-CLASS МЕТРИКИ")
logger.info("=" * 70)

def compute_per_class_metrics(matrix, classes):
    """Precision/Recall/F1 из confusion matrix."""
    n = len(matrix)
    result = {}
    for i in range(n):
        tp = matrix[i][i]
        fp = sum(matrix[j][i] for j in range(n) if j != i)
        fn = sum(matrix[i][j] for j in range(n) if j != i)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = (2 * precision * recall / (precision + recall)
              if (precision + recall) > 0 else 0)
        result[classes[i]] = {'precision': precision, 'recall': recall, 'f1': f1}
    return result


for territory_name in ['Historical', 'Greater']:
    subset = results_by_year.get(territory_name, {})
    if not subset:
        continue

    logger.info(f"\n[{territory_name}]")

    for year in sorted(subset.keys()):
        r = subset[year]
        metrics = compute_per_class_metrics(r['confusion_matrix'], CLASSES)

        logger.info(f"\n  {year}:")
        logger.info(f"    {'Класс':<28}{'Precision':>12}{'Recall':>12}{'F1':>12}")
        logger.info(f"    {'-' * 64}")

        for cls_name in CLASSES:
            m = metrics[cls_name]
            f1_marker = '✅' if m['f1'] >= 0.80 else ('⚠️' if m['f1'] >= 0.60 else '❌')
            logger.info(f"    {cls_name:<28}{m['precision']:>12.3f}"
                        f"{m['recall']:>12.3f}{m['f1']:>11.3f} {f1_marker}")


# ------------------------------------------------------------
# 3. ТАБЛИЦА 3: Separability Dense ↔ Sparse
# ------------------------------------------------------------
logger.info("")
logger.info("=" * 70)
logger.info("ТАБЛИЦА 3: SEPARABILITY Dense ↔ Sparse (главная проблема)")
logger.info("=" * 70)
logger.info("Separability = F1(Dense) и F1(Sparse) — чем выше, тем лучше.")

for territory_name in ['Historical', 'Greater']:
    subset = results_by_year.get(territory_name, {})
    if not subset:
        continue

    logger.info(f"\n[{territory_name}]")
    logger.info(f"  {'Год':<6}{'F1(Dense)':>12}{'F1(Sparse)':>12}{'Среднее':>12}  {'вердикт'}")
    logger.info(f"  {'-' * 62}")

    for year in sorted(subset.keys()):
        r = subset[year]
        metrics = compute_per_class_metrics(r['confusion_matrix'], CLASSES)
        f1_dense = metrics['Dense Vegetation']['f1']
        f1_sparse = metrics['Sparse Vegetation']['f1']
        avg = (f1_dense + f1_sparse) / 2

        if avg >= 0.80:
            verdict = '✅ отлично'
        elif avg >= 0.65:
            verdict = '⚠️ приемлемо'
        else:
            verdict = '❌ плохо'

        logger.info(f"  {year:<6}{f1_dense:>12.3f}{f1_sparse:>12.3f}"
                    f"{avg:>12.3f}  {verdict}")


# ------------------------------------------------------------
# 4. FEATURE IMPORTANCE
# ------------------------------------------------------------
logger.info("")
logger.info("=" * 70)
logger.info("FEATURE IMPORTANCE: важность признаков в Random Forest")
logger.info("=" * 70)

# Собираем все importance из результатов
all_importances = {}
for territory_name in ['Historical', 'Greater']:
    subset = results_by_year.get(territory_name, {})
    for year, r in subset.items():
        fi = r.get('feature_importance')
        if fi:
            all_importances[(territory_name, year)] = fi

if not all_importances:
    logger.warning("Feature importance не собрана. Проверьте, что E7 обновлён.")
else:
    # --- Топ-5 по каждой итерации ---
    for (territory_name, year), fi in sorted(all_importances.items()):
        top5 = sorted(fi.items(), key=lambda x: x[1], reverse=True)[:5]
        total = sum(fi.values())

        logger.info(f"\n[{territory_name} {year}]")
        logger.info(f"  {'Признак':<18}{'Важность':>12}{'Доля %':>10}")
        logger.info(f"  {'-' * 40}")
        for feat, imp in top5:
            pct = imp / total * 100
            logger.info(f"  {feat:<18}{imp:>12.4f}{pct:>9.1f}%")

    # --- Средняя важность по всем годам ---
    logger.info("")
    logger.info("-" * 70)
    logger.info("СРЕДНЯЯ ВАЖНОСТЬ ПРИЗНАКОВ (по всем 8 итерациям)")

    # Собираем усреднённую важность
    avg_importance = {f: [] for f in FEATURES}
    for fi in all_importances.values():
        for feat in FEATURES:
            if feat in fi:
                avg_importance[feat].append(fi[feat])

    # Считаем среднее и нормализуем
    mean_imp = {f: np.mean(v) for f, v in avg_importance.items() if v}
    total_mean = sum(mean_imp.values())

    if total_mean > 0:
        # Сортируем по убыванию
        sorted_imp = sorted(mean_imp.items(), key=lambda x: x[1], reverse=True)

        logger.info(f"  {'Место':<6}{'Признак':<18}{'Средняя важность':>18}{'Доля':>8}")
        logger.info(f"  {'-' * 52}")

        for rank, (feat, imp) in enumerate(sorted_imp, 1):
            pct = imp / total_mean * 100
            # Определяем тип признака
            if feat in SPECTRAL_BANDS:
                type_mark = '📷'
            elif feat in TEXTURE_BANDS:
                type_mark = '🧩'
            else:
                type_mark = '📊'

            marker = '⭐' if rank <= 5 else '  '
            logger.info(f"  {rank:<6}{feat:<18}{imp:>18.4f}{pct:>7.1f}%  {type_mark}{marker}")

        # Проверка: попали ли текстурные в топ-5?
        top5_names = [f for f, _ in sorted_imp[:5]]
        texture_in_top5 = [f for f in top5_names if f in TEXTURE_BANDS]

        logger.info("")
        if texture_in_top5:
            logger.success(f"✅ Текстурные признаки в топ-5: {texture_in_top5}")
        else:
            logger.warning(f"⚠️  Текстурные признаки не попали в топ-5")
            # Показываем их место
            for f in TEXTURE_BANDS:
                if f in mean_imp:
                    rank = [name for name, _ in sorted_imp].index(f) + 1
                    logger.info(f"     {f}: место {rank} из {len(sorted_imp)}")


# ------------------------------------------------------------
# 5. Bar chart OA + baseline
# ------------------------------------------------------------
logger.info("")
logger.info("Строю bar chart OA с baseline...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, territory_name in zip(axes, ['Historical', 'Greater']):
    subset = results_by_year.get(territory_name, {})
    if not subset:
        ax.text(0.5, 0.5, 'Нет данных', ha='center', va='center')
        ax.set_title(territory_name)
        continue

    years = sorted(subset.keys())
    oa_new = [subset[y]['oa'] for y in years]
    oa_base = [BASELINE_11[territory_name][y]['oa'] for y in years
               if y in BASELINE_11.get(territory_name, {})]

    color = (PLOT_STYLE['historical_color'] if territory_name == 'Historical'
             else PLOT_STYLE['greater_color'])

    x = np.arange(len(years))
    width = 0.38

    # Baseline (полупрозрачный)
    ax.bar(x - width/2, oa_base, width, label='Baseline (11 признаков)',
           color=color, alpha=0.35, edgecolor='gray')
    # Новая версия
    bars_new = ax.bar(x + width/2, oa_new, width, label='С 15 признаками',
                     color=color, alpha=0.85, edgecolor='black')

    # Порог
    ax.axhline(y=ACCURACY_THRESHOLD, color='green', linestyle='--',
               alpha=0.6, linewidth=1.5, label=f'Цель ({ACCURACY_THRESHOLD})')

    # Подписи значений
    for bar, val in zip(bars_new, oa_new):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f'{val:.3f}', ha='center', fontsize=8, fontweight='bold')

    # Δ сверху
    for i, (y, val) in enumerate(zip(years, oa_new)):
        baseline = BASELINE_11.get(territory_name, {}).get(y)
        if baseline:
            delta = val - baseline['oa']
            color_delta = 'green' if delta > 0 else 'red'
            ax.text(i, max(val, baseline['oa']) + 0.04,
                    f'Δ{delta:+.3f}', ha='center', fontsize=8,
                    color=color_delta, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(years)
    ax.set_ylim(0, 1.0)
    ax.set_title(f'{territory_name}: OA (baseline vs 15 признаков)', fontsize=12)
    ax.set_xlabel('Год')
    ax.set_ylabel('OA')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=PLOT_STYLE['grid_alpha'], axis='y')

plt.suptitle('Сравнение: baseline (11) vs обогащённый (15) набор признаков',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 6. Line chart OA + тренд
# ------------------------------------------------------------
logger.info("Строю line chart OA с трендом...")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for ax, territory_name in zip(axes, ['Historical', 'Greater']):
    subset = results_by_year.get(territory_name, {})
    if not subset:
        continue

    years = sorted(subset.keys())
    oa_vals = [subset[y]['oa'] for y in years]

    color = (PLOT_STYLE['historical_color'] if territory_name == 'Historical'
             else PLOT_STYLE['greater_color'])

    ax.plot(years, oa_vals, marker='o', linewidth=2, markersize=8,
            color=color, label='OA')

    # Линейный тренд
    if len(years) >= 3:
        coef = np.polyfit(years, oa_vals, 1)
        trend = np.polyval(coef, years)
        ax.plot(years, trend, '--', linewidth=1.5, color='black',
                alpha=0.6, label=f'Тренд: {coef[0]:+.5f}/год')

        # R²
        ss_res = np.sum((np.array(oa_vals) - trend) ** 2)
        ss_tot = np.sum((np.array(oa_vals) - np.mean(oa_vals)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        ax.text(0.02, 0.98, f'R² = {r2:.3f}',
                transform=ax.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.85),
                fontsize=10)

    ax.axhline(y=ACCURACY_THRESHOLD, color='green', linestyle=':',
               alpha=0.6, label=f'Цель ({ACCURACY_THRESHOLD})')
    ax.set_title(f'{territory_name}: тренд OA по годам', fontsize=12)
    ax.set_xlabel('Год')
    ax.set_ylabel('OA')
    ax.set_ylim(0.5, 1.0)
    ax.grid(True, alpha=PLOT_STYLE['grid_alpha'])
    ax.legend(loc='lower right', fontsize=9)

plt.suptitle('Тренд Overall Accuracy (8 целевых лет)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 7. Confusion Matrix Heatmap (все 8 лет)
# ------------------------------------------------------------
logger.info("Строю confusion matrix heatmap для всех 8 лет...")

def plot_confusion_heatmap(matrix, classes, title, ax):
    """Рисует нормализованную confusion matrix (по строкам)."""
    matrix = np.array(matrix)
    row_sums = matrix.sum(axis=1, keepdims=True)
    matrix_norm = matrix / np.where(row_sums > 0, row_sums, 1)

    im = ax.imshow(matrix_norm, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')

    # Подписи
    short_classes = ['Dense', 'Sparse', 'Built', 'Water']
    ax.set_xticks(range(len(classes)))
    ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(short_classes, rotation=0, fontsize=9)
    ax.set_yticklabels(short_classes, fontsize=9)
    ax.set_xlabel('Предсказание', fontsize=9)
    ax.set_ylabel('Истина', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')

    # Числа + доли
    for i in range(len(classes)):
        for j in range(len(classes)):
            count = matrix[i, j]
            frac = matrix_norm[i, j]
            text = f'{count:.0f}\n{frac:.2f}'
            text_color = 'white' if frac > 0.55 else 'black'
            ax.text(j, i, text, ha='center', va='center',
                    color=text_color, fontsize=8)


# 2 фигуры: одна для Historical, другая для Greater
for territory_name in ['Historical', 'Greater']:
    subset = results_by_year.get(territory_name, {})
    if not subset:
        continue

    years = sorted(subset.keys())
    n_years = len(years)

    # Сетка 1 строка × N лет (или 2×3 если 5-6 лет)
    if n_years <= 3:
        fig, axes = plt.subplots(1, n_years, figsize=(5 * n_years, 5))
        if n_years == 1:
            axes = [axes]
    else:
        fig, axes = plt.subplots(1, n_years, figsize=(4.5 * n_years, 5))

    for ax, year in zip(axes, years):
        r = subset[year]
        oa = r['oa']
        plot_confusion_heatmap(
            r['confusion_matrix'], CLASSES,
            f'{year}\nOA = {oa:.3f}',
            ax
        )

    plt.suptitle(f'{territory_name}: Confusion Matrix по годам '
                 f'(нормализовано по строкам)',
                 fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# 8. РАСШИРЕННЫЙ ВЕРДИКТ
# ------------------------------------------------------------
logger.info("")
logger.info("=" * 70)
logger.info("ВЕРДИКТ H2 (расширенный)")
logger.info("=" * 70)

issues = []
successes = []

n_total = len(RUN_PLAN)
n_success = sum(len(v) for v in results_by_year.values())

# --- Проверка 1: успешность прогонов ---
if n_success == n_total:
    successes.append(f"Все {n_total} прогонов успешны")
else:
    issues.append(f"Провалов: {n_total - n_success}")

# --- Проверка 2: обогащение признаков сработало ---
for territory_name in ['Historical', 'Greater']:
    subset = results_by_year.get(territory_name, {})
    if not subset:
        continue

    deltas = []
    for year, r in subset.items():
        baseline = BASELINE_11.get(territory_name, {}).get(year)
        if baseline:
            deltas.append(r['oa'] - baseline['oa'])

    if deltas:
        mean_delta = np.mean(deltas)
        if mean_delta >= 0.02:
            successes.append(f"[{territory_name}] ✅ OA вырос на {mean_delta:+.4f} "
                             f"(текстурные признаки работают)")
        elif mean_delta >= 0:
            successes.append(f"[{territory_name}] OA вырос незначительно "
                             f"({mean_delta:+.4f})")
        else:
            issues.append(f"[{territory_name}] OA упал на {mean_delta:+.4f} — "
                          f"текстурные признаки вредят")

# --- Проверка 3: площади согласованы ---
for territory_name in ['Historical', 'Greater']:
    subset = results_by_year.get(territory_name, {})
    for year, r in subset.items():
        area_diff_pct = abs(r['total_area'] - BOUNDARY_AREAS[territory_name]) \
                        / BOUNDARY_AREAS[territory_name] * 100
        if area_diff_pct > 2:
            issues.append(f"[{territory_name} {year}] площадь Δ={area_diff_pct:.1f}%")

if not any('площадь' in i for i in issues):
    successes.append("Площади согласованы с границами во всех итерациях")

# --- Проверка 4: текстурные признаки в топ-5 ---
if all_importances:
    avg_importance = {f: [] for f in FEATURES}
    for fi in all_importances.values():
        for feat in FEATURES:
            if feat in fi:
                avg_importance[feat].append(fi[feat])

    mean_imp = {f: np.mean(v) for f, v in avg_importance.items() if v}
    sorted_imp = sorted(mean_imp.items(), key=lambda x: x[1], reverse=True)
    top5_names = [f for f, _ in sorted_imp[:5]]
    texture_in_top5 = [f for f in top5_names if f in TEXTURE_BANDS]

    if texture_in_top5:
        successes.append(f"Текстурные признаки в топ-5: {texture_in_top5}")
    else:
        issues.append(f"Текстурные признаки не в топ-5 (не работают?)")

# --- Проверка 5: per-class F1 ---
low_f1_classes = set()
for territory_name in ['Historical', 'Greater']:
    subset = results_by_year.get(territory_name, {})
    for year, r in subset.items():
        metrics = compute_per_class_metrics(r['confusion_matrix'], CLASSES)
        for cls_name, m in metrics.items():
            if m['f1'] < 0.60:
                low_f1_classes.add(f"{territory_name} {year} {cls_name}")

if low_f1_classes:
    issues.append(f"Низкий F1 (<0.60) у {len(low_f1_classes)} классов")
else:
    successes.append("Все классы имеют F1 ≥ 0.60")

# --- Проверка 6: тренд OA ---
for territory_name in ['Historical', 'Greater']:
    subset = results_by_year.get(territory_name, {})
    if len(subset) >= 3:
        years = sorted(subset.keys())
        oa_vals = [subset[y]['oa'] for y in years]
        coef = np.polyfit(years, oa_vals, 1)
        slope = coef[0]

        if slope > 0.001:
            successes.append(f"[{territory_name}] OA растёт: {slope:+.5f}/год")
        elif slope < -0.001:
            issues.append(f"[{territory_name}] OA падает: {slope:+.5f}/год")

# --- Вывод ---
logger.info("")
for s in successes:
    logger.success(f"  ✅ {s}")
logger.info("")
if issues:
    for issue in issues:
        logger.warning(f"  ⚠️  {issue}")
else:
    logger.success("  Проблем не обнаружено")

logger.info("")
logger.info("=" * 70)
if not issues:
    logger.success("✅ H2 ПРОЙДЕН: пайплайн в отличном состоянии")
    logger.success("ГОТОВО К БЛОКУ I (карты, Excel, отчёт)")
else:
    logger.warning(f"⚠️  H2: {len(issues)} предупреждений")
    logger.info("Блок I можно запускать, но требуется внимание.")
logger.info("=" * 70)


# ------------------------------------------------------------
# 9. Обновление накопительной таблицы
# ------------------------------------------------------------
logger.info("")
logger.info("Обновляю накопительную таблицу...")

for territory_name in ['Historical', 'Greater']:
    subset = results_by_year.get(territory_name, {})
    for year, result in sorted(subset.items()):
        try:
            append_to_history(territory_name, year, result)
        except Exception as e:
            logger.warning(f"append_to_history {territory_name} {year}: {e}")

try:
    show_cumulative_table()
except Exception as e:
    logger.warning(f"Не удалось показать накопительную таблицу: {e}")


# ------------------------------------------------------------
# Итог
# ------------------------------------------------------------
logger.info("")
logger.info("Доступные переменные:")
logger.info(f"  results_by_year       — {n_success} прогонов")
logger.info(f"  failed_runs           — {len(failed_runs)} провалов")
logger.info(f"  all_importances       — feature importance по всем итерациям")
logger.info(f"  FINAL_FILE            — {FINAL_FILE.name}")
logger.info(f"  CUMULATIVE_TABLE_FILE — {CUMULATIVE_TABLE_FILE.name}")

In [ ]:
# ============================================================
# ЯЧЕЙКА H3: ЭКСПОРТ КЛАССИФИКАЦИЙ (ЗАГЛУШКА)
# ============================================================
# По умолчанию классификации живут только в памяти Python
# (results_by_year[territory][year]['classified']) и НЕ переживут
# перезапуск Colab.
#
# Эта ячейка — заглушка для случаев, когда нужно сохранить
# классификации надолго. Варианты:
#   A) Экспорт в GEE Assets (пользовательский аккаунт)
#   B) Экспорт в Google Drive как GeoTIFF
#   C) Экспорт локально (не имеет смысла для больших территорий)
#
# Пока НЕ АКТИВНА. Если понадобится — раскомментировать и настроить.
# ============================================================

logger.info("=" * 70)
logger.info("H3: ЭКСПОРТ КЛАССИФИКАЦИЙ (ЗАГЛУШКА)")
logger.info("=" * 70)

logger.info("Классификации в results_by_year доступны в текущей сессии.")
logger.info("Если Colab перезапустится — их придётся пересоздать (~20 мин).")

# ------------------------------------------------------------
# Пример: экспорт в GEE Assets
# ------------------------------------------------------------
# Раскомментируйте и адаптируйте путь, если нужно.

# GEE_USER = 'your_username'   # ← ваш GEE username
# GEE_FOLDER = f'users/{GEE_USER}/moscow_landcover'
#
# tasks = []
# for territory_name in ['Historical', 'Greater']:
#     subset = results_by_year.get(territory_name, {})
#     boundary = BOUNDARIES_MAP[territory_name]
#     for year, result in subset.items():
#         asset_id = f'{GEE_FOLDER}/{territory_name}_{year}'
#         task = ee.batch.Export.image.toAsset(
#             image=result['classified'].toInt(),
#             description=f'moscow_lc_{territory_name}_{year}',
#             assetId=asset_id,
#             scale=30,
#             region=boundary.geometry(),
#             maxPixels=1e13,
#             crs=ANALYSIS_CRS,
#         )
#         task.start()
#         tasks.append({'territory': territory_name, 'year': year,
#                       'task_id': task.id, 'asset_id': asset_id})
#         logger.info(f"Запущен экспорт: {asset_id} (task {task.id})")
#
# logger.success(f"Запущено {len(tasks)} задач экспорта.")
# logger.info("Проверить статус: ee.batch.Task.list()")

# ------------------------------------------------------------
# Пример: экспорт в Google Drive как GeoTIFF
# ------------------------------------------------------------
# for territory_name in ['Historical', 'Greater']:
#     subset = results_by_year.get(territory_name, {})
#     for year, result in subset.items():
#         export_dir = PATHS['INTERMEDIATE'] / 'classifications'
#         export_dir.mkdir(parents=True, exist_ok=True)
#         task = ee.batch.Export.image.toDrive(
#             image=result['classified'].toInt(),
#             description=f'moscow_lc_{territory_name}_{year}',
#             folder='Moscow_LandCover_Classifications',
#             fileNamePrefix=f'{territory_name}_{year}',
#             scale=30,
#             region=boundary.geometry(),
#             maxPixels=1e13,
#             crs=ANALYSIS_CRS,
#         )
#         task.start()

logger.info("")
logger.info("Заглушка активна. Для реального экспорта:")
logger.info("  1. Раскомментируйте один из блоков выше.")
logger.info("  2. Укажите ваш GEE_USER.")
logger.info("  3. Запустите ячейку.")
logger.info("  4. Мониторьте статус: ee.batch.Task.list()")

logger.success("=" * 70)
logger.success("✅ H3 ЗАВЕРШЁН (заглушка)")
logger.success("=" * 70)